# Radar Individual V8 — arquitetura final

V8 otimiza exclusivamente arquitetura, materialização e controles de execução. A
`Radar_Individual_v7_otimizada.ipynb` permanece a golden funcional e visual.

> **Precedência funcional:** para esta V8, divergências funcionais entre V7 e
> V1/V2 são resolvidas em favor da V7. V1/V2 permanece referência técnica somente
> onde não conflita com o comportamento homologado da V7.

O OOM observado em `limit(2).collect()` é tratado como **hipótese causal forte**
associada à resolução da DAG/cadeia de `crossJoin` usada pela V7 para compor o
resultado 1×80. Não é declarado como causa definitivamente provada.

O aceite corporativo deve registrar externamente o caso-âncora (`CD_CLI`,
`HOJE/CTMODATE`, `DATA_EXECUCAO`, `periodo` e estado das fontes) e executar V7/V8
sob o mesmo recorte. Nenhum identificador do caso-âncora é fixado neste notebook.


## A — Sessão, contrato de entrada e mapa estático

- Sessão criada exclusivamente pelo `GerenciadorLocal` corporativo.
- Nenhum aumento de memória e nenhum `spark.conf.set` no pipeline.
- Caminho principal: `CD_CLI`, `DATA_EXECUCAO` e `periodo` entre 1 e 6.
- CPF e amostra são helpers explícitos, separados e nunca executados automaticamente.


In [ ]:
from traceback import format_exc


In [ ]:
try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao='radar-financeiro-v8-otimizada',
        exibir_configuracao=False,
        ativar_logs=True,
    )
    spark = gerenciador_local.criar_sessao_spark(db2=True)
    print('[RADAR_V8] Sessão Spark inicializada pelo padrão corporativo.')
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
# Carrega conectores e utilitários de ambiente corporativo
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


In [ ]:
%%spark
import calendar
import datetime
import hashlib
import html
import json
import os
import re
import time
from decimal import Decimal, ROUND_HALF_UP, localcontext
from datetime import timedelta
from functools import reduce

from pyspark.sql import Row, functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, ShortType,
    StringType, DateType, TimestampType, DecimalType, ArrayType,
)
from pyspark.storagelevel import StorageLevel

FONTE_TRAN = 'DB2GFP.TRAN_RLZD_INST_PCT'
FONTE_CICLO = 'DB2GFP.CT_GRDR_FNCO'
FONTE_RENDA = 'DB2DFE.REN_AVLD_PF'
FONTE_PERFIL = 'DB2D1D.DVS_GRDR_FNCO_PF'
VIEW_RESULTADO = 'vw_radar_financeiro_cliente_mvp'

FETCHSIZE = 10_000
QUERY_TIMEOUT_SECONDS = 900
DIAS_CONTEXTO_RECONCILIACAO = 5
LIMITE_PAYLOAD_BYTES = 2 * 1024 * 1024

DATA_EXECUCAO = datetime.date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])
DT_MES_EXEA = DATA_EXECUCAO.replace(day=1)

def recuar_um_mes_calendario(data):
    total = data.year * 12 + data.month - 2
    ano, mes_zero = divmod(total, 12)
    mes = mes_zero + 1
    return datetime.date(ano, mes, min(data.day, calendar.monthrange(ano, mes)[1]))

DATA_INICIAL_PUBLICO = recuar_um_mes_calendario(DATA_EXECUCAO)
DATA_FINAL_EXCLUSIVA_PUBLICO = DATA_EXECUCAO
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))

ETAPAS_LEITURA_FUNCIONAL = ('Q1', 'Q2', 'Q3', 'Q4', 'Q5')
contagem_leituras_funcionais = {etapa: 0 for etapa in ETAPAS_LEITURA_FUNCIONAL}

def registrar_leitura_funcional(etapa):
    if etapa not in contagem_leituras_funcionais:
        raise ValueError(f'Etapa de leitura desconhecida: {etapa}.')
    contagem_leituras_funcionais[etapa] += 1
    if contagem_leituras_funcionais[etapa] > 1:
        raise RuntimeError(f'{etapa} tentou ler a fonte mais de uma vez.')

def limpar_views_temporarias_radar_v8():
    prefixos = (
        'vw_q1_', 'vw_q2_', 'vw_q3_', 'vw_q4_', 'vw_q5_', 'vw_mov_',
        'vw_pares_', 'vw_ids_', 'vw_reconciliado', 'vw_dashboard_',
        'vw_cenario_', '_vw_homologacao_', VIEW_RESULTADO,
    )
    for objeto in spark.catalog.listTables():
        if objeto.isTemporary and any(
            objeto.name.startswith(prefixo) or objeto.name == prefixo
            for prefixo in prefixos
        ):
            spark.catalog.dropTempView(objeto.name)

limpar_views_temporarias_radar_v8()


In [ ]:
%%spark
# Parâmetros do caminho principal. Informe CD_CLI; helpers não são autoexecutados.
CD_CLI = None
# CD_CLI = resolver_cd_cli_por_cpf(12345678901)
# CD_CLI = resolver_cd_cli_amostra()
periodo = 1

def normalizar_inteiro_exato(valor, nome, minimo=None, maximo=None):
    if isinstance(valor, bool):
        raise TypeError(f'{nome} deve ser inteiro, não booleano.')
    texto = str(valor).strip()
    if not re.fullmatch(r'[+-]?[0-9]+', texto):
        raise TypeError(f'{nome} deve possuir representação inteira exata.')
    numero = int(texto)
    if minimo is not None and numero < minimo:
        raise ValueError(f'{nome} abaixo do mínimo {minimo}.')
    if maximo is not None and numero > maximo:
        raise ValueError(f'{nome} acima do máximo {maximo}.')
    return numero

def normalizar_periodo(valor):
    return normalizar_inteiro_exato(valor, 'periodo', 1, 6)

periodo = normalizar_periodo(periodo)


In [ ]:
%%spark
# HELPERS OPCIONAIS — executar separadamente, copiar o CD_CLI retornado para a
# célula de parâmetros e reiniciar o caminho principal.
def resolver_cd_cli_por_cpf(cpf):
    cpf_num = normalizar_inteiro_exato(cpf, 'CPF', 0, 99999999999999)
    sql = f"""
SELECT CD_CLI
FROM {FONTE_TRAN}
WHERE NR_CPF_CNPJ_TITR = {cpf_num}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
ORDER BY TS_INCL_TRAN DESC
FETCH FIRST 1 ROW ONLY
"""
    linha = conector_db2.sql(
        sql, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS
    ).first()
    if linha is None:
        raise RuntimeError('CPF sem CD_CLI elegível na janela pública.')
    return int(linha['CD_CLI'])

def resolver_cd_cli_amostra(limite_candidatos=1000):
    limite = normalizar_inteiro_exato(limite_candidatos, 'limite_candidatos', 1, 10000)
    sql = f"""
SELECT CD_CLI
FROM {FONTE_TRAN}
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
GROUP BY CD_CLI
FETCH FIRST {limite} ROWS ONLY
"""
    linha = conector_db2.sql(
        sql, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS
    ).orderBy(F.rand()).first()
    if linha is None:
        raise RuntimeError('A amostra limitada não encontrou cliente elegível.')
    return int(linha['CD_CLI'])


In [ ]:
%%spark
# Início explícito do caminho principal. Até esta célula os helpers podem ser
# executados de forma independente para resolver um CD_CLI.
if CD_CLI is None:
    raise RuntimeError(
        'Informe CD_CLI para executar o caminho principal. '
        'Os helpers CPF/amostra acima são opcionais e nunca executam automaticamente.'
    )
CD_CLI = normalizar_inteiro_exato(CD_CLI, 'CD_CLI', -2147483648, 2147483647)


In [ ]:
%%spark
COLUNAS_CATEGORIAS = [
    'TIPO', 'CD_GRUPO', 'TX_GRUPO', 'CD_CATEGORIA', 'TX_CATEGORIA',
    'CD_IR', 'TX_IR', 'CD_CLASS_RADAR', 'TX_CLASS_RADAR',
    'IN_AGRO', 'IN_PARTICIPA_CALCULO', 'IN_PARTICIPA_ORCAMENTO',
]


In [ ]:
%%spark
LINHAS_CATEGORIAS = [
    (None, 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    (None, 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'N'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S', 'N'),
]


In [ ]:
%%spark
schema_categorias = StructType([
    StructField('TIPO', StringType(), True),
    StructField('CD_GRUPO', IntegerType(), False),
    StructField('TX_GRUPO', StringType(), False),
    StructField('CD_CATEGORIA', IntegerType(), False),
    StructField('TX_CATEGORIA', StringType(), False),
    StructField('CD_IR', IntegerType(), False),
    StructField('TX_IR', StringType(), False),
    StructField('CD_CLASS_RADAR', IntegerType(), False),
    StructField('TX_CLASS_RADAR', StringType(), False),
    StructField('IN_AGRO', StringType(), False),
    StructField('IN_PARTICIPA_CALCULO', StringType(), False),
    StructField('IN_PARTICIPA_ORCAMENTO', StringType(), False),
])

chaves_categorias = [(linha[3], linha[0]) for linha in LINHAS_CATEGORIAS]
if len(LINHAS_CATEGORIAS) != 70:
    raise RuntimeError(f'Mapa V7 deve possuir 70 linhas; obtido={len(LINHAS_CATEGORIAS)}.')
if len(chaves_categorias) != len(set(chaves_categorias)):
    raise RuntimeError('Mapa V7 possui chave (CD_CATEGORIA, TIPO) duplicada.')

FALLBACK_CLASSIFICACAO_V7 = {
    'TX_CATEGORIA': 'Sem Categoria',
    'CD_CLASS_RADAR': 0,
    'TX_CLASS_RADAR': 'Outras Entradas',
    'IN_AGRO': 'N',
    'IN_PARTICIPA_CALCULO': 'N',
    'IN_PARTICIPA_ORCAMENTO': 'N',
}

df_categorias = spark.createDataFrame(LINHAS_CATEGORIAS, schema_categorias)
df_categorias.createOrReplaceTempView('vw_categorias')


## B — Q1 a Q4 encerradas em escalares

Cada etapa possui no máximo uma leitura funcional. Nenhuma lineage dessas fontes
alcança Q5, a agregação financeira ou o resultado final.


In [ ]:
%%spark
# Q1 — uma leitura e uma agregação. Preserva CPF único e conta elegível única V7.
registrar_leitura_funcional('Q1')
sql_q1_db2 = f"""
SELECT
    CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR,
    NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
"""
df_q1 = conector_db2.sql(
    sql_q1_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS
)
condicao_conta = (
    (F.col('NR_MCA_PCT_OPB') == F.lit(999999999))
    & (F.col('CD_PRD') == F.lit(6))
    & F.col('NR_AG_TITR').isNotNull()
    & F.col('CD_CT_TITR').isNotNull()
    & (F.trim(F.col('CD_CT_TITR').cast('string')) != F.lit(''))
)

print('[INICIO] Q1_RESOLVER_CONTEXTO_ESCALAR')
_inicio_action = time.perf_counter()
q1 = df_q1.agg(
    F.count(F.lit(1)).cast('long').alias('QT_LINHAS'),
    F.max('TS_INCL_TRAN').alias('TS_INCL_TRAN_REF'),
    F.countDistinct('NR_CPF_CNPJ_TITR').alias('QT_CPFS'),
    F.max('NR_CPF_CNPJ_TITR').cast(DecimalType(14, 0)).alias('CD_CPF_MAX'),
    F.countDistinct(
        F.when(condicao_conta, F.struct('NR_AG_TITR', 'CD_CT_TITR'))
    ).alias('QT_CONTAS'),
    F.max(F.when(condicao_conta, F.col('NR_AG_TITR'))).alias('NR_AG_TITR_MAX'),
    F.max(F.when(condicao_conta, F.col('CD_CT_TITR'))).alias('CD_CT_TITR_MAX'),
).first()
print(f'[FIM] Q1_RESOLVER_CONTEXTO_ESCALAR | {time.perf_counter() - _inicio_action:.3f}s')

if q1 is None or int(q1['QT_LINHAS']) == 0:
    raise RuntimeError(
        f'O cliente {CD_CLI} não pertence à janela de formação do público '
        f'({DATA_INICIAL_PUBLICO} a {DATA_FINAL_EXCLUSIVA_PUBLICO}).'
    )

fl_cpf_unico = 'S' if int(q1['QT_CPFS']) == 1 else 'N'
cd_cpf = q1['CD_CPF_MAX'] if fl_cpf_unico == 'S' else None
fl_conta_unica = 'S' if int(q1['QT_CONTAS']) == 1 else 'N'
nr_ag_titr = q1['NR_AG_TITR_MAX'] if fl_conta_unica == 'S' else None
cd_ct_titr = q1['CD_CT_TITR_MAX'] if fl_conta_unica == 'S' else None

cd_uor_cc_norm = None
nr_cc_norm = None
if fl_conta_unica == 'S':
    agencia_txt = str(nr_ag_titr).strip()
    conta_txt = str(cd_ct_titr).strip()
    conta_significativa = conta_txt.lstrip('0')
    if (
        re.fullmatch(r'[0-9]+', agencia_txt)
        and re.fullmatch(r'[0-9]+', conta_txt)
        and -2147483648 <= int(agencia_txt) <= 2147483647
        and len(conta_significativa) <= 11
    ):
        cd_uor_cc_norm = int(agencia_txt)
        nr_cc_norm = Decimal(conta_significativa or '0')

estado_cliente = Row(
    CD_CLI=CD_CLI,
    TS_INCL_TRAN_REF=q1['TS_INCL_TRAN_REF'],
    FL_CPF_UNICO=fl_cpf_unico,
    CD_CPF=cd_cpf,
    FL_CONTA_ELEGIVEL_UNICA=fl_conta_unica,
    NR_AG_TITR=nr_ag_titr,
    CD_CT_TITR=cd_ct_titr,
    CD_UOR_CC_NORM=cd_uor_cc_norm,
    NR_CC_NORM=nr_cc_norm,
)
cpf_row = estado_cliente
res_cta = estado_cliente
cta_norm_row = estado_cliente
tem_conta_norm = cd_uor_cc_norm is not None and nr_cc_norm is not None


In [ ]:
%%spark
# Q2 — registro de ciclo mais recente, encerrado no driver.
row_ciclo = None
if tem_conta_norm:
    registrar_leitura_funcional('Q2')
    sql_q2_db2 = f"""
SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
FROM {FONTE_CICLO}
WHERE CD_UOR_CC = {cd_uor_cc_norm}
  AND NR_CC = {nr_cc_norm}
ORDER BY TS_ULT_EXEA_PSQ DESC
FETCH FIRST 1 ROW ONLY
"""
    print('[INICIO] Q2_RESOLVER_CICLO_ESCALAR')
    _inicio_action = time.perf_counter()
    linha_q2 = conector_db2.sql(
        sql_q2_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS
    ).first()
    print(f'[FIM] Q2_RESOLVER_CICLO_ESCALAR | {time.perf_counter() - _inicio_action:.3f}s')
    if linha_q2 is not None:
        row_ciclo = Row(
            TS_DD_INC_MM_CLC_BLC_REF=linha_q2['TS_ULT_EXEA_PSQ'],
            DD_INC_MM_CLC_BLC=(
                None if linha_q2['DD_INC_MM_CLC_BLC'] is None
                else int(linha_q2['DD_INC_MM_CLC_BLC'])
            ),
        )
else:
    print('[RADAR_V8] Q2: SKIPPED (conta normalizada indisponível).')

dd_ciclo_val = row_ciclo['DD_INC_MM_CLC_BLC'] if row_ciclo else None
if not tem_conta_norm:
    dd_fallback_val = None
elif dd_ciclo_val is None:
    dd_fallback_val = 1
else:
    dd_fallback_val = int(dd_ciclo_val)


In [ ]:
%%spark
# Q3 — renda mais recente do CPF único; Hive encerrado em uma Row.
res_renda = Row(DT_REN_PRES_REF=None, VL_REN_PRES=None)
if fl_cpf_unico == 'S' and cd_cpf is not None:
    registrar_leitura_funcional('Q3')
    cpf_num = int(cd_cpf)
    sql_q3_hive = f"""
SELECT
    CAST(DT_INCL_REN_AVLD AS DATE) AS DT_REN_PRES_REF,
    CAST(VL_REN * {periodo} AS DECIMAL(17,2)) AS VL_REN_PRES
FROM {FONTE_RENDA}
WHERE NR_CPF_BASE_SRF = {cpf_num}
ORDER BY DT_INCL_REN_AVLD DESC
LIMIT 1
"""
    print('[INICIO] Q3_RESOLVER_RENDA_ESCALAR')
    _inicio_action = time.perf_counter()
    linha_q3 = spark.sql(sql_q3_hive).first()
    print(f'[FIM] Q3_RESOLVER_RENDA_ESCALAR | {time.perf_counter() - _inicio_action:.3f}s')
    if linha_q3 is not None:
        res_renda = linha_q3
else:
    print('[RADAR_V8] Q3: SKIPPED (CPF único indisponível).')


In [ ]:
%%spark
# Q4 — pushdown temporal aprovado. Duas linhas preservam o gate de ambiguidade V7.
registrar_leitura_funcional('Q4')
sql_q4_db2 = f"""
SELECT
    CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
FROM {FONTE_PERFIL}
WHERE CD_CLI = {CD_CLI}
  AND DT_REF <= DATE('{DATA_EXECUCAO.isoformat()}')
ORDER BY DT_REF DESC
FETCH FIRST 2 ROWS ONLY
"""
print('[INICIO] Q4_RESOLVER_PERFIL_ESCALAR')
_inicio_action = time.perf_counter()
linhas_q4 = conector_db2.sql(
    sql_q4_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS
).collect()
print(f'[FIM] Q4_RESOLVER_PERFIL_ESCALAR | {time.perf_counter() - _inicio_action:.3f}s')

if len(linhas_q4) == 2 and linhas_q4[0]['DT_REF'] == linhas_q4[1]['DT_REF']:
    raise RuntimeError('Q4 retornou mais de uma linha na maior DT_REF elegível.')
if linhas_q4 and linhas_q4[0]['DT_REF'] > DATA_EXECUCAO:
    raise RuntimeError('Q4 retornou DT_REF posterior à DATA_EXECUCAO.')

if not linhas_q4:
    res_prfl = Row(
        DT_REF_PRFL=None, CD_MAC_PRFL_CLI=None, NM_MAC_PRFL_CLI=None,
        CD_MIC_PRFL_CLI=None, NM_MIC_PRFL_CLI=None,
    )
else:
    linha = linhas_q4[0]
    res_prfl = Row(
        DT_REF_PRFL=linha['DT_REF'],
        CD_MAC_PRFL_CLI=(None if linha['CD_MAC_PRFL_CLI'] is None else int(linha['CD_MAC_PRFL_CLI'])),
        NM_MAC_PRFL_CLI=linha['NM_MAC_PRFL_CLI'],
        CD_MIC_PRFL_CLI=(None if linha['CD_MIC_PRFL_CLI'] is None else int(linha['CD_MIC_PRFL_CLI'])),
        NM_MIC_PRFL_CLI=linha['NM_MIC_PRFL_CLI'],
    )


In [ ]:
%%spark
# Janela financeira fechada, calculada somente com escalares Q1/Q2.
def calcular_inicio_periodo_fechado(inicio_ciclo_aberto, dia_ciclo, quantidade_ciclos):
    mes_alvo_total = inicio_ciclo_aberto.year * 12 + inicio_ciclo_aberto.month - 1 - quantidade_ciclos
    ano_alvo, mes_alvo_zero = divmod(mes_alvo_total, 12)
    mes_alvo = mes_alvo_zero + 1
    dia_alvo = min(dia_ciclo, calendar.monthrange(ano_alvo, mes_alvo)[1])
    return datetime.date(ano_alvo, mes_alvo, dia_alvo)

if dd_fallback_val is None:
    dt_ref_ini_val = None
    dt_ref_fim_val = None
else:
    if not 1 <= dd_fallback_val <= 31:
        raise RuntimeError(f'Dia de ciclo fora do domínio 1..31: {dd_fallback_val}.')
    ts_ref = estado_cliente['TS_INCL_TRAN_REF']
    dia_mes_ref = min(dd_fallback_val, calendar.monthrange(ts_ref.year, ts_ref.month)[1])
    candidato = datetime.datetime(ts_ref.year, ts_ref.month, dia_mes_ref)
    if ts_ref >= candidato:
        inicio_aberto = candidato.date()
    else:
        total = ts_ref.year * 12 + ts_ref.month - 2
        ano_ant, mes_ant_zero = divmod(total, 12)
        mes_ant = mes_ant_zero + 1
        inicio_aberto = datetime.date(
            ano_ant, mes_ant,
            min(dd_fallback_val, calendar.monthrange(ano_ant, mes_ant)[1]),
        )
    dt_ref_fim_val = inicio_aberto - timedelta(days=1)
    dt_ref_ini_val = calcular_inicio_periodo_fechado(
        inicio_aberto, dd_fallback_val, periodo
    )

dt_ini_j = dt_ref_ini_val
dt_fim_j = dt_ref_fim_val


## C — Q5 única, reconciliação V7 e raiz 1:1

Q5 é a única leitura distribuída compartilhada. A materialização valida a identidade
física antes dos UDFs. Os algoritmos de pareamento exato e de borda abaixo são
copiados da V7 sem alteração funcional.


In [ ]:
%%spark
COLUNAS_Q5 = [
    'NR_TRAN_INST_PCT', 'CD_CLI', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN',
]
SCHEMA_Q5_FUNCIONAL = StructType([
    StructField('NR_TRAN_INST_PCT', LongType(), True),
    StructField('CD_CLI', IntegerType(), True),
    StructField('DT_TRAN', DateType(), True),
    StructField('CD_NTZ_CTB_TRAN', StringType(), True),
    StructField('CD_CTGR_TRAN_OGNL', IntegerType(), True),
    StructField('CD_TIP_MOE_CRR', StringType(), True),
    StructField('VL_TRAN', DecimalType(15, 2), True),
])
schema_q5_apresentacao = StructType(
    list(SCHEMA_Q5_FUNCIONAL.fields) + [
        StructField('TX_DCR_TRAN_OGNL', StringType(), True),
        StructField('NR_MCA_PCT_OPB', StringType(), True),
    ]
)

if dt_ini_j is None:
    df_q5_contexto_apresentacao = spark.createDataFrame([], schema_q5_apresentacao)
    print('[RADAR_V8] Q5: SKIPPED (janela financeira indisponível).')
else:
    registrar_leitura_funcional('Q5')
    dt_q5_ini = (dt_ini_j - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)).isoformat()
    dt_q5_fim = (dt_fim_j + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)).isoformat()
    sql_q5_db2 = f"""
SELECT
    NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN,
    TX_DCR_TRAN_OGNL, NR_MCA_PCT_OPB
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND DT_TRAN >= DATE('{dt_q5_ini}')
  AND DT_TRAN <= DATE('{dt_q5_fim}')
  AND (
      CD_NTZ_CTB_TRAN = 'C'
      OR (CD_NTZ_CTB_TRAN = 'D' AND IN_VSLO_CSM = 'S')
  )
"""
    df_q5_contexto_apresentacao = conector_db2.sql(
        sql_q5_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS
    )

df_q5_contexto_apresentacao = df_q5_contexto_apresentacao.persist(
    StorageLevel.MEMORY_AND_DISK
)

print('[INICIO] Q5_MATERIALIZAR_E_VALIDAR_IDENTIDADE')
_inicio_action = time.perf_counter()
metricas_q5 = df_q5_contexto_apresentacao.agg(
    F.count(F.lit(1)).cast('long').alias('QT_Q5'),
    F.countDistinct('NR_TRAN_INST_PCT').cast('long').alias('QT_IDS_Q5'),
    F.sum(F.when(F.col('NR_TRAN_INST_PCT').isNull(), 1).otherwise(0)).cast('long').alias('QT_IDS_NULOS'),
).first()
print(f'[FIM] Q5_MATERIALIZAR_E_VALIDAR_IDENTIDADE | {time.perf_counter() - _inicio_action:.3f}s')

qt_q5 = int(metricas_q5['QT_Q5'])
qt_ids_q5 = int(metricas_q5['QT_IDS_Q5'])
qt_ids_nulos_q5 = int(metricas_q5['QT_IDS_NULOS'] or 0)
if qt_ids_nulos_q5 != 0 or qt_ids_q5 != qt_q5:
    raise RuntimeError(
        'NR_TRAN_INST_PCT não é identidade física única da Q5: '
        f'linhas={qt_q5}, ids_distintos={qt_ids_q5}, ids_nulos={qt_ids_nulos_q5}.'
    )
print(f'[RADAR_V8] Q5 materializada uma vez: {qt_q5} registros.')

df_q5_contexto_apresentacao.createOrReplaceTempView('vw_q5_mov_contexto_apresentacao')
df_q5_contexto = df_q5_contexto_apresentacao.select(*COLUNAS_Q5)
df_q5_contexto.createOrReplaceTempView('vw_q5_mov_contexto')

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_marcado AS
SELECT
    NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN, NR_MCA_PCT_OPB,
    CASE
        WHEN DT_TRAN >= DATE('{dt_ini_j.isoformat() if dt_ini_j else '1900-01-01'}')
         AND DT_TRAN <= DATE('{dt_fim_j.isoformat() if dt_fim_j else '1900-01-01'}')
        THEN 'S' ELSE 'N'
    END AS IN_JANELA
FROM vw_q5_mov_contexto_apresentacao
""")


In [ ]:
%%spark
# 1. Identificação dos Pares Exatos Candidatos
# Preserva cliente, data, valor, moeda e janela; acrescenta apenas bancos conhecidos e diferentes.
SCHEMA_PAR_EXATO = ArrayType(StructType([
    StructField('ID_CREDITO', LongType(), False),
    StructField('ID_DEBITO', LongType(), False),
]))

def parear_listas_exatas_sql_impl(lista_creditos, lista_debitos):
    creditos = sorted(
        [(int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB']) for r in (lista_creditos or [])],
        key=lambda item: item[0],
    )
    debitos = sorted(
        [(int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB']) for r in (lista_debitos or [])],
        key=lambda item: item[0],
    )
    adjacencias = [
        [
            indice_debito
            for indice_debito, (_, banco_debito) in enumerate(debitos)
            if banco_credito is not None
            and banco_debito is not None
            and banco_credito != banco_debito
        ]
        for _, banco_credito in creditos
    ]
    credito_por_debito = {}
    debito_por_credito = {}

    for credito_inicial in range(len(creditos)):
        fila = [credito_inicial]
        vistos_creditos = {credito_inicial}
        vistos_debitos = set()
        credito_anterior_por_debito = {}
        debito_livre = None
        posicao_fila = 0

        while posicao_fila < len(fila) and debito_livre is None:
            indice_credito = fila[posicao_fila]
            posicao_fila += 1
            for indice_debito in adjacencias[indice_credito]:
                if indice_debito in vistos_debitos:
                    continue
                vistos_debitos.add(indice_debito)
                credito_anterior_por_debito[indice_debito] = indice_credito
                if indice_debito not in credito_por_debito:
                    debito_livre = indice_debito
                    break
                proximo_credito = credito_por_debito[indice_debito]
                if proximo_credito not in vistos_creditos:
                    vistos_creditos.add(proximo_credito)
                    fila.append(proximo_credito)

        if debito_livre is None:
            continue

        indice_debito = debito_livre
        while True:
            indice_credito = credito_anterior_por_debito[indice_debito]
            debito_anterior = debito_por_credito.get(indice_credito)
            credito_por_debito[indice_debito] = indice_credito
            debito_por_credito[indice_credito] = indice_debito
            if debito_anterior is None:
                break
            indice_debito = debito_anterior

    pares = [
        (creditos[indice_credito][0], debitos[indice_debito][0])
        for indice_credito, indice_debito in debito_por_credito.items()
    ]
    return sorted(pares, key=lambda par: (par[0], par[1]))

spark.udf.register('parear_exato_udf', parear_listas_exatas_sql_impl, SCHEMA_PAR_EXATO)

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pares_exatos_candidatos AS
WITH creditos_agg AS (
    SELECT
        CD_CLI,
        DT_TRAN,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        IN_JANELA,
        COLLECT_LIST(NAMED_STRUCT(
            'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT,
            'NR_MCA_PCT_OPB', NR_MCA_PCT_OPB
        )) AS LISTA_CREDITOS
    FROM vw_mov_marcado
    WHERE NR_TRAN_INST_PCT IS NOT NULL
      AND CD_NTZ_CTB_TRAN = 'C'
      AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL
      AND CD_TIP_MOE_CRR IS NOT NULL
    GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
),
debitos_agg AS (
    SELECT
        CD_CLI,
        DT_TRAN,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        IN_JANELA,
        COLLECT_LIST(NAMED_STRUCT(
            'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT,
            'NR_MCA_PCT_OPB', NR_MCA_PCT_OPB
        )) AS LISTA_DEBITOS
    FROM vw_mov_marcado
    WHERE NR_TRAN_INST_PCT IS NOT NULL
      AND CD_NTZ_CTB_TRAN = 'D'
      AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL
      AND CD_TIP_MOE_CRR IS NOT NULL
    GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
),
pares_array AS (
    SELECT
        c.CD_CLI,
        c.DT_TRAN,
        c.VL_TRAN,
        c.CD_TIP_MOE_CRR,
        c.IN_JANELA,
        EXPLODE(parear_exato_udf(c.LISTA_CREDITOS, d.LISTA_DEBITOS)) AS PAR
    FROM creditos_agg c
    INNER JOIN debitos_agg d
       ON c.CD_CLI = d.CD_CLI
      AND c.DT_TRAN = d.DT_TRAN
      AND c.VL_TRAN = d.VL_TRAN
      AND c.CD_TIP_MOE_CRR = d.CD_TIP_MOE_CRR
      AND c.IN_JANELA = d.IN_JANELA
)
SELECT
    CD_CLI,
    DT_TRAN,
    VL_TRAN,
    CD_TIP_MOE_CRR,
    IN_JANELA,
    PAR.ID_CREDITO AS ID_CREDITO,
    PAR.ID_DEBITO AS ID_DEBITO
FROM pares_array
""")


In [ ]:
%%spark
# 2. Seleção dos IDs Consumidos por Par Exato
# Cada ID provém de um único par máximo compatível com bancos conhecidos e diferentes.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_exatos AS
SELECT
    ID_CREDITO AS NR_TRAN_INST_PCT,
    CASE WHEN IN_JANELA = 'S' THEN 'EXATO_OFICIAL' ELSE 'EXATO_CONTEXTO' END AS TIPO_CONSUMO
FROM vw_pares_exatos_candidatos
UNION ALL
SELECT
    ID_DEBITO AS NR_TRAN_INST_PCT,
    CASE WHEN IN_JANELA = 'S' THEN 'EXATO_OFICIAL' ELSE 'EXATO_CONTEXTO' END AS TIPO_CONSUMO
FROM vw_pares_exatos_candidatos
""")


In [ ]:
%%spark
# B5 — IDs exatos são reutilizados por residual, consolidação e dashboard.
df_ids_consumidos_exatos_cache = spark.table('vw_ids_consumidos_exatos').persist(StorageLevel.MEMORY_AND_DISK)


In [ ]:
%%spark
df_ids_consumidos_exatos_cache.createOrReplaceTempView('vw_ids_consumidos_exatos')


In [ ]:
%%spark
# 3. Universo Residual
# Transações não consumidas por par exato que avançam para a reconciliação de bordas temporais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_contexto_residual AS
SELECT m.*
FROM vw_mov_marcado m
LEFT ANTI JOIN vw_ids_consumidos_exatos e ON m.NR_TRAN_INST_PCT = e.NR_TRAN_INST_PCT
""")


In [ ]:
%%spark
# UDF SQL para o matching determinístico de bordas temporais (±5 dias)
# Critérios de Otimização Contratuais:
# 1. Maximizar quantidade total de pares (cardinalidade)
# 2. Minimizar distância absoluta total em dias (proximidade)
# 3. Desempatar deterministicamente por IDs ascendentes
SCHEMA_PAR_BORDA = ArrayType(StructType([
    StructField('NR_TRAN_DENTRO', LongType(), False),
    StructField('NR_TRAN_FORA', LongType(), False),
    StructField('DT_TRAN_DENTRO', DateType(), False),
    StructField('DT_TRAN_FORA', DateType(), False),
    StructField('DIF_DIAS', IntegerType(), False),
]))


In [ ]:
%%spark
def parear_listas_residuais_sql_impl(lista_dentro, lista_fora):
    dentro = sorted(
        [
            (r['DT_TRAN'], int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB'])
            for r in (lista_dentro or [])
        ],
        key=lambda item: (item[0], item[1])
    )
    fora = sorted(
        [
            (r['DT_TRAN'], int(r['NR_TRAN_INST_PCT']), r['NR_MCA_PCT_OPB'])
            for r in (lista_fora or [])
        ],
        key=lambda item: (item[0], item[1])
    )
    n = len(dentro)
    m = len(fora)
    vazio = (0, 0, ())
    dp = [[vazio for _ in range(m + 1)] for _ in range(n + 1)]

    def chave_solucao(solucao):
        quantidade, custo, pares = solucao
        assinatura_ids = tuple((par[0], par[1]) for par in pares)
        return (-quantidade, custo, assinatura_ids)

    for i in range(n - 1, -1, -1):
        for j in range(m - 1, -1, -1):
            candidatos = [dp[i + 1][j], dp[i][j + 1]]
            dt_dentro, id_dentro, banco_dentro = dentro[i]
            dt_fora, id_fora, banco_fora = fora[j]
            dif_dias = abs((dt_dentro - dt_fora).days)
            if (
                1 <= dif_dias <= DIAS_CONTEXTO_RECONCILIACAO
                and banco_dentro is not None
                and banco_fora is not None
                and banco_dentro != banco_fora
            ):
                quantidade, custo, pares = dp[i + 1][j + 1]
                par = (id_dentro, id_fora, dt_dentro, dt_fora, dif_dias)
                candidatos.append((quantidade + 1, custo + dif_dias, (par,) + pares))
            dp[i][j] = min(candidatos, key=chave_solucao)

    return list(dp[0][0][2])


In [ ]:
%%spark
spark.udf.register('parear_borda_udf', parear_listas_residuais_sql_impl, SCHEMA_PAR_BORDA)


In [ ]:
%%spark
# 1. Agrupamento das listas de dentro e fora da janela com valores e moedas iguais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pares_borda_calculados AS
WITH dentro_agg AS (
    SELECT
        CD_CLI,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        CD_NTZ_CTB_TRAN AS NTZ_DENTRO,
        COLLECT_LIST(NAMED_STRUCT(
            'DT_TRAN', DT_TRAN,
            'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT,
            'NR_MCA_PCT_OPB', NR_MCA_PCT_OPB
        )) AS LISTA_DENTRO
    FROM vw_mov_contexto_residual
    WHERE IN_JANELA = 'S'
      AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN
),
fora_agg AS (
    SELECT
        CD_CLI,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END AS NTZ_DENTRO,
        COLLECT_LIST(NAMED_STRUCT(
            'DT_TRAN', DT_TRAN,
            'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT,
            'NR_MCA_PCT_OPB', NR_MCA_PCT_OPB
        )) AS LISTA_FORA
    FROM vw_mov_contexto_residual
    WHERE IN_JANELA = 'N'
      AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END
),
pares_array AS (
    SELECT
        d.CD_CLI,
        d.VL_TRAN,
        d.CD_TIP_MOE_CRR,
        EXPLODE(parear_borda_udf(d.LISTA_DENTRO, f.LISTA_FORA)) AS PAR
    FROM dentro_agg d
    INNER JOIN fora_agg f
       ON d.CD_CLI = f.CD_CLI
      AND d.VL_TRAN = f.VL_TRAN
      AND d.CD_TIP_MOE_CRR = f.CD_TIP_MOE_CRR
      AND d.NTZ_DENTRO = f.NTZ_DENTRO
)
SELECT
    PAR.NR_TRAN_DENTRO AS NR_TRAN_DENTRO,
    PAR.NR_TRAN_FORA AS NR_TRAN_FORA,
    PAR.DT_TRAN_DENTRO AS DT_TRAN_DENTRO,
    PAR.DT_TRAN_FORA AS DT_TRAN_FORA,
    PAR.DIF_DIAS AS DIF_DIAS
FROM pares_array
""")


In [ ]:
%%spark
# B5 — pares de borda são reutilizados por remoção, métricas e dashboard.
df_pares_borda_calculados_cache = spark.table('vw_pares_borda_calculados').persist(StorageLevel.MEMORY_AND_DISK)


In [ ]:
%%spark
df_pares_borda_calculados_cache.createOrReplaceTempView('vw_pares_borda_calculados')


In [ ]:
%%spark
# 2. Identificação dos IDs Consumidos por Borda Temporal
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_borda AS
SELECT NR_TRAN_DENTRO AS NR_TRAN_INST_PCT, 'BORDA_OFICIAL' AS TIPO_CONSUMO FROM vw_pares_borda_calculados
UNION ALL
SELECT NR_TRAN_FORA AS NR_TRAN_INST_PCT, 'BORDA_CONTEXTO' AS TIPO_CONSUMO FROM vw_pares_borda_calculados
""")


In [ ]:
%%spark
# Sidecar canônico: cada ID possui zero ou uma contraparte/forma de consumo.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_sidecar_reconciliacao_v8 AS
SELECT ID_CREDITO AS NR_TRAN_INST_PCT,
       ID_DEBITO AS NR_TRAN_CONTRAPARTE,
       CASE WHEN IN_JANELA = 'S' THEN 'EXATO_OFICIAL' ELSE 'EXATO_CONTEXTO' END AS TIPO_CONSUMO,
       CAST(0 AS INT) AS DIF_DIAS,
       'CREDITO' AS PAPEL_PAR
FROM vw_pares_exatos_candidatos
UNION ALL
SELECT ID_DEBITO, ID_CREDITO,
       CASE WHEN IN_JANELA = 'S' THEN 'EXATO_OFICIAL' ELSE 'EXATO_CONTEXTO' END,
       CAST(0 AS INT), 'DEBITO'
FROM vw_pares_exatos_candidatos
UNION ALL
SELECT NR_TRAN_DENTRO, NR_TRAN_FORA, 'BORDA_OFICIAL', DIF_DIAS, 'DENTRO'
FROM vw_pares_borda_calculados
UNION ALL
SELECT NR_TRAN_FORA, NR_TRAN_DENTRO, 'BORDA_CONTEXTO', DIF_DIAS, 'FORA'
FROM vw_pares_borda_calculados
""")

# LEFT JOIN V7 + fallbacks deliberadamente preservados. O DataFrame reconciliado
# também carrega a classificação de apresentação para que o dashboard dependa de
# uma única raiz materializada.
df_reconciliado = spark.sql("""
SELECT
    q.NR_TRAN_INST_PCT,
    q.CD_CLI,
    q.DT_TRAN,
    q.CD_NTZ_CTB_TRAN,
    q.CD_CTGR_TRAN_OGNL,
    q.CD_TIP_MOE_CRR,
    q.VL_TRAN,
    CAST(q.TX_DCR_TRAN_OGNL AS STRING) AS TX_DCR_TRAN_OGNL,
    CAST(q.NR_MCA_PCT_OPB AS STRING) AS NR_MCA_PCT_OPB,
    CASE
        WHEN q.DT_TRAN >= DATE('""" + (dt_ini_j.isoformat() if dt_ini_j else '1900-01-01') + r"""')
         AND q.DT_TRAN <= DATE('""" + (dt_fim_j.isoformat() if dt_fim_j else '1900-01-01') + r"""')
        THEN 'S' ELSE 'N'
    END AS IN_JANELA,
    s.TIPO_CONSUMO,
    s.PAPEL_PAR,
    s.DIF_DIAS,
    s.NR_TRAN_CONTRAPARTE,
    cp.DT_TRAN AS DT_TRAN_CONTRAPARTE,
    cp.CD_NTZ_CTB_TRAN AS CD_NTZ_CTB_TRAN_CONTRAPARTE,
    CAST(cp.TX_DCR_TRAN_OGNL AS STRING) AS TX_DCR_TRAN_OGNL_CONTRAPARTE,
    CAST(cp.NR_MCA_PCT_OPB AS STRING) AS NR_MCA_PCT_OPB_CONTRAPARTE,
    c.CD_GRUPO,
    c.TX_GRUPO,
    c.CD_IR,
    c.TX_IR,
    COALESCE(c.TX_CATEGORIA, 'Sem Categoria') AS TX_CATEGORIA,
    c.CD_CLASS_RADAR AS CD_CLASS_RADAR_MAPA,
    c.TX_CLASS_RADAR AS TX_CLASS_RADAR_MAPA,
    COALESCE(c.CD_CLASS_RADAR, 0) AS CD_CLASS_RADAR,
    COALESCE(c.TX_CLASS_RADAR, 'Outras Entradas') AS TX_CLASS_RADAR,
    CASE
        WHEN c.CD_CLASS_RADAR IS NOT NULL AND c.TX_CLASS_RADAR IS NOT NULL THEN 'S'
        ELSE 'N'
    END AS IN_CLASSIFICADA_APRESENTACAO,
    COALESCE(c.IN_AGRO, 'N') AS IN_AGRO,
    COALESCE(c.IN_PARTICIPA_CALCULO, 'N') AS IN_PARTICIPA_CALCULO,
    COALESCE(c.IN_PARTICIPA_ORCAMENTO, 'N') AS IN_PARTICIPA_ORCAMENTO
FROM vw_q5_mov_contexto_apresentacao q
LEFT JOIN vw_sidecar_reconciliacao_v8 s
  ON q.NR_TRAN_INST_PCT = s.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao cp
  ON s.NR_TRAN_CONTRAPARTE = cp.NR_TRAN_INST_PCT
LEFT JOIN vw_categorias c
  ON q.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND q.CD_NTZ_CTB_TRAN = c.TIPO
""").persist(StorageLevel.MEMORY_AND_DISK)

print('[INICIO] RECONCILIADO_MATERIALIZAR_E_VALIDAR_1_POR_Q5')
_inicio_action = time.perf_counter()
metricas_reconciliado = df_reconciliado.agg(
    F.count(F.lit(1)).cast('long').alias('QT_RECONCILIADO'),
    F.countDistinct('NR_TRAN_INST_PCT').cast('long').alias('QT_IDS_RECONCILIADO'),
    F.sum(F.when(F.col('NR_TRAN_INST_PCT').isNull(), 1).otherwise(0)).cast('long').alias('QT_IDS_NULOS'),
).first()
print(f'[FIM] RECONCILIADO_MATERIALIZAR_E_VALIDAR_1_POR_Q5 | {time.perf_counter() - _inicio_action:.3f}s')

qt_reconciliado = int(metricas_reconciliado['QT_RECONCILIADO'])
qt_ids_reconciliado = int(metricas_reconciliado['QT_IDS_RECONCILIADO'])
qt_ids_nulos_reconciliado = int(metricas_reconciliado['QT_IDS_NULOS'] or 0)
if (
    qt_reconciliado != qt_q5
    or qt_ids_reconciliado != qt_q5
    or qt_ids_nulos_reconciliado != 0
):
    raise RuntimeError(
        'Sidecar/classificação multiplicou ou perdeu transações: '
        f'Q5={qt_q5}, reconciliado={qt_reconciliado}, '
        f'ids={qt_ids_reconciliado}, nulos={qt_ids_nulos_reconciliado}.'
    )

df_reconciliado.createOrReplaceTempView('vw_reconciliado_v8')

# Só agora a lineage Q5 pode ser liberada: df_reconciliado está materializado.
if df_q5_contexto_apresentacao.is_cached:
    df_q5_contexto_apresentacao.unpersist(blocking=False)
if df_ids_consumidos_exatos_cache.is_cached:
    df_ids_consumidos_exatos_cache.unpersist(blocking=False)
if df_pares_borda_calculados_cache.is_cached:
    df_pares_borda_calculados_cache.unpersist(blocking=False)

# Nome semântico oficial: inclui fallbacks V7 e somente movimentos efetivos.
df_mov_pos_classificacao_v7 = df_reconciliado.filter(
    (F.col('IN_JANELA') == 'S') & F.col('TIPO_CONSUMO').isNull()
)
df_mov_pos_classificacao_v7.createOrReplaceTempView('vw_mov_pos_classificacao_v7')


## D — Uma agregação financeira e fronteira única do motor

Flags, quantidades, valores temáticos, valores orçamentários, entradas realizadas
e métricas de reconciliação saem em uma única Row. A partir dela não existe nova
dependência das fontes corporativas para orçamento, pontuação ou cenários.


In [ ]:
%%spark
# Entradas Realizadas mantém deliberadamente o INNER JOIN da V7. A view é lazy,
# parte da raiz reconciliada materializada e será consumida pela única agregação.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_cenario_entradas_realizadas_detalhe AS
SELECT
    m.NR_TRAN_INST_PCT,
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    c.CD_GRUPO,
    c.TX_GRUPO,
    c.TX_CATEGORIA,
    c.CD_CLASS_RADAR,
    c.TX_CLASS_RADAR
FROM vw_reconciliado_v8 m
INNER JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
WHERE m.IN_JANELA = 'S'
  AND m.TIPO_CONSUMO IS NULL
  AND m.CD_NTZ_CTB_TRAN = 'C'
  AND m.CD_TIP_MOE_CRR = 'BRL'
""")

SQL_AGREGACAO_FINANCEIRA_V8 = """
SELECT
    CASE
        WHEN COUNT(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL THEN 1 END) = 0 THEN NULL
        WHEN COUNT(DISTINCT CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL THEN CD_TIP_MOE_CRR END) = 1
         AND MAX(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL THEN CD_TIP_MOE_CRR END) = 'BRL' THEN 'S'
        ELSE 'N'
    END AS FL_SOMENTE_BRL,
    CASE
        WHEN COUNT(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' THEN 1 END) = 0 THEN NULL
        WHEN COUNT(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND IN_AGRO = 'S' THEN 1 END) > 0 THEN 'S'
        ELSE 'N'
    END AS FL_TEM_MOV_AGRO,

    COUNT(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' THEN 1 END) AS QT_TRANS_TOTAL,
    SUM(CASE
        WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' THEN 1
        WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' THEN 0
    END) AS QT_TRANS_ENT,
    SUM(CASE
        WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' THEN 1
        WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' THEN 0
    END) AS QT_TRANS_SAI,

    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 1 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 2 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 3 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 0 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 4 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_CRED,

    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 5 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 6 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 7 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 8 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 9 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_OBR,

    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'C' AND IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_TOTAL,
    CAST(COALESCE(SUM(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL AND CD_TIP_MOE_CRR = 'BRL' AND CD_NTZ_CTB_TRAN = 'D' AND IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_TOTAL,

    CAST(COALESCE(
        (SELECT SUM(VL_TRAN) FROM vw_cenario_entradas_realizadas_detalhe),
        0.00
    ) AS DECIMAL(25,2)) AS ENTRADAS_REALIZADAS,

    COUNT(CASE WHEN IN_JANELA = 'S' THEN 1 END) AS QT_OFICIAIS,
    COUNT(CASE WHEN TIPO_CONSUMO = 'EXATO_OFICIAL' AND CD_NTZ_CTB_TRAN = 'C' THEN 1 END) AS QT_PARES_EXATOS,
    COUNT(CASE WHEN TIPO_CONSUMO = 'BORDA_OFICIAL' THEN 1 END) AS QT_PARES_BORDA,
    COUNT(CASE WHEN IN_JANELA = 'S' AND TIPO_CONSUMO IS NULL THEN 1 END) AS QT_EFETIVAS
FROM vw_reconciliado_v8
"""

print('[INICIO] AGREGACAO_FINANCEIRA_UNICA_MATERIALIZAR')
_inicio_action = time.perf_counter()
linha_agregada = spark.sql(SQL_AGREGACAO_FINANCEIRA_V8).first()
print(f'[FIM] AGREGACAO_FINANCEIRA_UNICA_MATERIALIZAR | {time.perf_counter() - _inicio_action:.3f}s')
if linha_agregada is None:
    raise RuntimeError('A agregação financeira V8 não produziu a Row contratual.')
agregados = linha_agregada.asDict(recursive=True)

# Gates de semântica de vazio/BRL herdados da V7.
if int(agregados['QT_TRANS_TOTAL']) == 0:
    if agregados['QT_TRANS_ENT'] is not None or agregados['QT_TRANS_SAI'] is not None:
        raise RuntimeError('Sem BRL, QT_TRANS_ENT e QT_TRANS_SAI devem permanecer NULL como na V7.')
    for campo in (
        'VL_ENT_REN', 'VL_ENT_EST', 'VL_ENT_RESG', 'VL_ENT_OUT', 'VL_ENT_CRED',
        'VL_SAI_IND', 'VL_SAI_ESS', 'VL_SAI_NAO_ESS', 'VL_SAI_FUT', 'VL_SAI_OBR',
        'VL_ENT_TOTAL', 'VL_SAI_TOTAL', 'ENTRADAS_REALIZADAS',
    ):
        if agregados[campo] != Decimal('0.00'):
            raise RuntimeError(f'Sem BRL, {campo} deve ser 0.00; obtido={agregados[campo]}.')
else:
    if int(agregados['QT_TRANS_TOTAL']) != int(agregados['QT_TRANS_ENT']) + int(agregados['QT_TRANS_SAI']):
        raise RuntimeError('QT_TRANS_TOTAL != QT_TRANS_ENT + QT_TRANS_SAI.')


In [ ]:
%%spark
# Motor escalar oficial. Decimal e ROUND_HALF_UP reproduzem ROUND(..., 6) da V7.
Q2_DECIMAL = Decimal('0.01')
Q6_DECIMAL = Decimal('0.000001')

def _decimal(valor):
    if valor is None or isinstance(valor, Decimal):
        return valor
    return Decimal(str(valor))

def _quantizar(valor, quantum):
    if valor is None:
        return None
    return _decimal(valor).quantize(quantum, rounding=ROUND_HALF_UP)

def _decimal_9_6(valor):
    if valor is None:
        return None
    quantizado = _quantizar(valor, Q6_DECIMAL)
    return None if abs(quantizado) >= Decimal('1000') else quantizado

def _dividir_6(numerador, denominador):
    numerador = _decimal(numerador)
    denominador = _decimal(denominador)
    if numerador is None or denominador is None or denominador == 0:
        return None
    with localcontext() as contexto:
        contexto.prec = 50
        return _decimal_9_6(numerador / denominador)

def calcular_motor_v8(agg, base_orcamento, base_percentuais, cd_macro_perfil):
    qt = agg.get('QT_TRANS_TOTAL')
    base_orcamento = _decimal(base_orcamento)
    base_percentuais = _decimal(base_percentuais)
    saida_total = _decimal(agg.get('VL_SAI_TOTAL'))

    vl_res_orc = None
    if base_orcamento is not None and saida_total is not None:
        vl_res_orc = _quantizar(base_orcamento - saida_total, Q2_DECIMAL)

    pc_sai_ent = None
    if qt != 0 and base_orcamento not in (None, Decimal('0')):
        pc_sai_ent = _dividir_6(saida_total, base_orcamento)

    if pc_sai_ent is None:
        faixa = None
    elif Decimal('0.950000') <= pc_sai_ent <= Decimal('1.050000'):
        faixa = 0
    elif Decimal('1.050000') < pc_sai_ent <= Decimal('1.250000'):
        faixa = 1
    elif pc_sai_ent > Decimal('1.250000'):
        faixa = 2
    elif Decimal('0.750000') <= pc_sai_ent < Decimal('0.950000'):
        faixa = 3
    else:
        faixa = 4

    if faixa is None:
        cd_res_orc = tx_res_orc = tx_sts_res = tx_sts_final = None
    elif faixa == 0:
        cd_res_orc, tx_res_orc, tx_sts_res, tx_sts_final = 0, 'Neutro', None, 'Neutro'
    elif faixa in (1, 2):
        cd_res_orc = 2
        tx_res_orc = 'Deficitário'
        tx_sts_res = 'Moderado' if faixa == 1 else 'Acentuado'
        tx_sts_final = 'Deficitário Moderado' if faixa == 1 else 'Deficitário Acentuado'
    else:
        cd_res_orc = 1
        tx_res_orc = 'Superavitário'
        tx_sts_res = 'Moderado' if faixa == 3 else 'Acentuado'
        tx_sts_final = 'Superavitário Moderado' if faixa == 3 else 'Superavitário Acentuado'

    refs = {
        'IND': Decimal('0.750000'),
        'ESS': Decimal('0.500000'),
        'NAO_ESS': Decimal('0.300000'),
        'FUT': Decimal('0.200000'),
        'OBR': Decimal('0.300000'),
    }
    campos_valores = {
        'IND': 'VL_SAI_IND', 'ESS': 'VL_SAI_ESS',
        'NAO_ESS': 'VL_SAI_NAO_ESS', 'FUT': 'VL_SAI_FUT',
        'OBR': 'VL_SAI_OBR',
    }
    percentuais = {
        tema: (
            None if base_percentuais is None or base_percentuais <= 0
            else _dividir_6(agg.get(campo), base_percentuais)
        )
        for tema, campo in campos_valores.items()
    }

    if qt is None or qt == 0 or base_percentuais is None:
        conc = {tema: None for tema in refs}
    elif base_percentuais <= 0:
        conc = {tema: 0 for tema in refs}
    else:
        pc = percentuais
        conc = {
            'IND': 99 if pc['IND'] is not None and pc['IND'] > Decimal('0.750000') else 0,
            'ESS': (
                0 if pc['ESS'] is not None and pc['ESS'] < Decimal('0.500000')
                else 1 if pc['ESS'] is not None and pc['ESS'] < Decimal('0.750000')
                else 2
            ),
            'NAO_ESS': (
                0 if pc['NAO_ESS'] is not None and pc['NAO_ESS'] < Decimal('0.300000')
                else 1 if pc['NAO_ESS'] is not None and pc['NAO_ESS'] < Decimal('0.450000')
                else 2
            ),
            'FUT': (
                0 if pc['FUT'] is not None and pc['FUT'] >= Decimal('0.300000')
                else 1 if pc['FUT'] is not None and pc['FUT'] >= Decimal('0.200000')
                else 2
            ),
            'OBR': (
                0 if pc['OBR'] is not None and pc['OBR'] < Decimal('0.300000')
                else 1 if pc['OBR'] is not None and pc['OBR'] < Decimal('0.450000')
                else 2
            ),
        }

    if qt is None or qt == 0:
        pont_orc = {tema: None for tema in refs}
    else:
        pont_orc = {'IND': 0}
        for tema in ('ESS', 'NAO_ESS', 'OBR'):
            pont_orc[tema] = None if faixa is None else (2 if faixa == 2 else 1 if faixa in (0, 1) else 0)
        pont_orc['FUT'] = None if faixa is None else (2 if faixa == 4 else 1 if faixa in (0, 3) else 0)

    perfil_valido = cd_macro_perfil in (1, 2, 3)
    pont_prfl = {'IND': None if qt is None or qt == 0 else 0}
    if qt is None or qt == 0 or not perfil_valido:
        pont_prfl.update({tema: None for tema in ('ESS', 'NAO_ESS', 'FUT', 'OBR')})
    else:
        pont_prfl.update({
            'ESS': 0 if cd_macro_perfil == 1 else 1,
            'NAO_ESS': 1 if cd_macro_perfil == 1 else 0,
            'FUT': 2 if cd_macro_perfil == 3 else 1 if cd_macro_perfil == 2 else 0,
            'OBR': 2 if cd_macro_perfil == 1 else 0,
        })

    finais = {'IND': conc['IND']}
    for tema in ('ESS', 'NAO_ESS', 'FUT', 'OBR'):
        parcelas = (conc[tema], pont_orc[tema], pont_prfl[tema])
        finais[tema] = None if any(valor is None for valor in parcelas) else sum(parcelas)

    completa = 'S' if all(finais[tema] is not None for tema in ('IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR')) else 'N'
    if completa == 'N':
        pont_max = qt_temas_max = cd_tema = tx_tema = None
    else:
        ordem = ('IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR')
        codigos = {'IND': 1, 'ESS': 2, 'NAO_ESS': 3, 'FUT': 4, 'OBR': 5}
        rotulos = {
            'IND': 'Categorização dos Gastos',
            'ESS': 'Gestão de Orçamento',
            'NAO_ESS': 'Consumo Planejado',
            'FUT': 'Formação de Reserva',
            'OBR': 'Uso Consciente do Crédito',
        }
        pont_max = max(finais.values())
        vencedores = [tema for tema in ordem if finais[tema] == pont_max]
        qt_temas_max = len(vencedores)
        if qt_temas_max > 1:
            cd_tema, tx_tema = 9, 'Empate'
        else:
            tema = vencedores[0]
            cd_tema, tx_tema = codigos[tema], rotulos[tema]

    resultado = {
        'VL_RES_ORC': vl_res_orc,
        'PC_SAI_ENT': pc_sai_ent,
        'CD_RES_ORC': cd_res_orc,
        'TX_RES_ORC': tx_res_orc,
        'CD_FAIXA_ORC': faixa,
        'TX_STS_RES': tx_sts_res,
        'TX_STS_FINAL': tx_sts_final,
    }
    for tema in ('IND', 'ESS', 'NAO_ESS', 'FUT', 'OBR'):
        resultado[f'PC_SAI_{tema}'] = percentuais[tema]
        resultado[f'PC_REF_{tema}'] = refs[tema]
        resultado[f'NR_PONT_CONC_{tema}'] = conc[tema]
        resultado[f'NR_PONT_ORC_{tema}'] = pont_orc[tema]
        resultado[f'NR_PONT_PRFL_{tema}'] = pont_prfl[tema]
        resultado[f'NR_PONT_{tema}_FIM'] = finais[tema]
    resultado.update({
        'FL_PONTUACAO_COMPLETA': completa,
        'NR_PONT_MAX': pont_max,
        'QT_TEMAS_PONT_MAX': qt_temas_max,
        'CD_TEMA_VENCEDOR': cd_tema,
        'TX_TEMA_VENCEDOR': tx_tema,
    })
    return resultado


In [ ]:
%%spark
# Fixtures locais do motor: nulos, zeros, arredondamento HALF_UP, somente saídas,
# categorias sem match e limites das faixas orçamentárias.
def _agg_fixture(saida_total='0.00', qt=1):
    return {
        'QT_TRANS_TOTAL': qt,
        'VL_SAI_TOTAL': Decimal(saida_total),
        'VL_SAI_IND': Decimal('0.00'),
        'VL_SAI_ESS': Decimal('0.00'),
        'VL_SAI_NAO_ESS': Decimal('0.00'),
        'VL_SAI_FUT': Decimal('0.00'),
        'VL_SAI_OBR': Decimal('0.00'),
    }

assert _dividir_6(Decimal('1.00'), Decimal('128.00')) == Decimal('0.007813')
assert calcular_motor_v8(_agg_fixture('95.00'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 0
assert calcular_motor_v8(_agg_fixture('105.00'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 0
assert calcular_motor_v8(_agg_fixture('105.01'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 1
assert calcular_motor_v8(_agg_fixture('125.00'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 1
assert calcular_motor_v8(_agg_fixture('125.01'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 2
assert calcular_motor_v8(_agg_fixture('75.00'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 3
assert calcular_motor_v8(_agg_fixture('74.99'), Decimal('100.00'), Decimal('100.00'), 2)['CD_FAIXA_ORC'] == 4

fixture_zero = calcular_motor_v8(_agg_fixture('0.00', qt=0), Decimal('0.00'), None, None)
assert fixture_zero['PC_SAI_ENT'] is None and fixture_zero['FL_PONTUACAO_COMPLETA'] == 'N'
fixture_so_saidas = calcular_motor_v8(_agg_fixture('100.00'), Decimal('0.00'), Decimal('100.00'), 2)
assert fixture_so_saidas['VL_RES_ORC'] == Decimal('-100.00')
assert fixture_so_saidas['PC_SAI_ENT'] is None
assert FALLBACK_CLASSIFICACAO_V7 == {
    'TX_CATEGORIA': 'Sem Categoria', 'CD_CLASS_RADAR': 0,
    'TX_CLASS_RADAR': 'Outras Entradas', 'IN_AGRO': 'N',
    'IN_PARTICIPA_CALCULO': 'N', 'IN_PARTICIPA_ORCAMENTO': 'N',
}
print('[RADAR_V8] Fixtures locais do motor concluídas.')


## E — Contrato 1×80 e cenários

Produção cria dinamicamente somente `df_final` a partir do dicionário ordenado de
80 campos. A view pública aponta para o mesmo DataFrame. Os cenários são dicionários
laterais: apenas 31 campos derivados podem variar.


In [ ]:
%%spark
SCHEMA_RESULTADO_80 = StructType([
    StructField('CD_CLI', IntegerType(), False),
    StructField('DT_EXEA', DateType(), False),
    StructField('DT_MES_EXEA', DateType(), False),
    StructField('TS_INCL_TRAN_REF', TimestampType(), False),
    StructField('FL_CPF_UNICO', StringType(), False),
    StructField('CD_CPF', DecimalType(14, 0), True),
    StructField('FL_CONTA_ELEGIVEL_UNICA', StringType(), False),
    StructField('TS_DD_INC_MM_CLC_BLC_REF', TimestampType(), True),
    StructField('DD_INC_MM_CLC_BLC', ShortType(), True),
    StructField('DD_INC_MM_CLC_BLC_FALLBACK', ShortType(), True),
    StructField('DT_REN_PRES_REF', DateType(), True),
    StructField('VL_REN_PRES', DecimalType(17, 2), True),
    StructField('DT_REF_PRFL', DateType(), True),
    StructField('CD_MAC_PRFL_CLI', IntegerType(), True),
    StructField('NM_MAC_PRFL_CLI', StringType(), True),
    StructField('CD_MIC_PRFL_CLI', IntegerType(), True),
    StructField('NM_MIC_PRFL_CLI', StringType(), True),
    StructField('DT_REF_INI', DateType(), True),
    StructField('DT_REF_FIM', DateType(), True),
    StructField('FL_SOMENTE_BRL', StringType(), True),
    StructField('FL_TEM_MOV_AGRO', StringType(), True),
    StructField('QT_TRANS_TOTAL', LongType(), True),
    StructField('QT_TRANS_ENT', LongType(), True),
    StructField('QT_TRANS_SAI', LongType(), True),
    StructField('VL_TRANS_ENT', DecimalType(25, 2), True),
    StructField('VL_TRANS_SAI', DecimalType(25, 2), True),
    StructField('VL_ENT_REN', DecimalType(25, 2), True),
    StructField('VL_ENT_EST', DecimalType(25, 2), True),
    StructField('VL_ENT_RESG', DecimalType(25, 2), True),
    StructField('VL_ENT_OUT', DecimalType(25, 2), True),
    StructField('VL_ENT_CRED', DecimalType(25, 2), True),
    StructField('VL_ENT_TOTAL', DecimalType(25, 2), True),
    StructField('VL_SAI_IND', DecimalType(25, 2), True),
    StructField('VL_SAI_ESS', DecimalType(25, 2), True),
    StructField('VL_SAI_NAO_ESS', DecimalType(25, 2), True),
    StructField('VL_SAI_FUT', DecimalType(25, 2), True),
    StructField('VL_SAI_OBR', DecimalType(25, 2), True),
    StructField('VL_SAI_TOTAL', DecimalType(25, 2), True),
    StructField('VL_RES_ORC', DecimalType(25, 2), True),
    StructField('PC_SAI_ENT', DecimalType(9, 6), True),
    StructField('CD_RES_ORC', IntegerType(), True),
    StructField('TX_RES_ORC', StringType(), True),
    StructField('CD_FAIXA_ORC', IntegerType(), True),
    StructField('TX_STS_RES', StringType(), True),
    StructField('TX_STS_FINAL', StringType(), True),
    StructField('PC_SAI_IND', DecimalType(9, 6), True),
    StructField('PC_SAI_ESS', DecimalType(9, 6), True),
    StructField('PC_SAI_NAO_ESS', DecimalType(9, 6), True),
    StructField('PC_SAI_FUT', DecimalType(9, 6), True),
    StructField('PC_SAI_OBR', DecimalType(9, 6), True),
    StructField('PC_REF_IND', DecimalType(9, 6), False),
    StructField('PC_REF_ESS', DecimalType(9, 6), False),
    StructField('PC_REF_NAO_ESS', DecimalType(9, 6), False),
    StructField('PC_REF_FUT', DecimalType(9, 6), False),
    StructField('PC_REF_OBR', DecimalType(9, 6), False),
    StructField('NR_PONT_CONC_IND', IntegerType(), True),
    StructField('NR_PONT_CONC_ESS', IntegerType(), True),
    StructField('NR_PONT_CONC_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_CONC_FUT', IntegerType(), True),
    StructField('NR_PONT_CONC_OBR', IntegerType(), True),
    StructField('NR_PONT_ORC_IND', IntegerType(), True),
    StructField('NR_PONT_ORC_ESS', IntegerType(), True),
    StructField('NR_PONT_ORC_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_ORC_FUT', IntegerType(), True),
    StructField('NR_PONT_ORC_OBR', IntegerType(), True),
    StructField('NR_PONT_PRFL_IND', IntegerType(), True),
    StructField('NR_PONT_PRFL_ESS', IntegerType(), True),
    StructField('NR_PONT_PRFL_NAO_ESS', IntegerType(), True),
    StructField('NR_PONT_PRFL_FUT', IntegerType(), True),
    StructField('NR_PONT_PRFL_OBR', IntegerType(), True),
    StructField('NR_PONT_IND_FIM', IntegerType(), True),
    StructField('NR_PONT_ESS_FIM', IntegerType(), True),
    StructField('NR_PONT_NAO_ESS_FIM', IntegerType(), True),
    StructField('NR_PONT_FUT_FIM', IntegerType(), True),
    StructField('NR_PONT_OBR_FIM', IntegerType(), True),
    StructField('FL_PONTUACAO_COMPLETA', StringType(), False),
    StructField('NR_PONT_MAX', IntegerType(), True),
    StructField('QT_TEMAS_PONT_MAX', IntegerType(), True),
    StructField('CD_TEMA_VENCEDOR', IntegerType(), True),
    StructField('TX_TEMA_VENCEDOR', StringType(), True),
])

CAMPOS_ORCAMENTO_CENARIO = [
    'VL_RES_ORC', 'PC_SAI_ENT', 'CD_RES_ORC', 'TX_RES_ORC',
    'CD_FAIXA_ORC', 'TX_STS_RES', 'TX_STS_FINAL',
]
CAMPOS_MOTOR_CENARIO = [
    'PC_SAI_IND', 'PC_SAI_ESS', 'PC_SAI_NAO_ESS', 'PC_SAI_FUT', 'PC_SAI_OBR',
    'NR_PONT_CONC_IND', 'NR_PONT_CONC_ESS', 'NR_PONT_CONC_NAO_ESS',
    'NR_PONT_CONC_FUT', 'NR_PONT_CONC_OBR',
    'NR_PONT_ORC_ESS', 'NR_PONT_ORC_NAO_ESS', 'NR_PONT_ORC_FUT', 'NR_PONT_ORC_OBR',
    'NR_PONT_IND_FIM', 'NR_PONT_ESS_FIM', 'NR_PONT_NAO_ESS_FIM', 'NR_PONT_FUT_FIM',
    'NR_PONT_OBR_FIM', 'FL_PONTUACAO_COMPLETA', 'NR_PONT_MAX', 'QT_TEMAS_PONT_MAX',
    'CD_TEMA_VENCEDOR', 'TX_TEMA_VENCEDOR',
]
CAMPOS_VARIAVEIS_CENARIO = set(CAMPOS_ORCAMENTO_CENARIO + CAMPOS_MOTOR_CENARIO)
if len(CAMPOS_VARIAVEIS_CENARIO) != 31:
    raise RuntimeError('O contrato lateral deve possuir exatamente 31 campos variáveis.')


In [ ]:
%%spark
motor_oficial = calcular_motor_v8(
    agregados,
    agregados['VL_ENT_TOTAL'],
    res_renda['VL_REN_PRES'],
    res_prfl['CD_MAC_PRFL_CLI'],
)

resultado_bruto = {
    'CD_CLI': CD_CLI,
    'DT_EXEA': DATA_EXECUCAO,
    'DT_MES_EXEA': DT_MES_EXEA,
    'TS_INCL_TRAN_REF': estado_cliente['TS_INCL_TRAN_REF'],
    'FL_CPF_UNICO': fl_cpf_unico,
    'CD_CPF': cd_cpf,
    'FL_CONTA_ELEGIVEL_UNICA': fl_conta_unica,
    'TS_DD_INC_MM_CLC_BLC_REF': row_ciclo['TS_DD_INC_MM_CLC_BLC_REF'] if row_ciclo else None,
    'DD_INC_MM_CLC_BLC': dd_ciclo_val,
    'DD_INC_MM_CLC_BLC_FALLBACK': dd_fallback_val,
    'DT_REN_PRES_REF': res_renda['DT_REN_PRES_REF'],
    'VL_REN_PRES': res_renda['VL_REN_PRES'],
    'DT_REF_PRFL': res_prfl['DT_REF_PRFL'],
    'CD_MAC_PRFL_CLI': res_prfl['CD_MAC_PRFL_CLI'],
    'NM_MAC_PRFL_CLI': res_prfl['NM_MAC_PRFL_CLI'],
    'CD_MIC_PRFL_CLI': res_prfl['CD_MIC_PRFL_CLI'],
    'NM_MIC_PRFL_CLI': res_prfl['NM_MIC_PRFL_CLI'],
    'DT_REF_INI': dt_ref_ini_val,
    'DT_REF_FIM': dt_ref_fim_val,
    'FL_SOMENTE_BRL': agregados['FL_SOMENTE_BRL'],
    'FL_TEM_MOV_AGRO': agregados['FL_TEM_MOV_AGRO'],
    'QT_TRANS_TOTAL': int(agregados['QT_TRANS_TOTAL']),
    'QT_TRANS_ENT': None if agregados['QT_TRANS_ENT'] is None else int(agregados['QT_TRANS_ENT']),
    'QT_TRANS_SAI': None if agregados['QT_TRANS_SAI'] is None else int(agregados['QT_TRANS_SAI']),
    'VL_TRANS_ENT': agregados['VL_ENT_TOTAL'],
    'VL_TRANS_SAI': agregados['VL_SAI_TOTAL'],
    'VL_ENT_REN': agregados['VL_ENT_REN'],
    'VL_ENT_EST': agregados['VL_ENT_EST'],
    'VL_ENT_RESG': agregados['VL_ENT_RESG'],
    'VL_ENT_OUT': agregados['VL_ENT_OUT'],
    'VL_ENT_CRED': agregados['VL_ENT_CRED'],
    'VL_ENT_TOTAL': agregados['VL_ENT_TOTAL'],
    'VL_SAI_IND': agregados['VL_SAI_IND'],
    'VL_SAI_ESS': agregados['VL_SAI_ESS'],
    'VL_SAI_NAO_ESS': agregados['VL_SAI_NAO_ESS'],
    'VL_SAI_FUT': agregados['VL_SAI_FUT'],
    'VL_SAI_OBR': agregados['VL_SAI_OBR'],
    'VL_SAI_TOTAL': agregados['VL_SAI_TOTAL'],
}
resultado_bruto.update(motor_oficial)

nomes_resultado_80 = [campo.name for campo in SCHEMA_RESULTADO_80.fields]
ausentes = [nome for nome in nomes_resultado_80 if nome not in resultado_bruto]
extras = [nome for nome in resultado_bruto if nome not in nomes_resultado_80]
if ausentes or extras or len(nomes_resultado_80) != 80:
    raise RuntimeError(f'Contrato 80 inválido: ausentes={ausentes}, extras={extras}.')

resultado_oficial = {nome: resultado_bruto[nome] for nome in nomes_resultado_80}
for campo in SCHEMA_RESULTADO_80.fields:
    valor = resultado_oficial[campo.name]
    if isinstance(campo.dataType, DecimalType) and valor is not None and not isinstance(valor, Decimal):
        raise TypeError(f'{campo.name} deve ser Decimal; recebido={type(valor).__name__}.')

# Única criação dinâmica produtiva 1×N no driver.
df_final = spark.createDataFrame([resultado_oficial], schema=SCHEMA_RESULTADO_80)
df_final.createOrReplaceTempView(VIEW_RESULTADO)

bases_cenarios = {
    'RENDA_PRESUMIDA': (
        None if dt_ref_ini_val is None or dt_ref_fim_val is None
        else res_renda['VL_REN_PRES']
    ),
    'ENTRADAS_REALIZADAS': (
        None if dt_ref_ini_val is None or dt_ref_fim_val is None
        else agregados['ENTRADAS_REALIZADAS']
    ),
}
resultados_cenarios = {}
for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS'):
    motor_cenario = calcular_motor_v8(
        agregados,
        bases_cenarios[chave],
        bases_cenarios[chave],
        res_prfl['CD_MAC_PRFL_CLI'],
    )
    snapshot = dict(resultado_oficial)
    for campo in CAMPOS_VARIAVEIS_CENARIO:
        snapshot[campo] = motor_cenario[campo]
    resultados_cenarios[chave] = snapshot

campos_invariantes_cenario = set(nomes_resultado_80) - CAMPOS_VARIAVEIS_CENARIO
for chave, snapshot in resultados_cenarios.items():
    if list(snapshot) != nomes_resultado_80:
        raise RuntimeError(f'Cenário {chave} alterou ordem/schema do contrato.')
    divergentes = [
        campo for campo in campos_invariantes_cenario
        if snapshot[campo] != resultado_oficial[campo]
    ]
    if divergentes:
        raise RuntimeError(f'Cenário {chave} alterou campos invariantes: {divergentes}.')

res_dict = dict(resultados_cenarios['RENDA_PRESUMIDA'])


## F — Homologação SQL temporária

`MODO_HOMOLOGACAO_SQL` é uma exceção isolada. Quando ativado no ambiente
corporativo, cria uma única entrada/view 1×N a partir da mesma Row agregada e
compara o motor Spark SQL com o motor escalar. Não relê fontes. O bloco inteiro
deve ser removido após o aceite 80/80.


In [ ]:
%%spark
# Produção: False. Homologação controlada: alterar temporariamente para True.
MODO_HOMOLOGACAO_SQL = False

SCHEMA_ENTRADA_HOMOLOGACAO = StructType([
    StructField('QT_TRANS_TOTAL', LongType(), False),
    StructField('VL_ENT_BASE', DecimalType(25, 2), True),
    StructField('VL_REN_BASE', DecimalType(25, 2), True),
    StructField('VL_SAI_TOTAL', DecimalType(25, 2), False),
    StructField('VL_SAI_IND', DecimalType(25, 2), False),
    StructField('VL_SAI_ESS', DecimalType(25, 2), False),
    StructField('VL_SAI_NAO_ESS', DecimalType(25, 2), False),
    StructField('VL_SAI_FUT', DecimalType(25, 2), False),
    StructField('VL_SAI_OBR', DecimalType(25, 2), False),
    StructField('CD_MAC_PRFL_CLI', IntegerType(), True),
])

SQL_REFERENCIA_MOTOR_V8 = r"""
WITH calc AS (
    SELECT *,
        CAST(VL_ENT_BASE - VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_RES_ORC,
        CASE WHEN QT_TRANS_TOTAL = 0 OR VL_ENT_BASE = 0 THEN NULL
             ELSE CAST(ROUND(VL_SAI_TOTAL / VL_ENT_BASE, 6) AS DECIMAL(9,6)) END AS PC_SAI_ENT,
        CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
        CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
        CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
        CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
        CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,
        CASE WHEN VL_REN_BASE IS NULL OR VL_REN_BASE <= 0 THEN NULL ELSE CAST(ROUND(VL_SAI_IND / VL_REN_BASE, 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
        CASE WHEN VL_REN_BASE IS NULL OR VL_REN_BASE <= 0 THEN NULL ELSE CAST(ROUND(VL_SAI_ESS / VL_REN_BASE, 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
        CASE WHEN VL_REN_BASE IS NULL OR VL_REN_BASE <= 0 THEN NULL ELSE CAST(ROUND(VL_SAI_NAO_ESS / VL_REN_BASE, 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
        CASE WHEN VL_REN_BASE IS NULL OR VL_REN_BASE <= 0 THEN NULL ELSE CAST(ROUND(VL_SAI_FUT / VL_REN_BASE, 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
        CASE WHEN VL_REN_BASE IS NULL OR VL_REN_BASE <= 0 THEN NULL ELSE CAST(ROUND(VL_SAI_OBR / VL_REN_BASE, 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
    FROM _vw_homologacao_motor_v8
), faixa AS (
    SELECT *, CASE
        WHEN PC_SAI_ENT IS NULL THEN NULL
        WHEN PC_SAI_ENT BETWEEN 0.950000 AND 1.050000 THEN 0
        WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
        WHEN PC_SAI_ENT > 1.250000 THEN 2
        WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
        ELSE 4 END AS CD_FAIXA_ORC
    FROM calc
), status AS (
    SELECT *,
        CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 0 THEN 0 WHEN CD_FAIXA_ORC IN (1,2) THEN 2 ELSE 1 END AS CD_RES_ORC,
        CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 0 THEN 'Neutro' WHEN CD_FAIXA_ORC IN (1,2) THEN 'Deficitário' ELSE 'Superavitário' END AS TX_RES_ORC,
        CASE WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL WHEN CD_FAIXA_ORC IN (1,3) THEN 'Moderado' ELSE 'Acentuado' END AS TX_STS_RES,
        CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 0 THEN 'Neutro' WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado' WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado' WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado' ELSE 'Superavitário Acentuado' END AS TX_STS_FINAL
    FROM faixa
), conc AS (
    SELECT *,
        CASE WHEN QT_TRANS_TOTAL = 0 OR VL_REN_BASE IS NULL THEN NULL WHEN VL_REN_BASE <= 0 THEN 0 WHEN PC_SAI_IND > 0.750000 THEN 99 ELSE 0 END AS NR_PONT_CONC_IND,
        CASE WHEN QT_TRANS_TOTAL = 0 OR VL_REN_BASE IS NULL THEN NULL WHEN VL_REN_BASE <= 0 THEN 0 WHEN PC_SAI_ESS < 0.500000 THEN 0 WHEN PC_SAI_ESS < 0.750000 THEN 1 ELSE 2 END AS NR_PONT_CONC_ESS,
        CASE WHEN QT_TRANS_TOTAL = 0 OR VL_REN_BASE IS NULL THEN NULL WHEN VL_REN_BASE <= 0 THEN 0 WHEN PC_SAI_NAO_ESS < 0.300000 THEN 0 WHEN PC_SAI_NAO_ESS < 0.450000 THEN 1 ELSE 2 END AS NR_PONT_CONC_NAO_ESS,
        CASE WHEN QT_TRANS_TOTAL = 0 OR VL_REN_BASE IS NULL THEN NULL WHEN VL_REN_BASE <= 0 THEN 0 WHEN PC_SAI_FUT >= 0.300000 THEN 0 WHEN PC_SAI_FUT >= 0.200000 THEN 1 ELSE 2 END AS NR_PONT_CONC_FUT,
        CASE WHEN QT_TRANS_TOTAL = 0 OR VL_REN_BASE IS NULL THEN NULL WHEN VL_REN_BASE <= 0 THEN 0 WHEN PC_SAI_OBR < 0.300000 THEN 0 WHEN PC_SAI_OBR < 0.450000 THEN 1 ELSE 2 END AS NR_PONT_CONC_OBR
    FROM status
), pont_orc AS (
    SELECT *,
        CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 2 THEN 2 WHEN CD_FAIXA_ORC IN (0,1) THEN 1 ELSE 0 END AS NR_PONT_ORC_ESS,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 2 THEN 2 WHEN CD_FAIXA_ORC IN (0,1) THEN 1 ELSE 0 END AS NR_PONT_ORC_NAO_ESS,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 4 THEN 2 WHEN CD_FAIXA_ORC IN (0,3) THEN 1 ELSE 0 END AS NR_PONT_ORC_FUT,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL WHEN CD_FAIXA_ORC = 2 THEN 2 WHEN CD_FAIXA_ORC IN (0,1) THEN 1 ELSE 0 END AS NR_PONT_ORC_OBR
    FROM conc
), pont_prfl AS (
    SELECT *,
        CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI = 1 THEN 0 ELSE 1 END AS NR_PONT_PRFL_ESS,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI = 1 THEN 1 ELSE 0 END AS NR_PONT_PRFL_NAO_ESS,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI = 3 THEN 2 WHEN CD_MAC_PRFL_CLI = 2 THEN 1 ELSE 0 END AS NR_PONT_PRFL_FUT,
        CASE WHEN QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI NOT IN (1,2,3) OR CD_MAC_PRFL_CLI IS NULL THEN NULL WHEN CD_MAC_PRFL_CLI = 1 THEN 2 ELSE 0 END AS NR_PONT_PRFL_OBR
    FROM pont_orc
), finais AS (
    SELECT *,
        NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
        CASE WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS END AS NR_PONT_ESS_FIM,
        CASE WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS END AS NR_PONT_NAO_ESS_FIM,
        CASE WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT END AS NR_PONT_FUT_FIM,
        CASE WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR END AS NR_PONT_OBR_FIM
    FROM pont_prfl
), completo AS (
    SELECT *, CASE WHEN NR_PONT_IND_FIM IS NOT NULL AND NR_PONT_ESS_FIM IS NOT NULL AND NR_PONT_NAO_ESS_FIM IS NOT NULL AND NR_PONT_FUT_FIM IS NOT NULL AND NR_PONT_OBR_FIM IS NOT NULL THEN 'S' ELSE 'N' END AS FL_PONTUACAO_COMPLETA
    FROM finais
), maximo AS (
    SELECT *, CASE WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL ELSE GREATEST(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM) END AS NR_PONT_MAX
    FROM completo
), vencedores AS (
    SELECT *, CASE WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL ELSE
        (CASE WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1 ELSE 0 END + CASE WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END + CASE WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END + CASE WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 1 ELSE 0 END + CASE WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 1 ELSE 0 END) END AS QT_TEMAS_PONT_MAX
    FROM maximo
)
SELECT
    VL_RES_ORC, PC_SAI_ENT, CD_RES_ORC, TX_RES_ORC, CD_FAIXA_ORC, TX_STS_RES, TX_STS_FINAL,
    PC_SAI_IND, PC_SAI_ESS, PC_SAI_NAO_ESS, PC_SAI_FUT, PC_SAI_OBR,
    PC_REF_IND, PC_REF_ESS, PC_REF_NAO_ESS, PC_REF_FUT, PC_REF_OBR,
    NR_PONT_CONC_IND, NR_PONT_CONC_ESS, NR_PONT_CONC_NAO_ESS, NR_PONT_CONC_FUT, NR_PONT_CONC_OBR,
    NR_PONT_ORC_IND, NR_PONT_ORC_ESS, NR_PONT_ORC_NAO_ESS, NR_PONT_ORC_FUT, NR_PONT_ORC_OBR,
    NR_PONT_PRFL_IND, NR_PONT_PRFL_ESS, NR_PONT_PRFL_NAO_ESS, NR_PONT_PRFL_FUT, NR_PONT_PRFL_OBR,
    NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM,
    FL_PONTUACAO_COMPLETA, NR_PONT_MAX, QT_TEMAS_PONT_MAX,
    CASE WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL WHEN QT_TEMAS_PONT_MAX > 1 THEN 9 WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1 WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2 WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 3 WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4 ELSE 5 END AS CD_TEMA_VENCEDOR,
    CASE WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL WHEN QT_TEMAS_PONT_MAX > 1 THEN 'Empate' WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 'Categorização dos Gastos' WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 'Gestão de Orçamento' WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 'Consumo Planejado' WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 'Formação de Reserva' ELSE 'Uso Consciente do Crédito' END AS TX_TEMA_VENCEDOR
FROM vencedores
"""

def calcular_referencia_sql_v8(base_orcamento, base_percentuais):
    entrada = {
        'QT_TRANS_TOTAL': int(agregados['QT_TRANS_TOTAL']),
        'VL_ENT_BASE': base_orcamento,
        'VL_REN_BASE': base_percentuais,
        'VL_SAI_TOTAL': agregados['VL_SAI_TOTAL'],
        'VL_SAI_IND': agregados['VL_SAI_IND'],
        'VL_SAI_ESS': agregados['VL_SAI_ESS'],
        'VL_SAI_NAO_ESS': agregados['VL_SAI_NAO_ESS'],
        'VL_SAI_FUT': agregados['VL_SAI_FUT'],
        'VL_SAI_OBR': agregados['VL_SAI_OBR'],
        'CD_MAC_PRFL_CLI': res_prfl['CD_MAC_PRFL_CLI'],
    }
    df_homologacao = spark.createDataFrame([entrada], SCHEMA_ENTRADA_HOMOLOGACAO)
    df_homologacao.createOrReplaceTempView('_vw_homologacao_motor_v8')
    linha = spark.sql(SQL_REFERENCIA_MOTOR_V8).first()
    if linha is None:
        raise RuntimeError('Motor SQL de homologação não produziu uma linha.')
    return linha.asDict(recursive=True)

if MODO_HOMOLOGACAO_SQL:
    referencias = {
        'OFICIAL': calcular_referencia_sql_v8(agregados['VL_ENT_TOTAL'], res_renda['VL_REN_PRES']),
        'RENDA_PRESUMIDA': calcular_referencia_sql_v8(bases_cenarios['RENDA_PRESUMIDA'], bases_cenarios['RENDA_PRESUMIDA']),
        'ENTRADAS_REALIZADAS': calcular_referencia_sql_v8(bases_cenarios['ENTRADAS_REALIZADAS'], bases_cenarios['ENTRADAS_REALIZADAS']),
    }
    esperado_oficial = dict(resultado_oficial)
    esperado_oficial.update(referencias['OFICIAL'])
    snapshots_sql = {'OFICIAL': esperado_oficial}
    for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS'):
        snapshot = dict(esperado_oficial)
        for campo in CAMPOS_VARIAVEIS_CENARIO:
            snapshot[campo] = referencias[chave][campo]
        snapshots_sql[chave] = snapshot
    snapshots_escalares = {'OFICIAL': resultado_oficial, **resultados_cenarios}
    for chave, esperado in snapshots_sql.items():
        divergencias = [
            campo for campo in nomes_resultado_80
            if esperado[campo] != snapshots_escalares[chave][campo]
        ]
        if divergencias:
            raise RuntimeError(f'Homologação SQL {chave} divergiu: {divergencias}.')
    spark.catalog.dropTempView('_vw_homologacao_motor_v8')
    print('[RADAR_V8] Homologação SQL 80/80 concluída nos três snapshots.')


## G — Gates finais e adaptador do dashboard

O bundle visual deriva exclusivamente de `df_reconciliado`, já materializado. O
renderer HTML/CSS/JS é copiado literalmente da V7; somente este adaptador muda.


In [ ]:
%%spark
# Gates do contrato público e do plano de composição 1×80.
if df_final.schema != SCHEMA_RESULTADO_80 or df_final.columns != nomes_resultado_80:
    raise RuntimeError('df_final divergiu do schema explícito de 80 colunas.')

print('[INICIO] RESULTADO_V8_VALIDAR_1X80_LOCAL')
_inicio_action = time.perf_counter()
linhas_df_final = df_final.limit(2).collect()
linhas_view_final = spark.table(VIEW_RESULTADO).limit(2).collect()
print(f'[FIM] RESULTADO_V8_VALIDAR_1X80_LOCAL | {time.perf_counter() - _inicio_action:.3f}s')
if len(linhas_df_final) != 1 or len(linhas_view_final) != 1:
    raise RuntimeError('df_final e view devem possuir exatamente uma linha.')
if linhas_df_final[0].asDict(recursive=True) != linhas_view_final[0].asDict(recursive=True):
    raise RuntimeError('A view pública não representa o mesmo conteúdo de df_final.')

plano_resultado_1x80 = df_final._jdf.queryExecution().executedPlan().toString()
if 'CartesianProduct' in plano_resultado_1x80:
    raise RuntimeError('CartesianProduct detectado na composição do resultado 1×80.')

if contagem_leituras_funcionais['Q1'] != 1 or contagem_leituras_funcionais['Q4'] != 1:
    raise RuntimeError(f'Leituras obrigatórias inesperadas: {contagem_leituras_funcionais}.')
if contagem_leituras_funcionais['Q2'] != (1 if tem_conta_norm else 0):
    raise RuntimeError('Contagem funcional Q2 incompatível com disponibilidade da conta.')
if contagem_leituras_funcionais['Q3'] != (1 if fl_cpf_unico == 'S' and cd_cpf is not None else 0):
    raise RuntimeError('Contagem funcional Q3 incompatível com disponibilidade do CPF.')
if contagem_leituras_funcionais['Q5'] != (1 if dt_ini_j is not None else 0):
    raise RuntimeError('Contagem funcional Q5 incompatível com disponibilidade da janela.')
print(f'[RADAR_V8] Leituras funcionais: {contagem_leituras_funcionais}.')


In [ ]:
%%spark
df_dashboard_transacoes = df_reconciliado.filter(
    (F.col('IN_JANELA') == 'S')
    & F.col('TIPO_CONSUMO').isNull()
    & (F.col('CD_TIP_MOE_CRR') == 'BRL')
).select(
    'NR_TRAN_INST_PCT', 'CD_CLI', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN', 'IN_JANELA',
    'TX_DCR_TRAN_OGNL', 'NR_MCA_PCT_OPB', 'CD_GRUPO', 'TX_GRUPO',
    'CD_IR', 'TX_IR', 'TX_CATEGORIA',
    F.col('CD_CLASS_RADAR_MAPA').alias('CD_CLASS_RADAR'),
    F.col('TX_CLASS_RADAR_MAPA').alias('TX_CLASS_RADAR'),
    'IN_CLASSIFICADA_APRESENTACAO', 'IN_AGRO', 'IN_PARTICIPA_CALCULO',
    'IN_PARTICIPA_ORCAMENTO', 'TIPO_CONSUMO',
)

df_dashboard_pivot = df_dashboard_transacoes.groupBy(
    F.col('CD_NTZ_CTB_TRAN').alias('ntz'),
    F.col('CD_CLASS_RADAR').alias('cd_classe'),
    F.col('TX_CLASS_RADAR').alias('tx_classe'),
    F.col('IN_CLASSIFICADA_APRESENTACAO').alias('classificada'),
    F.col('CD_CTGR_TRAN_OGNL').alias('cd_cat'),
    F.col('TX_CATEGORIA').alias('tx_cat'),
    F.col('IN_PARTICIPA_CALCULO').alias('part_calc'),
    F.col('IN_PARTICIPA_ORCAMENTO').alias('part_orc'),
).agg(
    F.count(F.lit(1)).alias('qt'),
    F.sum('VL_TRAN').alias('vl_mov'),
    F.sum(F.when(F.col('IN_PARTICIPA_CALCULO') == 'S', F.col('VL_TRAN')).otherwise(F.lit(0))).alias('vl_tematico'),
    F.sum(F.when(F.col('IN_PARTICIPA_ORCAMENTO') == 'S', F.col('VL_TRAN')).otherwise(F.lit(0))).alias('vl_orcamentario'),
)

df_dashboard_exatos = df_reconciliado.filter(
    F.col('TIPO_CONSUMO').isin('EXATO_OFICIAL', 'EXATO_CONTEXTO')
    & (F.col('CD_NTZ_CTB_TRAN') == 'C')
).select(
    'TIPO_CONSUMO', 'IN_JANELA',
    F.col('NR_TRAN_INST_PCT').alias('ID_CREDITO'),
    F.col('NR_TRAN_CONTRAPARTE').alias('ID_DEBITO'),
    'DT_TRAN', 'VL_TRAN', 'CD_TIP_MOE_CRR',
    F.col('TX_DCR_TRAN_OGNL').alias('DESC_CREDITO'),
    F.col('NR_MCA_PCT_OPB').alias('BANCO_CREDITO'),
    F.col('TX_DCR_TRAN_OGNL_CONTRAPARTE').alias('DESC_DEBITO'),
    F.col('NR_MCA_PCT_OPB_CONTRAPARTE').alias('BANCO_DEBITO'),
)

df_dashboard_borda = df_reconciliado.filter(
    F.col('TIPO_CONSUMO') == 'BORDA_OFICIAL'
).select(
    F.col('NR_TRAN_INST_PCT').alias('NR_TRAN_DENTRO'),
    F.col('NR_TRAN_CONTRAPARTE').alias('NR_TRAN_FORA'),
    F.col('DT_TRAN').alias('DT_TRAN_DENTRO'),
    F.col('DT_TRAN_CONTRAPARTE').alias('DT_TRAN_FORA'),
    'DIF_DIAS',
    F.col('CD_NTZ_CTB_TRAN').alias('NTZ_DENTRO'),
    F.col('CD_NTZ_CTB_TRAN_CONTRAPARTE').alias('NTZ_FORA'),
    'VL_TRAN', 'CD_TIP_MOE_CRR',
    F.col('TX_DCR_TRAN_OGNL').alias('DESC_DENTRO'),
    F.col('NR_MCA_PCT_OPB').alias('BANCO_DENTRO'),
    F.col('TX_DCR_TRAN_OGNL_CONTRAPARTE').alias('DESC_FORA'),
    F.col('NR_MCA_PCT_OPB_CONTRAPARTE').alias('BANCO_FORA'),
)

df_dashboard_externas = df_reconciliado.filter(F.col('IN_JANELA') == 'N').select(
    'NR_TRAN_INST_PCT', 'CD_CLI', 'DT_TRAN', 'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL', 'CD_TIP_MOE_CRR', 'VL_TRAN', 'IN_JANELA',
    'TX_DCR_TRAN_OGNL', 'NR_MCA_PCT_OPB', 'CD_GRUPO', 'TX_GRUPO',
    'CD_IR', 'TX_IR', 'TX_CATEGORIA',
    F.col('CD_CLASS_RADAR_MAPA').alias('CD_CLASS_RADAR'),
    F.col('TX_CLASS_RADAR_MAPA').alias('TX_CLASS_RADAR'),
    'IN_CLASSIFICADA_APRESENTACAO', 'IN_AGRO', 'IN_PARTICIPA_CALCULO',
    'IN_PARTICIPA_ORCAMENTO', 'TIPO_CONSUMO',
    F.when(F.col('TIPO_CONSUMO') == 'BORDA_CONTEXTO', 'Contraparte de borda')
     .when(F.col('TIPO_CONSUMO') == 'EXATO_CONTEXTO', 'Consumida em par exato externo')
     .otherwise('Não utilizada').alias('USO_CONTEXTO'),
)

df_dashboard_metricas = spark.range(1).select(
    F.lit(int(agregados['QT_OFICIAIS'])).cast('long').alias('QT_OFICIAIS'),
    F.lit(int(agregados['QT_PARES_EXATOS'])).cast('long').alias('QT_PARES_EXATOS'),
    F.lit(int(agregados['QT_PARES_BORDA'])).cast('long').alias('QT_PARES_BORDA'),
    F.lit(int(agregados['QT_EFETIVAS'])).cast('long').alias('QT_EFETIVAS'),
)

def empacotar_dashboard(bloco, dataframe):
    return dataframe.select(
        F.lit(bloco).alias('BLOCO'),
        F.to_json(F.struct(*[F.col(campo) for campo in dataframe.columns])).alias('PAYLOAD'),
    )

pacotes_dashboard = [
    empacotar_dashboard('PIVOT', df_dashboard_pivot),
    empacotar_dashboard('TRANSACOES', df_dashboard_transacoes),
    empacotar_dashboard('EXATOS', df_dashboard_exatos),
    empacotar_dashboard('BORDA', df_dashboard_borda),
    empacotar_dashboard('EXTERNAS', df_dashboard_externas),
    empacotar_dashboard('METRICAS', df_dashboard_metricas),
]

print('[INICIO] DASHBOARD_V8_PACOTES_COLETAR')
_inicio_action = time.perf_counter()
linhas_dashboard = reduce(lambda a, b: a.unionByName(b), pacotes_dashboard).collect()
print(f'[FIM] DASHBOARD_V8_PACOTES_COLETAR | {time.perf_counter() - _inicio_action:.3f}s')


In [ ]:
%%spark
dados_dashboard = {}
for linha in linhas_dashboard:
    dados_dashboard.setdefault(linha['BLOCO'], []).append(json.loads(linha['PAYLOAD']))

lista_dashboard_pivot = dados_dashboard.get('PIVOT', [])
lista_dashboard_transacoes = dados_dashboard.get('TRANSACOES', [])
lista_dashboard_exatos = dados_dashboard.get('EXATOS', [])
lista_dashboard_borda = dados_dashboard.get('BORDA', [])
lista_dashboard_externas = dados_dashboard.get('EXTERNAS', [])

def chave_nula(valor):
    return (valor is None, valor)

lista_dashboard_pivot.sort(key=lambda r: (
    chave_nula(r.get('ntz')), chave_nula(r.get('cd_classe')), chave_nula(r.get('cd_cat'))
))
lista_dashboard_transacoes.sort(key=lambda r: (
    chave_nula(r.get('CD_NTZ_CTB_TRAN')), chave_nula(r.get('CD_CLASS_RADAR')),
    chave_nula(r.get('CD_CTGR_TRAN_OGNL')), chave_nula(r.get('DT_TRAN')),
    chave_nula(r.get('NR_TRAN_INST_PCT')),
))
lista_dashboard_exatos.sort(key=lambda r: (
    r.get('IN_JANELA') != 'S', chave_nula(r.get('DT_TRAN')),
    chave_nula(r.get('ID_CREDITO')), chave_nula(r.get('ID_DEBITO')),
))
lista_dashboard_borda.sort(key=lambda r: (
    chave_nula(r.get('DT_TRAN_DENTRO')), chave_nula(r.get('NR_TRAN_DENTRO')),
    chave_nula(r.get('NR_TRAN_FORA')),
))
lista_dashboard_externas.sort(key=lambda r: (
    chave_nula(r.get('DT_TRAN')), chave_nula(r.get('NR_TRAN_INST_PCT')),
))

metricas_dashboard = dados_dashboard['METRICAS'][0]
qt_raw = int(metricas_dashboard['QT_OFICIAIS'])
qt_pares_exatos_oficiais = int(metricas_dashboard['QT_PARES_EXATOS'])
qt_pares_borda = int(metricas_dashboard['QT_PARES_BORDA'])
qt_efetivo = int(metricas_dashboard['QT_EFETIVAS'])

if qt_raw != int(agregados['QT_OFICIAIS']) or qt_efetivo != int(agregados['QT_EFETIVAS']):
    raise RuntimeError('Métricas do bundle divergiram da Row agregada.')

if df_reconciliado.is_cached:
    df_reconciliado.unpersist(blocking=False)


## Bloco V — Dashboard HTML

- **RESPONSABILIDADE:** montar, transferir, salvar e exibir o produto.
- **ENTRADAS:** resultado e pacotes locais.
- **SAÍDAS:** HTML interativo exportável.
- **LAZY OU ACTION:** local.
- **LINEAGE ESPERADA:** não reabre a lineage do motor.


In [ ]:
%%spark
def esc(valor):
    return html.escape('—' if valor is None else str(valor), quote=True)


In [ ]:
%%spark
def fmt_data(valor, curta=False):
    if valor is None:
        return '—'
    if isinstance(valor, str):
        try:
            valor = datetime.date.fromisoformat(valor[:10])
        except ValueError:
            return esc(valor)
    return valor.strftime('%d/%m' if curta else '%d/%m/%Y')

def fmt_data_hora(valor):
    if valor is None:
        return '—'
    if isinstance(valor, str):
        try:
            valor = datetime.datetime.fromisoformat(valor.replace('Z', '+00:00'))
        except ValueError:
            return esc(valor)
    if isinstance(valor, datetime.datetime):
        return valor.strftime('%d/%m/%Y %H:%M:%S')
    if isinstance(valor, datetime.date):
        return valor.strftime('%d/%m/%Y')
    return esc(valor)


In [ ]:
%%spark
def fmt_moeda(valor, sinal=False):
    if valor is None:
        return '—'
    numero = Decimal(str(valor))
    prefixo = ''
    if sinal:
        prefixo = '+ ' if numero > 0 else ('− ' if numero < 0 else '')
    numero = abs(numero) if sinal else numero
    texto = f'{numero:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'{prefixo}R$ {texto}'


In [ ]:
%%spark
def fmt_percentual(valor):
    if valor is None:
        return '—'
    texto = f'{Decimal(str(valor)) * Decimal("100"):.2f}'.replace('.', ',')
    return f'{texto}%'


In [ ]:
%%spark
def fmt_inteiro(valor):
    return '—' if valor is None else str(int(valor))


In [ ]:
%%spark
def nome_natureza(codigo):
    return {'C': 'Crédito', 'D': 'Débito'}.get(codigo, 'Natureza não reconhecida')


In [ ]:
%%spark
def somar_metricas(linhas, campo):
    valores = [r[campo] for r in linhas if r[campo] is not None]
    return sum((Decimal(str(v)) for v in valores), Decimal('0')) if valores else None


In [ ]:
%%spark
def metricas_html(linhas):
    return (
        f'<b>{sum(int(r["qt"]) for r in linhas)}</b> tx · '
        f'Mov. <span data-private="money">{fmt_moeda(somar_metricas(linhas, "vl_mov"))}</span> · '
        f'Tema <span data-private="money">{fmt_moeda(somar_metricas(linhas, "vl_tematico"))}</span> · '
        f'Orç. <span data-private="money">{fmt_moeda(somar_metricas(linhas, "vl_orcamentario"))}</span>'
    )


In [ ]:
%%spark
def rotulo_tratamento(transacao):
    tema = transacao.get('IN_PARTICIPA_CALCULO') == 'S'
    orcamento = transacao.get('IN_PARTICIPA_ORCAMENTO') == 'S'
    classificada = transacao.get('IN_CLASSIFICADA_APRESENTACAO') == 'S'
    if tema and orcamento:
        return 'Tema + Orçamento'
    if tema:
        return 'Tema / Fora do orçamento'
    if orcamento:
        return 'Fora do tema / Orçamento'
    if classificada:
        return 'Classificada / Fora tema e orçamento'
    return 'Sem classificação / Fora tema e orçamento'


In [ ]:
%%spark
def rotulo_classe(cd_classe, tx_classe, classificada):
    if classificada != 'S' or cd_classe is None or tx_classe is None:
        return 'Sem classificação'
    return str(tx_classe)


In [ ]:
%%spark
def tabela_transacoes_html(transacoes):
    if not transacoes:
        corpo = '<tr><td colspan="7">Nenhuma transação nesta categoria.</td></tr>'
    else:
        corpo = ''.join(
            '<tr>'
            f'<td>{fmt_data(t["DT_TRAN"])}</td>'
            f'<td>{esc(nome_natureza(t["CD_NTZ_CTB_TRAN"]))}</td>'
            f'<td class="desc">{esc(t["TX_DCR_TRAN_OGNL"])}</td>'
            f'<td><code>{esc(t["NR_MCA_PCT_OPB"])}</code></td>'
            f'<td data-private="money">{fmt_moeda(t["VL_TRAN"])}</td>'
            f'<td>{esc(t["CD_TIP_MOE_CRR"])}</td>'
            f'<td>{esc(rotulo_tratamento(t))}</td>'
            '</tr>'
            for t in transacoes
        )
    return (
        '<div class="tx-wrap"><table class="tx-table"><thead><tr>'
        '<th>Data</th><th>Natureza</th>'
        '<th>Descrição original<br/><small>TX_DCR_TRAN_OGNL</small></th>'
        '<th>Banco / Marca<br/><small>NR_MCA_PCT_OPB</small></th>'
        '<th>Valor</th><th>Moeda</th><th>Tratamento</th>'
        f'</tr></thead><tbody>{corpo}</tbody></table></div>'
    )


In [ ]:
%%spark
def render_composicao():
    blocos_natureza = []
    for ntz, titulo in [('C', 'Entradas'), ('D', 'Saídas')]:
        linhas_ntz = [r for r in lista_dashboard_pivot if r['ntz'] == ntz]
        if not linhas_ntz:
            continue
        blocos_classe = []
        classes = []
        for r in linhas_ntz:
            chave = (r['cd_classe'], r['tx_classe'], r['classificada'])
            if chave not in classes:
                classes.append(chave)
        for cd_classe, tx_classe, classificada in classes:
            linhas_classe = [
                r for r in linhas_ntz
                if (r['cd_classe'], r['tx_classe'], r['classificada'])
                == (cd_classe, tx_classe, classificada)
            ]
            blocos_categoria = []
            for cat in linhas_classe:
                transacoes = [
                    t for t in lista_dashboard_transacoes
                    if t['CD_NTZ_CTB_TRAN'] == ntz
                    and t['CD_CLASS_RADAR'] == cd_classe
                    and t['IN_CLASSIFICADA_APRESENTACAO'] == classificada
                    and t['CD_CTGR_TRAN_OGNL'] == cat['cd_cat']
                ]
                codigo = '—' if cat['cd_cat'] is None else str(cat['cd_cat'])
                blocos_categoria.append(
                    '<details class="level category" data-depth="category">'
                    f'<summary><span>{esc(codigo)} — {esc(cat["tx_cat"])}</span>'
                    f'<span class="metrics">{metricas_html([cat])}</span></summary>'
                    '<div class="detail-body"><div class="mini-grid">'
                    f'<div><span>Valor movimentado</span><strong data-private="money">{fmt_moeda(cat["vl_mov"])}</strong></div>'
                    f'<div><span>Valor temático</span><strong data-private="money">{fmt_moeda(cat["vl_tematico"])}</strong></div>'
                    f'<div><span>Valor orçamentário</span><strong data-private="money">{fmt_moeda(cat["vl_orcamentario"])}</strong></div>'
                    f'</div>{tabela_transacoes_html(transacoes)}</div></details>'
                )
            blocos_classe.append(
                '<details class="level class" data-depth="class">'
                f'<summary><span>{esc(rotulo_classe(cd_classe, tx_classe, classificada))}</span>'
                f'<span class="metrics">{metricas_html(linhas_classe)}</span></summary>'
                f'<div class="detail-body">{"".join(blocos_categoria)}</div></details>'
            )
        total_orc = res_dict.get('VL_ENT_TOTAL' if ntz == 'C' else 'VL_SAI_TOTAL')
        blocos_natureza.append(
            '<details class="level nature" data-depth="nature">'
            f'<summary><span>{titulo}</span><span class="metrics">'
            f'{sum(int(r["qt"]) for r in linhas_ntz)} tx · Orçamento <span data-private="money">{fmt_moeda(total_orc)}</span></span></summary>'
            f'<div class="detail-body">{"".join(blocos_classe)}</div></details>'
        )
    if not blocos_natureza:
        return '<div class="result-note">Nenhuma transação efetiva em BRL para compor.</div>'
    return ''.join(blocos_natureza)


In [ ]:
%%spark
def lado_reconciliacao(titulo, data, ntz, valor, moeda, descricao, banco, fora=False):
    classe = 'side out' if fora else 'side'
    return (
        f'<div class="{classe}"><div class="tx-title">{esc(nome_natureza(ntz))} · {esc(titulo)}</div>'
        f'<div>{fmt_data(data)}</div><div class="tx-value">{fmt_moeda(valor)}</div><div>{esc(moeda)}</div>'
        '<dl><dt>Descrição original<span class="field-tag">TX_DCR_TRAN_OGNL</span></dt>'
        f'<dd>{esc(descricao)}</dd><dt>Banco / Marca<span class="field-tag">NR_MCA_PCT_OPB</span></dt>'
        f'<dd><code>{esc(banco)}</code></dd></dl></div>'
    )


In [ ]:
%%spark
def render_eventos_reconciliacao():
    eventos = []
    for par in lista_dashboard_exatos:
        if par['IN_JANELA'] != 'S':
            continue
        eventos.append(
            '<details class="event"><summary>'
            f'Par exato · {fmt_moeda(par["VL_TRAN"])} · {fmt_data(par["DT_TRAN"])}</summary>'
            '<div class="event-body"><div class="pair">'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN'], 'D', par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_DEBITO'], par['BANCO_DEBITO'])
            + '<div class="link">↔</div>'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN'], 'C', par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_CREDITO'], par['BANCO_CREDITO'])
            + '</div><div class="checks">✓ mesma data · ✓ mesmo valor · ✓ mesma moeda · ✓ naturezas opostas · ✓ bancos conhecidos e diferentes</div>'
            '<div class="result-note">Resultado: as duas movimentações foram anuladas e não participaram dos cálculos.</div>'
            '</div></details>'
        )
    for par in lista_dashboard_borda:
        eventos.append(
            '<details class="event"><summary>'
            f'Reconciliação de borda · {fmt_moeda(par["VL_TRAN"])} · diferença de {par["DIF_DIAS"]} dias</summary>'
            '<div class="event-body"><div class="pair">'
            + lado_reconciliacao('Fora do ciclo', par['DT_TRAN_FORA'], par['NTZ_FORA'], par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_FORA'], par['BANCO_FORA'], True)
            + f'<div class="link">{par["DIF_DIAS"]} dias<br/>↔</div>'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN_DENTRO'], par['NTZ_DENTRO'], par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_DENTRO'], par['BANCO_DENTRO'])
            + '</div><div class="checks">✓ mesmo valor · ✓ mesma moeda · ✓ naturezas opostas · ✓ diferença de 1 a 5 dias · ✓ bancos conhecidos e diferentes</div>'
            '<div class="result-note">A movimentação oficial foi anulada. A contraparte externa foi usada somente como evidência e não entrou nos cálculos financeiros.</div>'
            '</div></details>'
        )
    return ''.join(eventos) or '<div class="result-note">Nenhum evento de reconciliação removeu movimentações oficiais.</div>'


In [ ]:
%%spark
def render_contexto_externo():
    linhas = ''.join(
        '<tr>'
        f'<td>{fmt_data(t["DT_TRAN"])}</td><td>{esc(nome_natureza(t["CD_NTZ_CTB_TRAN"]))}</td>'
        f'<td class="desc">{esc(t["TX_DCR_TRAN_OGNL"])}</td><td><code>{esc(t["NR_MCA_PCT_OPB"])}</code></td>'
        f'<td>{fmt_moeda(t["VL_TRAN"])}</td><td>{esc(t["CD_TIP_MOE_CRR"])}</td><td>{esc(t["USO_CONTEXTO"])}</td>'
        '</tr>'
        for t in lista_dashboard_externas
    )
    if not linhas:
        linhas = '<tr><td colspan="7">Nenhuma movimentação fora do ciclo.</td></tr>'
    return (
        f'<details class="event"><summary>Contexto externo analisado · {len(lista_dashboard_externas)} movimentações</summary>'
        '<div class="event-body"><div class="ext-table"><table><thead><tr>'
        '<th>Data</th><th>Natureza</th><th>Descrição original<br/><small>TX_DCR_TRAN_OGNL</small></th>'
        '<th>Banco / Marca<br/><small>NR_MCA_PCT_OPB</small></th><th>Valor</th><th>Moeda</th><th>Uso no contexto</th>'
        f'</tr></thead><tbody>{linhas}</tbody></table></div>'
        '<div class="result-note" style="margin-top:12px">Nenhuma movimentação externa participou dos cálculos financeiros do ciclo.</div>'
        '</div></details>'
    )


In [ ]:
%%spark
temas = [
    (1, 'Categorização dos Gastos', 'VL_SAI_IND', 'PC_SAI_IND', 'PC_REF_IND', 'NR_PONT_CONC_IND', None, None, 'NR_PONT_IND_FIM'),
    (2, 'Gestão de Orçamento', 'VL_SAI_ESS', 'PC_SAI_ESS', 'PC_REF_ESS', 'NR_PONT_CONC_ESS', 'NR_PONT_ORC_ESS', 'NR_PONT_PRFL_ESS', 'NR_PONT_ESS_FIM'),
    (3, 'Consumo Planejado', 'VL_SAI_NAO_ESS', 'PC_SAI_NAO_ESS', 'PC_REF_NAO_ESS', 'NR_PONT_CONC_NAO_ESS', 'NR_PONT_ORC_NAO_ESS', 'NR_PONT_PRFL_NAO_ESS', 'NR_PONT_NAO_ESS_FIM'),
    (4, 'Formação de Reserva', 'VL_SAI_FUT', 'PC_SAI_FUT', 'PC_REF_FUT', 'NR_PONT_CONC_FUT', 'NR_PONT_ORC_FUT', 'NR_PONT_PRFL_FUT', 'NR_PONT_FUT_FIM'),
    (5, 'Uso Consciente do Crédito', 'VL_SAI_OBR', 'PC_SAI_OBR', 'PC_REF_OBR', 'NR_PONT_CONC_OBR', 'NR_PONT_ORC_OBR', 'NR_PONT_PRFL_OBR', 'NR_PONT_OBR_FIM'),
]


In [ ]:
%%spark
def render_linhas_pontuacao(resultado):
    linhas = []
    for codigo, nome, vl, pc, ref, conc, orc, prfl, final in temas:
        vencedor = (
            resultado.get('CD_TEMA_VENCEDOR') == codigo
            or (
                resultado.get('CD_TEMA_VENCEDOR') == 9
                and resultado.get(final) is not None
                and resultado.get(final) == resultado.get('NR_PONT_MAX')
            )
        )
        classe = ' class="winner-row"' if vencedor else ''
        valor_final = fmt_inteiro(resultado.get(final))
        celula_final = f'<span class="score-final-badge">{valor_final}</span>' if vencedor else valor_final
        regra_final = 'Final = Concentração' if codigo == 1 else 'Final = Concentração + Orçamento + Perfil'
        linhas.append(
            f'<tr{classe}><td class="theme-cell">{esc(nome)}<small>{esc(regra_final)}</small></td>'
            f'<td class="num" data-private="money">{fmt_moeda(resultado.get(vl))}</td>'
            f'<td class="num">{fmt_percentual(resultado.get(pc))}</td>'
            f'<td class="num">{fmt_percentual(resultado.get(ref))}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(conc))}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(orc)) if orc else "—"}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(prfl)) if prfl else "—"}</td>'
            f'<td class="final-cell">{celula_final}</td></tr>'
        )
    return ''.join(linhas)


In [ ]:
%%spark
def tag_execucao(rotulo, valor, positivo='S'):
    classe = ' ok' if valor == positivo else ''
    prefixo = '✓ ' if classe else ''
    exibido = {'S': 'Sim', 'N': 'Não', None: '—'}.get(valor, str(valor))
    return f'<span class="run-tag{classe}">{prefixo}{esc(rotulo)}: {esc(exibido)}</span>'


In [ ]:
%%spark
dt_contexto_ini = dt_ini_j - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO) if dt_ini_j else None
dt_contexto_fim = dt_fim_j + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO) if dt_fim_j else None
fallback_usado = (
    '—' if res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK') is None
    else ('Sim' if res_dict.get('DD_INC_MM_CLC_BLC') is None else 'Não')
)


In [ ]:
%%spark
def texto_payload(valor):
    return '—' if valor is None else str(valor)


In [ ]:
%%spark
def montar_payload_cenario(chave, resultado):
    presumida = chave == 'RENDA_PRESUMIDA'
    base_financeira = bases_cenarios[chave]
    pc_saida_entrada_cenario = resultado.get('PC_SAI_ENT')
    largura_cenario = (
        0 if pc_saida_entrada_cenario is None
        else max(0, min(100, float(pc_saida_entrada_cenario) * 100))
    )
    rotulo = 'Renda Presumida' if presumida else 'Entradas Realizadas'
    referencia_rotulo = 'Referência da renda' if presumida else 'Origem da base'
    referencia_valor = (
        fmt_data(resultado.get('DT_REN_PRES_REF'))
        if presumida
        else f'Entradas efetivas do ciclo · {fmt_data(resultado.get("DT_REF_INI"), True)} → {fmt_data(resultado.get("DT_REF_FIM"), True)}'
    )
    return {
        'base_kicker': 'Base financeira',
        'base_badge': f'{rotulo} ativa',
        'valor_base': fmt_moeda(base_financeira),
        'referencia_rotulo': referencia_rotulo,
        'referencia_valor': referencia_valor,
        'entrada_rotulo': 'Base financeira',
        'entrada_valor': fmt_moeda(base_financeira),
        'saldo_status': texto_payload(resultado.get('TX_STS_FINAL')),
        'saldo_valor': fmt_moeda(resultado.get('VL_RES_ORC'), True),
        'razao_valor': fmt_percentual(pc_saida_entrada_cenario),
        'largura_medidor': f'{largura_cenario:.2f}%',
        'pontuacao_html': render_linhas_pontuacao(resultado),
        'tag_base_html': f'<span class="run-tag base-active">Base ativa: {esc(rotulo)}</span>',
        'tag_pontuacao_html': tag_execucao('Pontuação completa', resultado.get('FL_PONTUACAO_COMPLETA')),
        'cenario_codigo': chave,
        'cenario_rotulo': rotulo,
        'base_visao': f'Base usada nesta visão: {rotulo} — {fmt_moeda(base_financeira)}',
        'historico_base_financeira': fmt_moeda(base_financeira),
        'historico_resultado_orcamentario': fmt_moeda(resultado.get('VL_RES_ORC'), True),
        'historico_pc_saida_base': fmt_percentual(pc_saida_entrada_cenario),
        'historico_status_final': texto_payload(resultado.get('TX_STS_FINAL')),
        'historico_pontuacao_completa': texto_payload(resultado.get('FL_PONTUACAO_COMPLETA')),
        'historico_pont_max': fmt_inteiro(resultado.get('NR_PONT_MAX')),
        'historico_qt_temas_max': fmt_inteiro(resultado.get('QT_TEMAS_PONT_MAX')),
        'historico_cd_tema': fmt_inteiro(resultado.get('CD_TEMA_VENCEDOR')),
        'prioridade_orientacao': texto_payload(resultado.get('TX_TEMA_VENCEDOR')),
    }


In [ ]:
%%spark
payload_cenarios = {
    chave: montar_payload_cenario(chave, resultados_cenarios[chave])
    for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS')
}
payload_padrao = payload_cenarios['RENDA_PRESUMIDA']
payload_cenarios_json = (
    json.dumps(payload_cenarios, ensure_ascii=False, separators=(',', ':'))
    .replace('&', '\\u0026')
    .replace('<', '\\u003c')
    .replace('>', '\\u003e')
)
CSS_COMPLEMENTO = r"""
[data-radar-root="individual"] .base-switch{display:grid;grid-template-columns:1fr 1fr;gap:4px;margin-top:12px;padding:4px;background:#f0f3f8;border:1px solid var(--line);border-radius:12px}
[data-radar-root="individual"] .base-switch-btn{border:0;background:transparent;color:var(--muted);border-radius:9px;padding:7px 6px;font-size:9px;font-weight:850;line-height:1.15;cursor:pointer}
[data-radar-root="individual"] .base-switch-btn.active{background:#fff;color:var(--bb-blue-deep);box-shadow:0 2px 8px rgba(27,36,74,.12)}
[data-radar-root="individual"] .base-switch-btn:focus-visible{outline:3px solid rgba(70,94,255,.2);outline-offset:1px}
[data-radar-root="individual"] .scenario-active-tag{display:inline-flex;align-items:center;border:1px solid #cce7da;background:var(--good-bg);color:var(--good);border-radius:999px;padding:5px 8px;font-size:9px;font-weight:850}
[data-radar-root="individual"] .run-tag.base-active{background:#eef8f3;border-color:#cce7da;color:#176848}
"""


In [ ]:
%%spark
CSS_APROVADO = '[data-radar-root="individual"]{\n  --bb-blue:#465eff;--bb-blue-deep:#252d84;--bb-yellow:#fcfc30;--ink:#13162b;--muted:#667085;\n  --bg:#f4f6fb;--card:#fff;--line:#e3e7ef;--soft:#f8f9fc;--good:#087a55;--good-bg:#e8f7f0;\n  --warn:#8c5a00;--warn-bg:#fff5d9;--danger:#b42318;--danger-bg:#fff0ee;--shadow:0 14px 40px rgba(27,36,74,.08);\n  --radius:22px;--radius-sm:14px;--max:1280px\n}\n[data-radar-root="individual"],[data-radar-root="individual"] *{box-sizing:border-box}\n[data-radar-root="individual"]{scroll-behavior:smooth}\n[data-radar-root="individual"]{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",Arial,sans-serif;line-height:1.45}\n[data-radar-root="individual"] button,[data-radar-root="individual"] input{font:inherit}\n[data-radar-root="individual"] a{color:inherit}\n[data-radar-root="individual"] .shell{max-width:var(--max);margin:auto;padding:0 24px 84px}\n[data-radar-root="individual"] .skip{position:absolute;left:-9999px;top:auto}\n[data-radar-root="individual"] .skip:focus{left:16px;top:16px;z-index:9999;background:#fff;padding:10px 14px;border-radius:10px}\n[data-radar-root="individual"] .topbar{position:sticky;top:0;z-index:50;background:rgba(244,246,251,.88);backdrop-filter:blur(18px);border-bottom:1px solid rgba(227,231,239,.8)}\n[data-radar-root="individual"] .topbar-inner{max-width:var(--max);margin:auto;padding:12px 24px;display:flex;gap:14px;align-items:center;justify-content:space-between}\n[data-radar-root="individual"] .brand{display:flex;align-items:center;gap:10px;font-weight:850;white-space:nowrap}\n[data-radar-root="individual"] .brand-mark{width:28px;height:28px;border-radius:9px;background:var(--bb-yellow);border:7px solid var(--bb-blue);transform:rotate(45deg);box-shadow:inset 0 0 0 2px #fff}\n[data-radar-root="individual"] .nav{display:flex;gap:4px;overflow:auto;scrollbar-width:none}\n[data-radar-root="individual"] .nav::-webkit-scrollbar{display:none}\n[data-radar-root="individual"] .nav a{text-decoration:none;color:#525b75;padding:8px 10px;border-radius:10px;font-size:12px;font-weight:720;white-space:nowrap}\n[data-radar-root="individual"] .nav a:hover,[data-radar-root="individual"] .nav a.active{background:#fff;color:var(--bb-blue-deep);box-shadow:0 1px 4px rgba(0,0,0,.06)}\n[data-radar-root="individual"] .privacy{border:1px solid var(--line);background:#fff;border-radius:999px;padding:8px 11px;font-weight:750;font-size:12px;cursor:pointer;white-space:nowrap}\n[data-radar-root="individual"] .hero{margin-top:24px;background:linear-gradient(135deg,var(--bb-blue-deep),#3342b6 52%,var(--bb-blue));color:#fff;border-radius:28px;padding:26px 28px;box-shadow:0 20px 60px rgba(37,45,132,.18);position:relative;overflow:hidden}\n[data-radar-root="individual"] .hero:after{content:"";position:absolute;width:320px;height:320px;border-radius:50%;background:var(--bb-yellow);right:-165px;top:-175px;opacity:.96}\n[data-radar-root="individual"] .hero-grid{display:grid;grid-template-columns:1.4fr .8fr;gap:28px;position:relative;z-index:1}\n[data-radar-root="individual"] .eyebrow{font-size:10px;font-weight:850;letter-spacing:.12em;text-transform:uppercase;opacity:.72}\n[data-radar-root="individual"] .hero h1{margin:7px 0 14px;font-size:clamp(30px,4vw,48px);line-height:1.02;letter-spacing:-.04em}\n[data-radar-root="individual"] .hero p{margin:0;color:rgba(255,255,255,.76);max-width:760px}\n[data-radar-root="individual"] .hero-status{align-self:end;background:rgba(255,255,255,.1);border:1px solid rgba(255,255,255,.18);padding:18px;border-radius:18px}\n[data-radar-root="individual"] .hero-status small{display:block;color:rgba(255,255,255,.7)}\n[data-radar-root="individual"] .hero-status strong{font-size:21px;display:block;margin:3px 0}\n[data-radar-root="individual"] .hero-status span{font-size:12px}\n[data-radar-root="individual"] .hero-chips{display:flex;gap:7px;flex-wrap:wrap;margin-top:0}\n[data-radar-root="individual"] .chip{border:1px solid rgba(255,255,255,.2);background:rgba(255,255,255,.1);padding:6px 9px;border-radius:999px;font-size:10px;font-weight:720}\n[data-radar-root="individual"] .context-strip{display:flex;align-items:stretch;gap:0;background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}\n[data-radar-root="individual"] .context-item{flex:1;min-width:0;padding:14px 16px;border-right:1px solid var(--line)}\n[data-radar-root="individual"] .context-item:last-child{border-right:0}\n[data-radar-root="individual"] .context-label{font-size:9px;color:var(--muted);font-weight:850;text-transform:uppercase;letter-spacing:.07em}\n[data-radar-root="individual"] .context-value{font-size:14px;font-weight:820;margin-top:4px;overflow-wrap:anywhere}\n[data-radar-root="individual"] .context-meta{font-size:9px;color:var(--muted);margin-top:2px}\n[data-radar-root="individual"] .money-grid{display:grid;grid-template-columns:1fr auto 1fr auto 1.15fr;gap:0;align-items:stretch;background:#fff;border:1px solid var(--line);border-radius:20px;overflow:hidden}\n[data-radar-root="individual"] .money-card{background:#fff;padding:20px 22px;border-right:1px solid var(--line)}\n[data-radar-root="individual"] .money-card:last-child{border-right:0}\n[data-radar-root="individual"] .money-card .label{font-size:10px;color:var(--muted);text-transform:uppercase;font-weight:850;letter-spacing:.08em}\n[data-radar-root="individual"] .money-card .value{font-size:clamp(24px,3vw,36px);font-weight:900;margin:5px 0;letter-spacing:-.04em}\n[data-radar-root="individual"] .money-card .meta{font-size:11px;color:var(--muted)}\n[data-radar-root="individual"] .operator{align-self:center;font-size:22px;color:#a7afc3;font-weight:300;padding:0 10px}\n[data-radar-root="individual"] .money-card.balance-card{background:#f6fbf8}\n[data-radar-root="individual"] .balance-card .value{color:var(--good)}\n[data-radar-root="individual"] .status-pill{display:inline-flex;margin-top:10px;background:var(--good-bg);color:var(--good);padding:7px 10px;border-radius:999px;font-size:11px;font-weight:850}\n[data-radar-root="individual"] .ratio-bar{margin-top:12px;height:8px;background:#e9edf4;border-radius:999px;overflow:hidden}\n[data-radar-root="individual"] .ratio-bar span{display:block;height:100%;width:72.39%;background:var(--bb-blue)}\n[data-radar-root="individual"] .trace-box{display:grid;grid-template-columns:1.25fr .75fr;gap:12px;margin-top:12px}\n[data-radar-root="individual"] .trace-main,[data-radar-root="individual"] .trace-side{background:#fff;border:1px solid var(--line);border-radius:20px;padding:18px}\n[data-radar-root="individual"] .trace-main h3,[data-radar-root="individual"] .trace-side h3{margin:0 0 7px;font-size:15px}\n[data-radar-root="individual"] .trace-main p,[data-radar-root="individual"] .trace-side p{margin:0;color:var(--muted);font-size:12px}\n[data-radar-root="individual"] .formula{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin-top:15px}\n[data-radar-root="individual"] .formula span{background:var(--soft);border:1px solid var(--line);border-radius:11px;padding:8px 10px;font-size:11px;font-weight:800}\n[data-radar-root="individual"] .formula b{color:var(--muted)}\n[data-radar-root="individual"] .controls{display:flex;gap:8px;flex-wrap:wrap;align-items:center}\n[data-radar-root="individual"] .control-btn{border:1px solid var(--line);background:#fff;border-radius:11px;padding:8px 11px;font-size:11px;font-weight:750;cursor:pointer}\n[data-radar-root="individual"] .control-btn:hover{border-color:#bbc3d4}\n[data-radar-root="individual"] .search{border:1px solid var(--line);background:#fff;border-radius:11px;padding:8px 11px;font-size:11px;min-width:220px;outline:none}\n[data-radar-root="individual"] .search:focus{border-color:var(--bb-blue);box-shadow:0 0 0 3px rgba(70,94,255,.12)}\n[data-radar-root="individual"] .explorer{background:#fff;border:1px solid var(--line);border-radius:22px;padding:10px;box-shadow:var(--shadow)}\n[data-radar-root="individual"] .level{background:#fff;border:1px solid var(--line);border-radius:14px;margin:8px 0;overflow:hidden}\n[data-radar-root="individual"] .level summary{cursor:pointer;list-style:none;padding:14px 15px;display:flex;justify-content:space-between;gap:16px;align-items:center}\n[data-radar-root="individual"] .level summary::-webkit-details-marker{display:none}\n[data-radar-root="individual"] .level summary:before{content:"+";display:grid;place-items:center;flex:0 0 24px;height:24px;border-radius:8px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:900}\n[data-radar-root="individual"] .level[open]>summary:before{content:"−"}\n[data-radar-root="individual"] .level>summary{font-weight:750}\n[data-radar-root="individual"] .nature>summary{background:#f4f5ff;font-size:15px}\n[data-radar-root="individual"] .class>summary{background:#fafbff}\n[data-radar-root="individual"] .category>summary{font-weight:650}\n[data-radar-root="individual"] .metrics{font-size:11px;color:var(--muted);font-weight:500;text-align:right;margin-left:auto}\n[data-radar-root="individual"] .detail-body{padding:0 13px 13px 26px}\n[data-radar-root="individual"] .mini-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:8px 0 12px}\n[data-radar-root="individual"] .mini-grid div{background:#f8f9fc;border-radius:11px;padding:10px}\n[data-radar-root="individual"] .mini-grid span{font-size:9px;text-transform:uppercase;color:var(--muted);display:block}\n[data-radar-root="individual"] .mini-grid strong{font-size:13px}\n[data-radar-root="individual"] .tx-wrap,[data-radar-root="individual"] .ext-table{overflow-x:auto;border-radius:12px;border:1px solid var(--line);background:#fff}\n[data-radar-root="individual"] table{width:100%;border-collapse:collapse}\n[data-radar-root="individual"] th,[data-radar-root="individual"] td{padding:10px;border-bottom:1px solid var(--line);font-size:11px;text-align:left;vertical-align:top}\n[data-radar-root="individual"] th{font-size:9px;text-transform:uppercase;letter-spacing:.04em;color:var(--muted);background:#f8f9fc;position:sticky;top:0}\n[data-radar-root="individual"] th small{display:block;font-size:8px;letter-spacing:0;text-transform:none;color:#98a2b3}\n[data-radar-root="individual"] td.desc{min-width:220px;max-width:360px;white-space:normal}\n[data-radar-root="individual"] td code{font-size:10px;white-space:nowrap}\n[data-radar-root="individual"] .sim-note{font-size:10px;color:var(--muted);font-style:italic}\n[data-radar-root="individual"] .score-layout{display:grid;grid-template-columns:1.35fr .65fr;gap:12px}\n[data-radar-root="individual"] .score-list{background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}\n[data-radar-root="individual"] .score-item{background:#fff;padding:15px 16px;border-bottom:1px solid var(--line)}\n[data-radar-root="individual"] .score-item:last-of-type{border-bottom:0}\n[data-radar-root="individual"] .score-head{display:flex;justify-content:space-between;gap:14px;align-items:start}\n[data-radar-root="individual"] .score-head strong{display:block;font-size:13px}\n[data-radar-root="individual"] .score-head span{display:block;font-size:10px;color:var(--muted);margin-top:2px}\n[data-radar-root="individual"] .score-head>b{font-size:21px}\n[data-radar-root="individual"] .score-track{height:7px;background:#edf0f5;border-radius:999px;overflow:hidden;margin:11px 0 8px}\n[data-radar-root="individual"] .score-track span{display:block;height:100%;background:var(--bb-blue);min-width:0}\n[data-radar-root="individual"] .score-memory{display:flex;gap:12px;flex-wrap:wrap;color:var(--muted);font-size:10px}\n[data-radar-root="individual"] .score-memory b{color:var(--ink)}\n[data-radar-root="individual"] .winner-card{background:var(--bb-yellow);border:1px solid #e2df4f;border-radius:18px;padding:20px;position:sticky;top:84px;align-self:start}\n[data-radar-root="individual"] .winner-card .kicker{color:#333}\n[data-radar-root="individual"] .winner-card h3{font-size:25px;margin:6px 0;letter-spacing:-.03em}\n[data-radar-root="individual"] .winner-card .big{font-size:34px;font-weight:900}\n[data-radar-root="individual"] .winner-card p{font-size:11px;margin:8px 0 0;max-width:300px}\n[data-radar-root="individual"] .rule-note{background:#fff8df;border:1px solid #f1df9b;color:#725000;padding:10px 12px;border-radius:11px;font-size:10px;margin-top:10px}\n[data-radar-root="individual"] .recon{background:#fff;border:1px solid var(--line);border-radius:22px;padding:16px;box-shadow:var(--shadow)}\n[data-radar-root="individual"] .funnel{display:flex;align-items:center;justify-content:center;gap:9px;flex-wrap:wrap;margin:4px 0 16px}\n[data-radar-root="individual"] .step{padding:10px 13px;border-radius:11px;background:#f5f7fb;text-align:center;font-size:10px}\n[data-radar-root="individual"] .step b{font-size:18px;display:block}\n[data-radar-root="individual"] .minus{color:var(--danger)}\n[data-radar-root="individual"] .event{border:1px solid var(--line);border-radius:14px;margin:9px 0;overflow:hidden}\n[data-radar-root="individual"] .event summary{cursor:pointer;padding:13px 15px;font-weight:750;background:#fafbfe}\n[data-radar-root="individual"] .event-body{padding:13px 15px}\n[data-radar-root="individual"] .pair{display:grid;grid-template-columns:1fr 58px 1fr;gap:9px;align-items:stretch}\n[data-radar-root="individual"] .side{background:#f7f9fc;border-radius:12px;padding:13px}\n[data-radar-root="individual"] .side.out{background:var(--warn-bg)}\n[data-radar-root="individual"] .link{text-align:center;color:var(--muted);font-weight:850;align-self:center}\n[data-radar-root="individual"] .side .tx-title{font-size:13px;font-weight:850;margin-bottom:6px}\n[data-radar-root="individual"] .side .tx-value{font-size:18px;font-weight:850;margin:4px 0}\n[data-radar-root="individual"] .side dl{display:grid;grid-template-columns:120px 1fr;gap:5px 8px;margin:11px 0 0;font-size:10px}\n[data-radar-root="individual"] .side dt{color:var(--muted)}\n[data-radar-root="individual"] .side dd{margin:0;font-weight:650;word-break:break-word}\n[data-radar-root="individual"] .field-tag{font-size:7px;color:#8b98aa;display:block}\n[data-radar-root="individual"] .checks{margin:11px 0;font-size:10px;color:var(--muted)}\n[data-radar-root="individual"] .result-note{background:var(--good-bg);color:#155e3c;padding:10px;border-radius:10px;font-size:10px}\n[data-radar-root="individual"] .tech{background:#fff;border:1px solid var(--line);border-radius:18px;padding:0;overflow:hidden}\n[data-radar-root="individual"] .tech>summary{cursor:pointer;font-weight:820;padding:15px 17px;background:#fbfcfe}\n[data-radar-root="individual"] .tech .scroll{max-height:620px;overflow:auto;margin:0;border-top:1px solid var(--line)}\n[data-radar-root="individual"] .privacy-on [data-sensitive="true"]{filter:blur(5px);user-select:none}\n[data-radar-root="individual"] .privacy-on .tech tbody tr[data-private="true"] td:last-child{filter:blur(5px);user-select:none}\n[data-radar-root="individual"] mark{background:var(--bb-yellow);color:inherit;padding:0 .08em}\n[data-radar-root="individual"] .section{margin-top:24px}\n[data-radar-root="individual"] .section-head{display:flex;align-items:center;justify-content:space-between;gap:14px;margin-bottom:10px}\n[data-radar-root="individual"] .section-head h2{font-size:18px;letter-spacing:-.02em;margin:0}\n[data-radar-root="individual"] .run-header{margin-top:24px;background:#fff;border:1px solid var(--line);border-radius:20px;padding:17px 19px;display:flex;align-items:center;justify-content:space-between;gap:18px;box-shadow:0 8px 28px rgba(27,36,74,.05)}\n[data-radar-root="individual"] .run-title{display:flex;align-items:baseline;gap:10px;white-space:nowrap}\n[data-radar-root="individual"] .run-title .eyebrow{color:var(--muted);opacity:1}\n[data-radar-root="individual"] .run-title h1{font-size:22px;line-height:1;margin:0;letter-spacing:-.035em}\n[data-radar-root="individual"] .run-tags{display:flex;justify-content:flex-end;gap:7px;flex-wrap:wrap}\n[data-radar-root="individual"] .run-tag{display:inline-flex;align-items:center;min-height:28px;padding:5px 9px;border-radius:999px;border:1px solid var(--line);background:var(--soft);font-size:10px;font-weight:780;color:#4c556f;white-space:nowrap}\n[data-radar-root="individual"] .run-tag.ok{background:#f2f8f5;border-color:#d6eadf;color:#176848}\n[data-radar-root="individual"] .context-grid{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="individual"] .context-card{position:relative;background:#fff;border:1px solid var(--line);border-radius:18px;padding:16px 17px;min-height:138px;overflow:hidden;box-shadow:0 5px 18px rgba(27,36,74,.035)}\n[data-radar-root="individual"] .context-card:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:#dfe3ff}\n[data-radar-root="individual"] .context-card:hover{border-color:#d4d9e6;box-shadow:0 10px 28px rgba(27,36,74,.065)}\n[data-radar-root="individual"] .identity-card{grid-column:span 4;min-height:174px}\n[data-radar-root="individual"] .identity-card:before{background:var(--bb-blue)}\n[data-radar-root="individual"] .cycle-card{grid-column:span 8;min-height:174px}\n[data-radar-root="individual"] .cycle-card:before{background:#8090ff}\n[data-radar-root="individual"] .income-card{grid-column:span 3}\n[data-radar-root="individual"] .income-card:before{background:#22a06b}\n[data-radar-root="individual"] .profile-card{grid-column:span 3}\n[data-radar-root="individual"] .profile-card:before{background:#8c7ae6}\n[data-radar-root="individual"] .coverage-card{grid-column:span 3}\n[data-radar-root="individual"] .coverage-card:before{background:#19a47b}\n[data-radar-root="individual"] .window-card{grid-column:span 3}\n[data-radar-root="individual"] .window-card:before{background:#5a84d6}\n[data-radar-root="individual"] .context-top{display:flex;align-items:center;justify-content:space-between;gap:10px}\n[data-radar-root="individual"] .context-kicker{font-size:11px;font-weight:880;letter-spacing:.08em;text-transform:uppercase;color:var(--muted)}\n[data-radar-root="individual"] .context-dot{width:8px;height:8px;border-radius:50%;background:var(--bb-yellow);box-shadow:0 0 0 4px #fffbd3}\n[data-radar-root="individual"] .context-badge{font-size:11px;font-weight:820;padding:4px 7px;border-radius:999px;background:#f0f2ff;color:var(--bb-blue-deep);white-space:nowrap}\n[data-radar-root="individual"] .context-badge.subtle{background:#f5f6f9;color:var(--muted)}\n[data-radar-root="individual"] .context-badge.good{background:var(--good-bg);color:var(--good)}\n[data-radar-root="individual"] .context-main{font-size:23px;font-weight:900;letter-spacing:-.035em;margin:12px 0 12px;line-height:1.05}\n[data-radar-root="individual"] .context-main small{font-size:11px;font-weight:720;letter-spacing:0;color:var(--muted);margin-left:3px}\n[data-radar-root="individual"] .date-range span{color:#a2a9ba;font-weight:500;margin:0 4px}\n[data-radar-root="individual"] .context-labels{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}\n[data-radar-root="individual"] .micro-label{display:inline-flex;align-items:center;gap:5px;padding:6px 8px;border:1px solid #edf0f5;border-radius:8px;background:#fafbfe;font-size:11px;color:#596276;font-weight:720}\n[data-radar-root="individual"] .micro-label b{color:var(--ink);font-weight:840}\n[data-radar-root="individual"] .context-data{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-top:12px}\n[data-radar-root="individual"] .context-data.single{grid-template-columns:1fr}\n[data-radar-root="individual"] .data-cell{padding-top:9px;border-top:1px solid #eef0f4}\n[data-radar-root="individual"] .data-cell span{display:block;font-size:10px;text-transform:uppercase;letter-spacing:.06em;color:#98a2b3;font-weight:820}\n[data-radar-root="individual"] .data-cell strong{display:block;margin-top:3px;font-size:11px;line-height:1.25;overflow-wrap:anywhere}\n[data-radar-root="individual"] .context-foot{margin-top:8px;color:#8a93a7;font-size:10px;white-space:normal}\n[data-radar-root="individual"] .section-head>div:first-child{min-width:0}\n[data-radar-root="individual"] .section-label{display:block;font-size:9px;font-weight:900;letter-spacing:.11em;text-transform:uppercase;color:#98a2b3;margin-bottom:2px}\n[data-radar-root="individual"] .context-card,[data-radar-root="individual"] .finance-card,[data-radar-root="individual"] .score-card-radar{padding:0}\n[data-radar-root="individual"] .context-card>summary,[data-radar-root="individual"] .finance-card>summary,[data-radar-root="individual"] .score-card-radar>summary{list-style:none;cursor:pointer;padding:15px 16px;display:flex;align-items:center;justify-content:space-between;gap:12px;min-height:76px}\n[data-radar-root="individual"] .context-card>summary::-webkit-details-marker,[data-radar-root="individual"] .finance-card>summary::-webkit-details-marker,[data-radar-root="individual"] .score-card-radar>summary::-webkit-details-marker{display:none}\n[data-radar-root="individual"] .context-card>summary:after,[data-radar-root="individual"] .finance-card>summary:after,[data-radar-root="individual"] .score-card-radar>summary:after{content:"+";display:grid;place-items:center;flex:0 0 25px;height:25px;border-radius:8px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:900}\n[data-radar-root="individual"] .context-card[open]>summary:after,[data-radar-root="individual"] .finance-card[open]>summary:after,[data-radar-root="individual"] .score-card-radar[open]>summary:after{content:"−"}\n[data-radar-root="individual"] .card-summary-main{min-width:0;flex:1}\n[data-radar-root="individual"] .card-summary-row{display:flex;align-items:center;gap:8px;flex-wrap:wrap}\n[data-radar-root="individual"] .card-summary-value{font-size:20px;font-weight:900;letter-spacing:-.035em;line-height:1.1;margin-top:5px;overflow-wrap:anywhere}\n[data-radar-root="individual"] .card-summary-value.good{color:var(--good)}\n[data-radar-root="individual"] .card-body{border-top:1px solid #edf0f4;padding:12px 16px 15px}\n[data-radar-root="individual"] .context-card .context-labels,[data-radar-root="individual"] .context-card .context-data,[data-radar-root="individual"] .context-card .context-foot{margin-top:0}\n[data-radar-root="individual"] .context-card .context-data{margin-top:10px}\n[data-radar-root="individual"] .context-card .context-foot{padding-top:9px}\n[data-radar-root="individual"] .finance-card .card-labels{margin-top:0}\n[data-radar-root="individual"] .finance-card .meter{margin-top:4px}\n[data-radar-root="individual"] .score-card-radar .score-data{margin-top:0}\n[data-radar-root="individual"] .score-card-radar .score-parts{margin-top:10px}\n[data-radar-root="individual"] .score-card-radar.featured>summary{background:linear-gradient(135deg,#fffef0,#fff)}\n[data-radar-root="individual"] .finance-grid{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="individual"] .finance-card,[data-radar-root="individual"] .score-card-radar{position:relative;background:#fff;border:1px solid var(--line);border-radius:18px;padding:16px 17px;overflow:hidden;box-shadow:0 5px 18px rgba(27,36,74,.035);transition:.18s ease}\n[data-radar-root="individual"] .finance-card:before,[data-radar-root="individual"] .score-card-radar:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:#dfe3ff}\n[data-radar-root="individual"] .finance-card:hover,[data-radar-root="individual"] .score-card-radar:hover{border-color:#d4d9e6;box-shadow:0 10px 28px rgba(27,36,74,.065);transform:translateY(-1px)}\n[data-radar-root="individual"] .finance-in{grid-column:span 3}\n[data-radar-root="individual"] .finance-out{grid-column:span 3}\n[data-radar-root="individual"] .finance-balance{grid-column:span 4;background:#f8fcfa}\n[data-radar-root="individual"] .finance-ratio{grid-column:span 2}\n[data-radar-root="individual"] .finance-balance:before{background:#8fd8bd}\n[data-radar-root="individual"] .finance-ratio:before{background:#b9c1ff}\n[data-radar-root="individual"] .card-top{display:flex;align-items:center;justify-content:space-between;gap:10px}\n[data-radar-root="individual"] .card-kicker{font-size:9px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted);font-weight:900}\n[data-radar-root="individual"] .card-badge{padding:4px 7px;border-radius:999px;background:#f1f3ff;color:var(--bb-blue-deep);font-size:8px;font-weight:850;white-space:nowrap}\n[data-radar-root="individual"] .card-badge.good{background:var(--good-bg);color:var(--good)}\n[data-radar-root="individual"] .card-value{font-size:clamp(24px,3vw,34px);font-weight:900;letter-spacing:-.04em;line-height:1.05;margin:14px 0 12px}\n[data-radar-root="individual"] .finance-balance .card-value{color:var(--good)}\n[data-radar-root="individual"] .card-labels{display:flex;gap:6px;flex-wrap:wrap}\n[data-radar-root="individual"] .card-label{display:inline-flex;gap:4px;align-items:center;background:#f7f8fb;border:1px solid #eceff4;border-radius:8px;padding:5px 7px;font-size:9px;color:var(--muted)}\n[data-radar-root="individual"] .card-label b{color:var(--ink)}\n[data-radar-root="individual"] .meter{height:7px;background:#edf0f5;border-radius:999px;overflow:hidden;margin:12px 0 7px}\n[data-radar-root="individual"] .meter span{display:block;height:100%;background:var(--bb-blue);width:72.39%}\n[data-radar-root="individual"] .score-grid-radar{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="individual"] .score-card-radar{grid-column:span 3;min-height:190px}\n[data-radar-root="individual"] .score-card-radar.featured{grid-column:span 6;background:linear-gradient(135deg,#fffef0,#fff);border-color:#e8e584}\n[data-radar-root="individual"] .score-card-radar.featured:before{background:var(--bb-yellow);width:5px}\n[data-radar-root="individual"] .score-card-radar.wide{grid-column:span 3}\n[data-radar-root="individual"] .score-title{font-size:13px;font-weight:850;line-height:1.2;max-width:220px}\n[data-radar-root="individual"] .score-final{font-size:36px;font-weight:950;letter-spacing:-.05em;line-height:1}\n[data-radar-root="individual"] .score-final small{font-size:9px;color:var(--muted);font-weight:850;letter-spacing:.04em;text-transform:uppercase;display:block;margin-bottom:3px}\n[data-radar-root="individual"] .score-data{display:grid;grid-template-columns:repeat(3,1fr);gap:6px;margin-top:14px}\n[data-radar-root="individual"] .score-data div{border-top:1px solid #edf0f4;padding-top:8px}\n[data-radar-root="individual"] .score-data span{display:block;font-size:8px;text-transform:uppercase;letter-spacing:.05em;color:#98a2b3;font-weight:850}\n[data-radar-root="individual"] .score-data b{display:block;font-size:11px;margin-top:2px}\n[data-radar-root="individual"] .score-parts{display:flex;gap:6px;flex-wrap:wrap;margin-top:12px}\n[data-radar-root="individual"] .score-part{background:#f7f8fb;border:1px solid #eceff4;border-radius:8px;padding:5px 7px;font-size:9px;color:var(--muted)}\n[data-radar-root="individual"] .score-part b{color:var(--ink)}\n[data-radar-root="individual"] .score-rule{margin-top:10px;font-size:9px;color:#725000;background:#fff8df;border:1px solid #f1df9b;border-radius:9px;padding:7px 8px}\n@media(max-width:1050px){[data-radar-root="individual"] .finance-in,[data-radar-root="individual"] .finance-out{grid-column:span 3}\n[data-radar-root="individual"] .finance-balance,[data-radar-root="individual"] .finance-ratio{grid-column:span 6}\n[data-radar-root="individual"] .score-card-radar,[data-radar-root="individual"] .score-card-radar.featured,[data-radar-root="individual"] .score-card-radar.wide{grid-column:span 6}\n[data-radar-root="individual"] .context-grid{grid-template-columns:repeat(6,minmax(0,1fr))}\n[data-radar-root="individual"] .identity-card,[data-radar-root="individual"] .cycle-card{grid-column:span 3}\n[data-radar-root="individual"] .income-card,[data-radar-root="individual"] .profile-card,[data-radar-root="individual"] .coverage-card,[data-radar-root="individual"] .window-card{grid-column:span 3}\n[data-radar-root="individual"] .run-header{align-items:flex-start}\n[data-radar-root="individual"] .run-tags{justify-content:flex-start}\n[data-radar-root="individual"] .money-grid{grid-template-columns:1fr 30px 1fr}\n[data-radar-root="individual"] .money-grid .operator:nth-of-type(2){display:none}\n[data-radar-root="individual"] .money-grid .balance-card{grid-column:1/-1}\n[data-radar-root="individual"] .score-layout{grid-template-columns:1fr}\n[data-radar-root="individual"] .winner-card{position:static}\n[data-radar-root="individual"] .hero-grid{grid-template-columns:1fr}}\n@media(max-width:760px){[data-radar-root="individual"] .finance-in,[data-radar-root="individual"] .finance-out,[data-radar-root="individual"] .finance-balance,[data-radar-root="individual"] .finance-ratio,[data-radar-root="individual"] .score-card-radar,[data-radar-root="individual"] .score-card-radar.featured,[data-radar-root="individual"] .score-card-radar.wide{grid-column:1/-1}\n[data-radar-root="individual"] .score-data{grid-template-columns:repeat(3,1fr)}\n[data-radar-root="individual"] .shell{padding:0 13px 60px}\n[data-radar-root="individual"] .topbar-inner{padding:10px 13px}\n[data-radar-root="individual"] .nav{display:none}\n[data-radar-root="individual"] .brand span:last-child{display:none}\n[data-radar-root="individual"] .run-header{margin-top:14px;display:block}\n[data-radar-root="individual"] .run-tags{margin-top:12px}\n[data-radar-root="individual"] .context-grid{grid-template-columns:repeat(2,minmax(0,1fr))}\n[data-radar-root="individual"] .identity-card,[data-radar-root="individual"] .cycle-card,[data-radar-root="individual"] .income-card,[data-radar-root="individual"] .profile-card,[data-radar-root="individual"] .coverage-card,[data-radar-root="individual"] .window-card{grid-column:span 1}\n[data-radar-root="individual"] .section-head{align-items:start;flex-direction:column}\n[data-radar-root="individual"] .pair{grid-template-columns:1fr}\n[data-radar-root="individual"] .link{transform:rotate(90deg)}\n[data-radar-root="individual"] .metrics{display:none}}\n@media(max-width:480px){[data-radar-root="individual"] .context-grid{grid-template-columns:1fr}\n[data-radar-root="individual"] .identity-card,[data-radar-root="individual"] .cycle-card,[data-radar-root="individual"] .income-card,[data-radar-root="individual"] .profile-card,[data-radar-root="individual"] .coverage-card,[data-radar-root="individual"] .window-card{grid-column:span 1}\n[data-radar-root="individual"] .run-title{display:block}\n[data-radar-root="individual"] .run-title .eyebrow{margin-bottom:5px}\n[data-radar-root="individual"] .run-tags{gap:5px}\n[data-radar-root="individual"] .run-tag{font-size:9px;padding:5px 8px}\n[data-radar-root="individual"] .money-grid{grid-template-columns:1fr}\n[data-radar-root="individual"] .operator{transform:rotate(90deg);text-align:center}\n[data-radar-root="individual"] .mini-grid{grid-template-columns:1fr}\n[data-radar-root="individual"] .context-card{min-height:auto}\n[data-radar-root="individual"] .context-main{font-size:22px}\n[data-radar-root="individual"] .score-memory{display:grid;grid-template-columns:1fr 1fr}}\n@media(prefers-reduced-motion:reduce){[data-radar-root="individual"]{scroll-behavior:auto}\n[data-radar-root="individual"],[data-radar-root="individual"] *{transition:none!important;animation:none!important}}\n[data-radar-root="individual"] .context-card,[data-radar-root="individual"] .finance-card,[data-radar-root="individual"] .score-card-radar{padding:16px 17px}\n[data-radar-root="individual"] .card-static-head{display:block;min-height:auto;padding:0}\n[data-radar-root="individual"] .card-static-head .card-summary-main{width:100%}\n[data-radar-root="individual"] .context-card .card-body,[data-radar-root="individual"] .finance-card .card-body,[data-radar-root="individual"] .score-card-radar .card-body{padding:11px 0 0;margin-top:11px;border-top:1px solid #edf0f4}\n[data-radar-root="individual"] .context-card .context-labels{margin-top:0}\n[data-radar-root="individual"] .context-card .context-data{margin-top:10px}\n[data-radar-root="individual"] .context-card .context-foot{padding-top:0;margin-top:8px}\n[data-radar-root="individual"] .finance-card .card-labels{margin-top:0}\n[data-radar-root="individual"] .finance-card .meter{margin-top:4px}\n[data-radar-root="individual"] .score-card-radar .score-data{margin-top:0}\n[data-radar-root="individual"] .score-card-radar .score-parts{margin-top:10px}\n[data-radar-root="individual"] .score-card-radar.featured .card-static-head{background:transparent}\n[data-radar-root="individual"] .score-table-shell{background:#fff;border:1px solid var(--line);border-radius:20px;overflow:hidden;box-shadow:0 8px 26px rgba(27,36,74,.045)}\n[data-radar-root="individual"] .score-table-shell>summary{list-style:none;cursor:pointer;display:flex;align-items:center;justify-content:space-between;gap:16px;padding:16px 18px;background:#fff;font-weight:850}\n[data-radar-root="individual"] .score-table-shell>summary::-webkit-details-marker{display:none}\n[data-radar-root="individual"] .score-table-shell>summary:after{content:"+";display:grid;place-items:center;flex:0 0 28px;height:28px;border-radius:9px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:950}\n[data-radar-root="individual"] .score-table-shell[open]>summary:after{content:"−"}\n[data-radar-root="individual"] .score-summary-main{display:flex;gap:9px;align-items:center;flex-wrap:wrap}\n[data-radar-root="individual"] .score-summary-main strong{font-size:13px}\n[data-radar-root="individual"] .score-summary-meta{font-size:10px;color:var(--muted);font-weight:650}\n[data-radar-root="individual"] .score-winner-tag{display:inline-flex;align-items:center;gap:5px;background:#fffbd9;border:1px solid #ebe47f;color:#655f00;border-radius:999px;padding:5px 8px;font-size:9px;font-weight:850}\n[data-radar-root="individual"] .score-table-body{border-top:1px solid var(--line);padding:14px}\n[data-radar-root="individual"] .score-table-wrap{overflow-x:auto;border:1px solid var(--line);border-radius:14px;background:#fff}\n[data-radar-root="individual"] .score-table{min-width:760px}\n[data-radar-root="individual"] .score-table th,[data-radar-root="individual"] .score-table td{padding:11px 12px}\n[data-radar-root="individual"] .score-table thead th{background:#f8f9fc;top:0;z-index:1}\n[data-radar-root="individual"] .score-table tbody tr:last-child td{border-bottom:0}\n[data-radar-root="individual"] .score-table .theme-cell{min-width:220px;font-weight:800}\n[data-radar-root="individual"] .score-table .theme-cell small{display:block;margin-top:3px;color:var(--muted);font-size:9px;font-weight:650}\n[data-radar-root="individual"] .score-table .num{text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums}\n[data-radar-root="individual"] .score-table .final-cell{text-align:center;font-size:18px;font-weight:950;color:var(--bb-blue-deep)}\n[data-radar-root="individual"] .score-table .winner-row td{background:#fffef0}\n[data-radar-root="individual"] .score-table .winner-row td:first-child{box-shadow:inset 4px 0 0 var(--bb-yellow)}\n[data-radar-root="individual"] .score-inline-badge{display:inline-flex;margin-left:6px;background:#fff8c7;color:#645d00;border:1px solid #ebe47f;border-radius:999px;padding:3px 6px;font-size:8px;font-weight:900;vertical-align:middle;white-space:nowrap}\n[data-radar-root="individual"] .score-subhead{text-align:right!important}\n[data-radar-root="individual"] .score-subhead:first-child{text-align:left!important}\n@media(max-width:760px){[data-radar-root="individual"] .score-table-body{padding:10px}\n[data-radar-root="individual"] .score-table-shell>summary{padding:14px}\n[data-radar-root="individual"] .score-summary-meta{width:100%}}\n[data-radar-root="individual"] .topbar-inner{justify-content:space-between}\n[data-radar-root="individual"] .score-table-simple{min-width:0}\n[data-radar-root="individual"] .score-table-simple th:last-child,[data-radar-root="individual"] .score-table-simple td:last-child{width:140px;text-align:center}\n[data-radar-root="individual"] .score-table-simple .theme-cell{min-width:0}\n[data-radar-root="individual"] .score-table-simple .final-cell{font-size:20px}\n[data-radar-root="individual"] .score-table-static{padding:14px;overflow:hidden}\n[data-radar-root="individual"] .score-table-static .score-table-wrap{height:100%;margin:0}\n[data-radar-root="individual"] .score-table-complete{min-width:980px;width:100%}\n[data-radar-root="individual"] .score-table-complete th,[data-radar-root="individual"] .score-table-complete td{padding:12px 11px}\n[data-radar-root="individual"] .score-table-complete th{font-size:9px}\n[data-radar-root="individual"] .score-table-complete .theme-cell{min-width:230px}\n[data-radar-root="individual"] .score-table-complete .final-cell{font-size:18px;font-weight:900;text-align:center}\n[data-radar-root="individual"] .score-table-complete .winner-row td{background:#fffdf0}\n[data-radar-root="individual"] .score-table-complete .winner-row td:first-child{font-weight:900}\n[data-radar-root="individual"] .score-final-badge{display:inline-flex;align-items:center;justify-content:center;width:38px;height:38px;border-radius:999px;background:var(--bb-yellow);color:#173b67;font-weight:950;line-height:1;box-shadow:inset 0 0 0 1px #e7de58}\n@media(max-width:760px){[data-radar-root="individual"] .score-table-static{padding:10px}\n[data-radar-root="individual"] .score-table-complete{min-width:920px}}\n[data-radar-root="individual"] .top-actions{display:flex;align-items:center;gap:8px;white-space:nowrap}\n[data-radar-root="individual"] .export-btn{border:1px solid var(--bb-blue);background:var(--bb-blue);color:#fff;border-radius:999px;padding:8px 12px;font-weight:800;font-size:12px;cursor:pointer;white-space:nowrap}\n[data-radar-root="individual"] .export-btn:hover{background:var(--bb-blue-deep);border-color:var(--bb-blue-deep)}\n[data-radar-root="individual"] .export-btn:focus-visible,[data-radar-root="individual"] .privacy:focus-visible{outline:3px solid rgba(70,94,255,.22);outline-offset:2px}\n[data-radar-root="individual"] .export-toast{position:fixed;right:22px;bottom:22px;z-index:9999;max-width:360px;background:#171b2f;color:#fff;border-radius:12px;padding:11px 14px;box-shadow:0 12px 36px rgba(0,0,0,.22);font-size:11px;line-height:1.4;opacity:0;transform:translateY(8px);pointer-events:none;transition:.18s ease}\n[data-radar-root="individual"] .export-toast.show{opacity:1;transform:translateY(0)}\n@media(max-width:520px){[data-radar-root="individual"] .top-actions{gap:5px}\n[data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{font-size:10px;padding:7px 9px}\n[data-radar-root="individual"] .export-toast{left:12px;right:12px;bottom:12px;max-width:none}}'


In [ ]:
%%spark
CSS_REFINADO_V1 = r"""
[data-radar-root="individual"]{
  --max:1320px;--radius:20px;--radius-sm:12px;--ink:#14213d;--muted:#667085;
  --bg:#f4f7fb;--card:#fff;--line:#dfe6ef;--soft:#f7f9fc;--good:#087a55;
  --good-bg:#eaf8f1;--blue:#2457a7;--blue-soft:#edf4ff;--yellow:#f5c242;
  --shadow:0 12px 34px rgba(20,33,61,.07);background:
  radial-gradient(circle at 10% -8%,rgba(36,87,167,.10),transparent 34rem),
  linear-gradient(180deg,#fbfcff 0,#f4f7fb 34rem);color:var(--ink);letter-spacing:-.005em
}
[data-radar-root="individual"] .shell{max-width:var(--max);padding:30px 28px 80px}
[data-radar-root="individual"] .topbar{display:none!important}
[data-radar-root="individual"] .run-header{margin:0;display:grid;grid-template-columns:minmax(0,1fr) auto;grid-template-areas:"title actions" "tags tags";gap:20px 24px;align-items:center;padding:29px 30px 24px;border:1px solid rgba(215,224,237,.95);border-radius:26px;background:rgba(255,255,255,.92);box-shadow:0 20px 58px rgba(20,33,61,.09);backdrop-filter:blur(14px)}
[data-radar-root="individual"] .run-title{grid-area:title;display:block;white-space:normal;min-width:0}
[data-radar-root="individual"] .run-title .eyebrow{margin:0;color:#76839b;font-size:10px;letter-spacing:.15em}
[data-radar-root="individual"] .run-title h1{margin:5px 0 0;font-size:34px;line-height:1;letter-spacing:-.055em}
[data-radar-root="individual"] .run-title p{margin:9px 0 0;color:var(--muted);font-size:12px;line-height:1.55}
[data-radar-root="individual"] .top-actions{grid-area:actions;display:flex;gap:8px;justify-content:flex-end}
[data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn,[data-radar-root="individual"] .control-btn{min-height:39px;padding:9px 14px;border:1px solid #d8e0eb;border-radius:11px;background:#fff;color:var(--ink);font-size:11px;font-weight:800;box-shadow:0 1px 2px rgba(16,24,40,.03);cursor:pointer;white-space:nowrap}
[data-radar-root="individual"] .privacy:hover,[data-radar-root="individual"] .control-btn:hover{background:#f7f9fc;border-color:#bdc9d9}
[data-radar-root="individual"] .export-btn{background:var(--ink);border-color:var(--ink);color:#fff}
[data-radar-root="individual"] .export-btn:hover{background:#243453;border-color:#243453}
[data-radar-root="individual"] button:focus-visible,[data-radar-root="individual"] input:focus-visible{outline:3px solid rgba(36,87,167,.22);outline-offset:2px}
[data-radar-root="individual"] .run-tags{grid-area:tags;justify-content:flex-start;max-width:none;gap:7px;padding-top:18px;border-top:1px solid #edf1f6}
[data-radar-root="individual"] .run-tag{min-height:30px;padding:6px 10px;border-color:#e1e7ef;background:#f8fafc;color:#536078;font-size:9px}
[data-radar-root="individual"] .run-tag.base-active,[data-radar-root="individual"] .run-tag.ok,[data-radar-root="individual"] .scenario-active-tag{background:var(--good-bg);border-color:#cfe9dd;color:#176848}
[data-radar-root="individual"] .section{margin-top:36px;scroll-margin-top:24px}
[data-radar-root="individual"] .section-head{margin-bottom:14px;align-items:flex-end;gap:14px}
[data-radar-root="individual"] .section-label{display:block;margin-bottom:3px;color:#8a96aa;font-size:9px;letter-spacing:.15em;text-transform:uppercase}
[data-radar-root="individual"] .section-head h2{font-size:21px;letter-spacing:-.035em}
[data-radar-root="individual"] .section-head p{max-width:760px;margin:6px 0 0;color:var(--muted);font-size:12px;line-height:1.55}
[data-radar-root="individual"] .context-grid,[data-radar-root="individual"] .finance-grid{gap:14px}
[data-radar-root="individual"] .context-card,[data-radar-root="individual"] .finance-card{padding:19px 20px;border:1px solid #e0e7f0;border-radius:19px;background:rgba(255,255,255,.97);box-shadow:0 7px 20px rgba(20,33,61,.04)}
[data-radar-root="individual"] .context-card:before,[data-radar-root="individual"] .finance-card:before{width:0!important}
[data-radar-root="individual"] .context-card:hover,[data-radar-root="individual"] .finance-card:hover{transform:none;border-color:#cdd7e5;box-shadow:0 13px 30px rgba(20,33,61,.065)}
[data-radar-root="individual"] .context-grid .identity-card{grid-column:span 4}
[data-radar-root="individual"] .context-grid .cycle-card{grid-column:span 5}
[data-radar-root="individual"] .context-grid .income-card{grid-column:span 3}
[data-radar-root="individual"] .context-grid .profile-card{grid-column:span 6}
[data-radar-root="individual"] .context-grid .coverage-card{grid-column:span 6}
[data-radar-root="individual"] .context-grid .identity-card,[data-radar-root="individual"] .context-grid .cycle-card,[data-radar-root="individual"] .context-grid .income-card{min-height:190px}
[data-radar-root="individual"] .context-grid .profile-card,[data-radar-root="individual"] .context-grid .coverage-card{min-height:150px}
[data-radar-root="individual"] .context-kicker,[data-radar-root="individual"] .card-kicker{font-size:9px;letter-spacing:.12em;color:#748199}
[data-radar-root="individual"] .context-dot{width:8px;height:8px;background:var(--blue);box-shadow:0 0 0 4px #eaf1fb}
[data-radar-root="individual"] .context-badge,[data-radar-root="individual"] .card-badge{padding:5px 8px;background:#f3f6fa;color:#46556f;border:1px solid #e4e9f1}
[data-radar-root="individual"] .context-badge.good,[data-radar-root="individual"] .card-badge.good{background:var(--good-bg);color:var(--good);border-color:#d1ebdf}
[data-radar-root="individual"] .card-summary-value{font-size:27px;letter-spacing:-.05em;margin:14px 0 15px}
[data-radar-root="individual"] .context-labels{gap:6px}
[data-radar-root="individual"] .micro-label{background:#f7f9fc;border-color:#e8edf4;border-radius:9px;padding:6px 9px}
[data-radar-root="individual"] .data-cell{border-top-color:#ebeff5;padding-top:9px}
[data-radar-root="individual"] .base-switch{margin:12px 0 10px;padding:3px;border:0;background:#edf1f6;border-radius:10px}
[data-radar-root="individual"] .base-switch-btn{min-height:34px;border-radius:8px}
[data-radar-root="individual"] .base-switch-btn.active{color:var(--ink);box-shadow:0 1px 5px rgba(16,24,40,.12)}
[data-radar-root="individual"] .finance-grid .finance-card{grid-column:span 3;min-height:168px}
[data-radar-root="individual"] .finance-grid .finance-balance{background:linear-gradient(180deg,#f9fdfb,#f1faf6)}
[data-radar-root="individual"] .meter{height:6px;background:#e9eef4;margin:13px 0 9px}
[data-radar-root="individual"] .meter span{background:linear-gradient(90deg,var(--blue),#4d83cf)}
[data-radar-root="individual"] .history-note,[data-radar-root="individual"] .composition-note,[data-radar-root="individual"] .recon-rule{padding:15px 17px;border:1px solid #d6e4f7;border-radius:15px;background:var(--blue-soft);color:#324b70;font-size:12px;line-height:1.6}
[data-radar-root="individual"] .history-note strong,[data-radar-root="individual"] .recon-rule strong{color:#173d78}
[data-radar-root="individual"] .history-panel{padding:13px;border:1px solid #dfe6ef;border-radius:20px;background:#fff;box-shadow:var(--shadow)}
[data-radar-root="individual"] .history-event{border-color:#e4e9f1;border-radius:14px;margin:8px 0;overflow:hidden}
[data-radar-root="individual"] .history-event summary{display:grid;grid-template-columns:34px minmax(190px,1fr) minmax(220px,1.4fr);align-items:center;gap:12px;min-height:58px;padding:12px 15px;background:#fbfcfe}
[data-radar-root="individual"] .history-index{display:grid;place-items:center;width:30px;height:30px;border-radius:9px;background:#eaf1fb;color:#1f559d;font-size:11px;font-weight:900}
[data-radar-root="individual"] .history-title{font-size:12px;font-weight:900;color:var(--ink)}
[data-radar-root="individual"] .history-title small{display:block;margin-top:3px;color:#8490a4;font-size:9px;font-weight:700;letter-spacing:.04em}
[data-radar-root="individual"] .history-summary-fact{color:#43516a;font-size:11px;font-weight:750;line-height:1.45;text-align:right}
[data-radar-root="individual"] .history-body{padding:16px;background:#fff;border-top:1px solid #edf1f5}
[data-radar-root="individual"] .fact-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:9px}
[data-radar-root="individual"] .fact-cell{min-width:0;padding:11px 12px;border:1px solid #e8edf4;border-radius:11px;background:#fafbfd}
[data-radar-root="individual"] .fact-cell span{display:block;color:#7b879a;font-size:8px;font-weight:900;letter-spacing:.08em;text-transform:uppercase}
[data-radar-root="individual"] .fact-cell strong{display:block;margin-top:5px;color:var(--ink);font-size:12px;line-height:1.45;overflow-wrap:anywhere}
[data-radar-root="individual"] .fact-use{margin:11px 0 0;padding:10px 12px;border-left:3px solid #8eafd8;background:#f7faff;color:#52627b;font-size:11px;line-height:1.55}
[data-radar-root="individual"] .fact-use b{color:#244f8c}
[data-radar-root="individual"] .history-funnel{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:8px;margin-bottom:10px}
[data-radar-root="individual"] .history-funnel .step{min-height:76px;display:flex;flex-direction:column;justify-content:center;text-align:center}
[data-radar-root="individual"] .explorer,[data-radar-root="individual"] .score-table-shell,[data-radar-root="individual"] .recon-events{padding:14px;border:1px solid #dfe6ef;border-radius:19px;background:#fff;box-shadow:var(--shadow)}
[data-radar-root="individual"] .composition-note{margin-bottom:12px;background:#fffaf0;border-color:#f0dfba;color:#705b2c}
[data-radar-root="individual"] .search{min-height:39px;border-radius:11px;border-color:#d8e0eb;background:#fff;padding:9px 12px}
[data-radar-root="individual"] .search:focus{border-color:#86a6d0;box-shadow:0 0 0 3px rgba(36,87,167,.12)}
[data-radar-root="individual"] .level,[data-radar-root="individual"] .event{border-color:#e4e9f1;border-radius:12px;margin:7px 0}
[data-radar-root="individual"] .level summary,[data-radar-root="individual"] .event summary{min-height:50px;background:#fbfcfd;padding:12px 14px}
[data-radar-root="individual"] .level summary:before{background:#eaf1fb;color:#2457a7}
[data-radar-root="individual"] .nature>summary,[data-radar-root="individual"] .class>summary{background:#fbfcfd}
[data-radar-root="individual"] .table-wrap,[data-radar-root="individual"] .score-table-wrap{border-color:#dfe6ef;border-radius:13px}
[data-radar-root="individual"] th{background:#f7f9fc;color:#718096}
[data-radar-root="individual"] th,[data-radar-root="individual"] td{border-bottom-color:#e9eef4;padding:12px 13px}
[data-radar-root="individual"] .score-table{min-width:980px}
[data-radar-root="individual"] .score-table .winner-row td{background:#f5f9ff}
[data-radar-root="individual"] .score-table .winner-row td:first-child{box-shadow:inset 4px 0 0 var(--blue)}
[data-radar-root="individual"] .score-final-badge{background:#e8f1ff;color:#174c92;box-shadow:inset 0 0 0 1px #cbdcf4}
[data-radar-root="individual"] .recon{display:grid;grid-template-columns:minmax(300px,4fr) minmax(0,8fr);gap:14px;align-items:start;padding:0;border:0;border-radius:0;background:transparent;box-shadow:none}
[data-radar-root="individual"] .recon>.window-card{grid-column:auto;min-height:178px;margin:0}
[data-radar-root="individual"] .recon-rule{margin-bottom:11px}
[data-radar-root="individual"] .funnel{gap:9px;padding:7px 0 11px;margin:0 0 7px;border-bottom:1px solid #edf1f5}
[data-radar-root="individual"] .step{padding:11px 10px;border-radius:11px;background:#f4f7fb;color:#526078}
[data-radar-root="individual"] .step b{font-size:18px;color:var(--ink)}
[data-radar-root="individual"] .privacy-pending [data-private]{visibility:hidden}
[data-radar-root="individual"] [data-private],[data-radar-root="individual"] .masked-value{filter:none!important;user-select:auto!important}
[data-radar-root="individual"] .masked-value{font-variant-ligatures:none;letter-spacing:.04em}
@media(max-width:1050px){
  [data-radar-root="individual"] .shell{max-width:960px;padding-top:22px}
  [data-radar-root="individual"] .context-grid .identity-card,[data-radar-root="individual"] .context-grid .cycle-card,[data-radar-root="individual"] .context-grid .income-card,[data-radar-root="individual"] .context-grid .profile-card,[data-radar-root="individual"] .context-grid .coverage-card{grid-column:span 6}
  [data-radar-root="individual"] .finance-grid .finance-card{grid-column:span 6}
  [data-radar-root="individual"] .recon{grid-template-columns:1fr}
}
@media(max-width:780px){
  [data-radar-root="individual"] .shell{padding:14px 13px 54px}
  [data-radar-root="individual"] .run-header{grid-template-columns:1fr;grid-template-areas:"title" "actions" "tags";gap:16px;padding:20px;border-radius:20px}
  [data-radar-root="individual"] .run-title h1{font-size:28px}
  [data-radar-root="individual"] .top-actions{justify-content:flex-start;width:100%}
  [data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{flex:1}
  [data-radar-root="individual"] .section-head{align-items:flex-start;flex-direction:column}
  [data-radar-root="individual"] .context-grid,[data-radar-root="individual"] .finance-grid{grid-template-columns:1fr}
  [data-radar-root="individual"] .context-grid .identity-card,[data-radar-root="individual"] .context-grid .cycle-card,[data-radar-root="individual"] .context-grid .income-card,[data-radar-root="individual"] .context-grid .profile-card,[data-radar-root="individual"] .context-grid .coverage-card,[data-radar-root="individual"] .finance-grid .finance-card{grid-column:1/-1;min-height:auto}
  [data-radar-root="individual"] .controls{width:100%}
  [data-radar-root="individual"] .search{width:100%;min-width:0}
  [data-radar-root="individual"] .history-event summary{grid-template-columns:34px 1fr}
  [data-radar-root="individual"] .history-summary-fact{grid-column:2;text-align:left}
  [data-radar-root="individual"] .fact-grid{grid-template-columns:1fr}
  [data-radar-root="individual"] .history-funnel{grid-template-columns:repeat(2,minmax(0,1fr))}
}
@media(max-width:480px){
  [data-radar-root="individual"] .run-header{padding:18px 16px}
  [data-radar-root="individual"] .run-title h1{font-size:26px}
  [data-radar-root="individual"] .top-actions{display:grid;grid-template-columns:1fr 1fr}
  [data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{width:100%}
  [data-radar-root="individual"] .context-data,[data-radar-root="individual"] .mini-grid{grid-template-columns:1fr}
  [data-radar-root="individual"] .history-funnel{grid-template-columns:1fr 1fr}
  [data-radar-root="individual"] .score-table{min-width:820px}
}
"""


In [ ]:
%%spark
CSS_DASHBOARD = re.sub(r'\s+', ' ', f'{CSS_APROVADO}{CSS_COMPLEMENTO}{CSS_REFINADO_V1}').strip().replace('filter:blur(5px);', 'filter:none;')
radar_root_id = f'radar-financeiro-individual-{CD_CLI}-{DATA_EXECUCAO.strftime("%Y%m%d")}'


In [ ]:
%%spark
moedas_identificadas = 'BRL' if res_dict.get('FL_SOMENTE_BRL') == 'S' else 'Indisponível na preparação local'
quantidade_mapa_categorias = len(LINHAS_CATEGORIAS)
periodo_rotulo = f'{periodo} ciclo' if periodo == 1 else f'{periodo} ciclos'

html_dashboard = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Radar Financeiro</title>
<style>{CSS_DASHBOARD}</style>
</head>
<body>
<section class="privacy-pending" data-radar-root="individual" id="{radar_root_id}" tabindex="-1">
<a class="skip" href="#{radar_root_id}">Ir para o conteúdo</a>
<main class="shell" data-role="content">

<section aria-label="Radar Financeiro" class="run-header">
  <div class="run-title"><div class="eyebrow">Memória factual da execução</div><h1>Radar Financeiro</h1><p>Leitura do ciclo, composição financeira e prioridade de orientação com fatos produzidos pelo motor.</p></div>
  <div class="top-actions"><button aria-pressed="false" class="privacy" data-role="privacy-button" type="button">Mostrar dados</button><button class="export-btn" data-role="export-button" type="button">Exportar HTML</button></div>
  <div aria-label="Condições da execução" class="run-tags">
    {tag_execucao('CPF único', res_dict.get('FL_CPF_UNICO'))}
    {tag_execucao('Conta única', res_dict.get('FL_CONTA_ELEGIVEL_UNICA'))}
    {tag_execucao('Somente BRL', res_dict.get('FL_SOMENTE_BRL'))}
    {tag_execucao('Agro', res_dict.get('FL_TEM_MOV_AGRO'))}
    <span class="run-tag">Período: {esc(periodo_rotulo)}</span>
    <span data-radar-slot-html="tag_base_html">{payload_padrao['tag_base_html']}</span>
    <span data-radar-slot-html="tag_pontuacao_html">{payload_padrao['tag_pontuacao_html']}</span>
    <span class="run-tag">Prioridade: <b data-radar-slot="prioridade_orientacao">{esc(payload_padrao['prioridade_orientacao'])}</b></span>
  </div>
</section>

<section class="section" data-section="contexto">
  <div class="section-head"><div><span class="section-label">Contexto</span><h2>Contexto da análise</h2><p>Identidade, ciclo, base principal, perfil e cobertura observados nesta execução.</p></div></div>
  <div class="context-grid">
    <article class="context-card identity-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Cliente</span><span aria-hidden="true" class="context-dot"></span></div><div class="card-summary-value" data-private="identifier">{esc(res_dict.get('CD_CLI'))}</div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label"><code>NR_AG_TITR</code> <b data-private="identifier">{esc(res_cta['NR_AG_TITR'])}</b></span><span class="micro-label"><code>CD_CT_TITR</code> <b data-private="identifier">{esc(res_cta['CD_CT_TITR'])}</b></span></div><div class="context-labels"><span class="micro-label"><code>CD_UOR_CC_NORM</code> <b data-private="identifier">{esc(cta_norm_row['CD_UOR_CC_NORM'])}</b></span><span class="micro-label"><code>NR_CC_NORM</code> <b data-private="identifier">{esc(cta_norm_row['NR_CC_NORM'])}</b></span></div></div></article>
    <article class="context-card cycle-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Ciclo</span><span class="context-badge">Dia {fmt_inteiro(res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK'))}</span></div><div class="card-summary-value date-range">{fmt_data(res_dict.get('DT_REF_INI'), True)} <span>→</span> {fmt_data(res_dict.get('DT_REF_FIM'), True)}</div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Janela oficial <b>{fmt_data(res_dict.get('DT_REF_INI'))} → {fmt_data(res_dict.get('DT_REF_FIM'))}</b></span><span class="micro-label">Contexto <b>{fmt_data(dt_contexto_ini)} → {fmt_data(dt_contexto_fim)}</b></span><span class="micro-label">Fallback <b>{fallback_usado}</b></span></div><div class="context-foot">Janela oficial e contexto de reconciliação são períodos distintos.</div></div></article>
    <article class="context-card income-card" data-radar-component="base-financeira"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker" data-radar-slot="base_kicker">{payload_padrao['base_kicker']}</span><span class="context-badge good">BRL</span></div><div aria-live="polite" class="card-summary-value" data-private="money" data-radar-slot="valor_base">{payload_padrao['valor_base']}</div></div></div><div aria-label="Selecionar base financeira" class="base-switch" role="group"><button aria-pressed="true" class="base-switch-btn active" data-radar-mode="RENDA_PRESUMIDA" type="button">Renda Presumida</button><button aria-pressed="false" class="base-switch-btn" data-radar-mode="ENTRADAS_REALIZADAS" type="button">Entradas Realizadas</button></div><div class="card-body"><div class="context-data single"><div class="data-cell"><span data-radar-slot="referencia_rotulo">{payload_padrao['referencia_rotulo']}</span><strong data-radar-slot="referencia_valor">{payload_padrao['referencia_valor']}</strong></div></div><div class="context-foot"><span class="scenario-active-tag" data-radar-slot="base_badge">{payload_padrao['base_badge']}</span></div></div></article>
    <article class="context-card profile-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Perfil</span><span class="context-badge subtle">Ref. {fmt_data(res_dict.get('DT_REF_PRFL'), True)}</span></div><div class="card-summary-value">{esc(res_dict.get('NM_MAC_PRFL_CLI'))}</div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Macro <b>{esc(res_dict.get('CD_MAC_PRFL_CLI'))}</b></span><span class="micro-label">Microperfil <b>{esc(res_dict.get('NM_MIC_PRFL_CLI'))}</b></span><span class="micro-label">Código micro <b>{esc(res_dict.get('CD_MIC_PRFL_CLI'))}</b></span></div><div class="context-foot">O macroperfil participa do motor; o microperfil é contexto.</div></div></article>
    <article class="context-card coverage-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Base analisada</span><span class="context-badge good">{('Somente BRL' if res_dict.get('FL_SOMENTE_BRL') == 'S' else 'Cobertura monetária verificada')}</span></div><div class="card-summary-value">{fmt_inteiro(res_dict.get('QT_TRANS_TOTAL'))} <small>transações</small></div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Entradas <b>{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))}</b></span><span class="micro-label">Saídas <b>{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))}</b></span><span class="micro-label">Agro <b>{esc(res_dict.get('FL_TEM_MOV_AGRO'))}</b></span></div><div class="context-foot">Moedas identificadas: {esc(moedas_identificadas)}</div></div></article>
  </div>
</section>

<section class="section" data-section="resultado">
  <div class="section-head"><div><span class="section-label">Resultado</span><h2>Resumo financeiro</h2><p data-private="money" data-radar-slot="base_visao">{esc(payload_padrao['base_visao'])}</p></div></div>
  <div class="finance-grid">
    <article class="finance-card finance-in"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker" data-radar-slot="entrada_rotulo">{payload_padrao['entrada_rotulo']}</span><span class="card-badge">{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))} transações</span></div><div class="card-summary-value" data-private="money" data-radar-slot="entrada_valor">{payload_padrao['entrada_valor']}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Base previamente calculada para esta visão</span></div></div></article>
    <article class="finance-card finance-out"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saídas realizadas</span><span class="card-badge">{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))} transações</span></div><div class="card-summary-value" data-private="money">{fmt_moeda(res_dict.get('VL_TRANS_SAI'))}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Fato efetivo do ciclo</span></div></div></article>
    <article class="finance-card finance-balance"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Resultado orçamentário</span><span class="card-badge good" data-radar-slot="saldo_status">{payload_padrao['saldo_status']}</span></div><div class="card-summary-value good" data-private="money" data-radar-slot="saldo_valor">{payload_padrao['saldo_valor']}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Situação pré-calculada pelo motor</span></div></div></article>
    <article class="finance-card finance-ratio"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saídas / Base</span></div><div class="card-summary-value" data-radar-slot="razao_valor">{payload_padrao['razao_valor']}</div></div></div><div class="card-body"><div aria-label="Relação Saídas sobre Base Financeira" class="meter"><span data-radar-meter="true" style="width:{payload_padrao['largura_medidor']}"></span></div><div class="card-labels"><span class="card-label">Percentual pré-calculado para a visão</span></div></div></article>
  </div>
</section>

<section class="section" data-section="historico">
  <div class="section-head"><div><span class="section-label">Memória da execução</span><h2>Como chegamos até aqui</h2><p>Variável observada, valor real preservado e consequência no fluxo.</p></div><div class="controls"><button class="control-btn" data-action="expand-history" type="button">Expandir tudo</button><button class="control-btn" data-action="collapse-history" type="button">Recolher</button></div></div>
  <div class="history-note"><strong>Leitura factual.</strong> Esta seção apenas apresenta fatos já disponíveis na V6; não consulta fontes, não cria métricas e não recalcula o resultado.</div>
  <div class="history-panel" data-role="history-panel">
    <details class="history-event" data-default-open="true"><summary><span class="history-index">01</span><span class="history-title">Formação do público<small>Execução e elegibilidade temporal</small></span><span class="history-summary-fact">{fmt_data(res_dict.get('DT_EXEA'))} · cliente <span data-private="identifier">{esc(res_dict.get('CD_CLI'))}</span></span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>DT_EXEA</span><strong>{fmt_data(res_dict.get('DT_EXEA'))}</strong></div><div class="fact-cell"><span>DT_MES_EXEA</span><strong>{fmt_data(res_dict.get('DT_MES_EXEA'))}</strong></div><div class="fact-cell"><span>DATA_INICIAL_PUBLICO</span><strong>{fmt_data(DATA_INICIAL_PUBLICO)}</strong></div><div class="fact-cell"><span>DATA_FINAL_EXCLUSIVA_PUBLICO</span><strong>{fmt_data(DATA_FINAL_EXCLUSIVA_PUBLICO)}</strong></div><div class="fact-cell"><span>TS_INCL_TRAN_REF</span><strong>{fmt_data_hora(res_dict.get('TS_INCL_TRAN_REF'))}</strong></div><div class="fact-cell"><span>CD_CLI</span><strong data-private="identifier">{esc(res_dict.get('CD_CLI'))}</strong></div></div><p class="fact-use"><b>Como foi usada:</b> o público já foi formado com <code>CD_TIP_PSS = 1</code>, <code>CD_EST_TRAN_INST = 0</code> e a janela temporal acima.</p></div></details>
    <details class="history-event"><summary><span class="history-index">02</span><span class="history-title">CPF e conta<small>Unicidade e conta elegível</small></span><span class="history-summary-fact">CPF único: {esc(res_dict.get('FL_CPF_UNICO'))} · conta única: {esc(res_dict.get('FL_CONTA_ELEGIVEL_UNICA'))}</span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>FL_CPF_UNICO</span><strong>{esc(res_dict.get('FL_CPF_UNICO'))}</strong></div><div class="fact-cell"><span>CD_CPF</span><strong data-private="identifier">{esc(res_dict.get('CD_CPF'))}</strong></div><div class="fact-cell"><span>FL_CONTA_ELEGIVEL_UNICA</span><strong>{esc(res_dict.get('FL_CONTA_ELEGIVEL_UNICA'))}</strong></div><div class="fact-cell"><span>NR_AG_TITR · CD_CT_TITR</span><strong data-private="identifier">{esc(res_cta['NR_AG_TITR'])} · {esc(res_cta['CD_CT_TITR'])}</strong></div><div class="fact-cell"><span>CD_UOR_CC_NORM · NR_CC_NORM</span><strong data-private="identifier">{esc(cta_norm_row['CD_UOR_CC_NORM'])} · {esc(cta_norm_row['NR_CC_NORM'])}</strong></div><div class="fact-cell"><span>Critério da conta</span><strong>NR_MCA_PCT_OPB = 999999999 · CD_PRD = 6</strong></div></div><p class="fact-use"><b>Como foi usada:</b> os indicadores registram o estado observado. Um valor “N” não atribui causa entre ausência e multiplicidade.</p></div></details>
    <details class="history-event"><summary><span class="history-index">03</span><span class="history-title">Ciclo e janela<small>Período oficial e contexto</small></span><span class="history-summary-fact">{fmt_data(res_dict.get('DT_REF_INI'))} → {fmt_data(res_dict.get('DT_REF_FIM'))} · contexto ±{DIAS_CONTEXTO_RECONCILIACAO} dias</span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>TS_DD_INC_MM_CLC_BLC_REF</span><strong>{fmt_data_hora(res_dict.get('TS_DD_INC_MM_CLC_BLC_REF'))}</strong></div><div class="fact-cell"><span>DD_INC_MM_CLC_BLC</span><strong>{fmt_inteiro(res_dict.get('DD_INC_MM_CLC_BLC'))}</strong></div><div class="fact-cell"><span>DD_INC_MM_CLC_BLC_FALLBACK</span><strong>{fmt_inteiro(res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK'))} · usado: {fallback_usado}</strong></div><div class="fact-cell"><span>PERÍODO</span><strong>{esc(periodo_rotulo)}</strong></div><div class="fact-cell"><span>DT_REF_INI · DT_REF_FIM</span><strong>{fmt_data(res_dict.get('DT_REF_INI'))} → {fmt_data(res_dict.get('DT_REF_FIM'))}</strong></div><div class="fact-cell"><span>Contexto de reconciliação</span><strong>{fmt_data(dt_contexto_ini)} → {fmt_data(dt_contexto_fim)}</strong></div></div><p class="fact-use"><b>Como foi usada:</b> a janela oficial define os fatos financeiros; o contexto ampliado serve exclusivamente à reconciliação.</p></div></details>
    <details class="history-event" data-default-open="true"><summary><span class="history-index">04</span><span class="history-title">Renda presumida e cenários<small>Base principal e comparação</small></span><span class="history-summary-fact" data-private="money" data-radar-slot="base_visao">{esc(payload_padrao['base_visao'])}</span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>CD_CPF utilizado</span><strong data-private="identifier">{esc(res_dict.get('CD_CPF'))}</strong></div><div class="fact-cell"><span>DT_REN_PRES_REF</span><strong>{fmt_data(res_dict.get('DT_REN_PRES_REF'))}</strong></div><div class="fact-cell"><span>VL_REN_PRES</span><strong data-private="money">{fmt_moeda(res_dict.get('VL_REN_PRES'))}</strong></div><div class="fact-cell"><span>CD_CENARIO ativo</span><strong data-radar-slot="cenario_codigo">{esc(payload_padrao['cenario_codigo'])}</strong></div><div class="fact-cell"><span>BASE_FINANCEIRA ativa</span><strong data-private="money" data-radar-slot="historico_base_financeira">{payload_padrao['historico_base_financeira']}</strong></div><div class="fact-cell"><span>Visão</span><strong data-radar-slot="cenario_rotulo">{esc(payload_padrao['cenario_rotulo'])}</strong></div></div><p class="fact-use"><b>Como foi usada:</b> <code>RENDA_PRESUMIDA</code> abre como visão principal. <code>ENTRADAS_REALIZADAS</code> exibe o snapshot comparativo já preparado pelo motor.</p></div></details>
    <details class="history-event"><summary><span class="history-index">05</span><span class="history-title">Perfil<small>Macroperfil no motor; microperfil como contexto</small></span><span class="history-summary-fact">{esc(res_dict.get('NM_MAC_PRFL_CLI'))} · {esc(res_dict.get('NM_MIC_PRFL_CLI'))}</span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>DT_REF_PRFL</span><strong>{fmt_data(res_dict.get('DT_REF_PRFL'))}</strong></div><div class="fact-cell"><span>CD_MAC_PRFL_CLI</span><strong>{esc(res_dict.get('CD_MAC_PRFL_CLI'))}</strong></div><div class="fact-cell"><span>NM_MAC_PRFL_CLI</span><strong>{esc(res_dict.get('NM_MAC_PRFL_CLI'))}</strong></div><div class="fact-cell"><span>CD_MIC_PRFL_CLI</span><strong>{esc(res_dict.get('CD_MIC_PRFL_CLI'))}</strong></div><div class="fact-cell"><span>NM_MIC_PRFL_CLI</span><strong>{esc(res_dict.get('NM_MIC_PRFL_CLI'))}</strong></div></div><p class="fact-use"><b>Como foi usada:</b> o macroperfil participa da pontuação; o microperfil é exibido sem atribuição causal.</p></div></details>
    <details class="history-event"><summary><span class="history-index">06</span><span class="history-title">Movimentações e reconciliação<small>Pares e linhas preservados como conceitos distintos</small></span><span class="history-summary-fact">{qt_raw} oficiais · {qt_pares_exatos_oficiais} pares exatos · {qt_pares_borda} pares de borda · {qt_efetivo} efetivas</span></summary><div class="history-body"><div class="history-funnel"><div class="step"><b>{qt_raw}</b>QT_OFICIAIS</div><div class="step"><b>{qt_pares_exatos_oficiais}</b>QT_PARES_EXATOS</div><div class="step"><b>{qt_pares_borda}</b>QT_PARES_BORDA</div><div class="step"><b>{qt_efetivo}</b>QT_EFETIVAS</div></div><p class="fact-use"><b>Como foi usada:</b> transferências entre contas próprias só formam par quando os códigos <code>NR_MCA_PCT_OPB</code> são conhecidos e diferentes, além dos demais critérios vigentes. As quatro medidas acima são apresentadas diretamente, sem conversão entre pares e movimentos.</p></div></details>
    <details class="history-event"><summary><span class="history-index">07</span><span class="history-title">Cobertura monetária e classificação<small>Universo efetivo preparado pelo motor</small></span><span class="history-summary-fact">{fmt_inteiro(res_dict.get('QT_TRANS_TOTAL'))} movimentos · moedas: {esc(moedas_identificadas)}</span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>FL_SOMENTE_BRL</span><strong>{esc(res_dict.get('FL_SOMENTE_BRL'))}</strong></div><div class="fact-cell"><span>FL_TEM_MOV_AGRO</span><strong>{esc(res_dict.get('FL_TEM_MOV_AGRO'))}</strong></div><div class="fact-cell"><span>QT_TRANS_TOTAL</span><strong>{fmt_inteiro(res_dict.get('QT_TRANS_TOTAL'))}</strong></div><div class="fact-cell"><span>QT_TRANS_ENT · QT_TRANS_SAI</span><strong>{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))} · {fmt_inteiro(res_dict.get('QT_TRANS_SAI'))}</strong></div><div class="fact-cell"><span>Moedas identificadas</span><strong>{esc(moedas_identificadas)}</strong></div><div class="fact-cell"><span>Mapa de categorias</span><strong>{fmt_inteiro(quantidade_mapa_categorias)} linhas já preparadas</strong></div></div><p class="fact-use"><b>Como foi usada:</b> composição, classes, categorias e movimentos sem classificação permanecem disponíveis no detalhamento abaixo, sem nova agregação.</p></div></details>
    <details class="history-event"><summary><span class="history-index">08</span><span class="history-title">Orçamento, percentuais e prioridade<small>Snapshot do cenário ativo</small></span><span class="history-summary-fact"><span data-radar-slot="historico_status_final">{esc(payload_padrao['historico_status_final'])}</span> · prioridade: <span data-radar-slot="prioridade_orientacao">{esc(payload_padrao['prioridade_orientacao'])}</span></span></summary><div class="history-body"><div class="fact-grid"><div class="fact-cell"><span>VL_ENT_TOTAL · VL_SAI_TOTAL</span><strong data-private="money">{fmt_moeda(res_dict.get('VL_ENT_TOTAL'))} · {fmt_moeda(res_dict.get('VL_SAI_TOTAL'))}</strong></div><div class="fact-cell"><span>BASE_FINANCEIRA</span><strong data-private="money" data-radar-slot="historico_base_financeira">{payload_padrao['historico_base_financeira']}</strong></div><div class="fact-cell"><span>VL_RES_ORC</span><strong data-private="money" data-radar-slot="historico_resultado_orcamentario">{payload_padrao['historico_resultado_orcamentario']}</strong></div><div class="fact-cell"><span>PC_SAI_ENT</span><strong data-radar-slot="historico_pc_saida_base">{payload_padrao['historico_pc_saida_base']}</strong></div><div class="fact-cell"><span>TX_STS_FINAL</span><strong data-radar-slot="historico_status_final">{esc(payload_padrao['historico_status_final'])}</strong></div><div class="fact-cell"><span>FL_PONTUACAO_COMPLETA</span><strong data-radar-slot="historico_pontuacao_completa">{esc(payload_padrao['historico_pontuacao_completa'])}</strong></div><div class="fact-cell"><span>NR_PONT_MAX · QT_TEMAS_PONT_MAX</span><strong><span data-radar-slot="historico_pont_max">{payload_padrao['historico_pont_max']}</span> · <span data-radar-slot="historico_qt_temas_max">{payload_padrao['historico_qt_temas_max']}</span></strong></div><div class="fact-cell"><span>CD_TEMA_VENCEDOR · TX_TEMA_VENCEDOR</span><strong><span data-radar-slot="historico_cd_tema">{payload_padrao['historico_cd_tema']}</span> · <span data-radar-slot="prioridade_orientacao">{esc(payload_padrao['prioridade_orientacao'])}</span></strong></div></div><p class="fact-use"><b>Como foi usada:</b> o valor é uma <b>prioridade de orientação</b>, não risco, nota de crédito ou escala linear de gravidade.</p></div></details>
  </div>
</section>

<section class="section" data-section="composicao"><div class="section-head"><div><span class="section-label">Explicabilidade</span><h2>Composição</h2><p>Drill-down do universo efetivo por natureza, classe, categoria e movimentação.</p></div><div class="controls"><input aria-label="Buscar na composição" class="search" data-role="composition-search" placeholder="Buscar classe ou categoria" type="search"/><button class="control-btn" data-action="expand-composition" type="button">Expandir tudo</button><button class="control-btn" data-action="collapse-composition" type="button">Recolher</button></div></div><div class="composition-note">Valores e tratamentos abaixo são fatos já preparados pela V6. O HTML não reclassifica nem recalcula movimentações.</div><div class="explorer" data-role="composition-explorer">{render_composicao()}</div></section>

<section class="section" data-section="pontuacao"><div class="section-head"><div><span class="section-label">Motor</span><h2>Pontuação · prioridade de orientação</h2><p data-private="money" data-radar-slot="base_visao">{esc(payload_padrao['base_visao'])}</p></div><span class="scenario-active-tag" data-radar-slot="base_badge">{payload_padrao['base_badge']}</span></div><div aria-label="Tabela completa de pontuação" class="score-table-shell score-table-static"><div class="score-table-wrap"><table class="score-table score-table-complete"><thead><tr><th>Tema</th><th class="num">Valor temático</th><th class="num">% base</th><th class="num">Referência</th><th class="num">Concentração</th><th class="num">Orçamento</th><th class="num">Perfil</th><th class="score-subhead">Prioridade final</th></tr></thead><tbody data-radar-slot-html="pontuacao_html">{payload_padrao['pontuacao_html']}</tbody></table></div></div></section>

<section class="section" data-section="reconciliacao"><div class="section-head"><div><span class="section-label">Controle</span><h2>Reconciliação detalhada</h2><p>Janela oficial, contexto externo e pares efetivamente selecionados pela V6.</p></div><div class="controls"><button class="control-btn" data-action="expand-reconciliation" type="button">Expandir eventos</button><button class="control-btn" data-action="collapse-reconciliation" type="button">Recolher</button></div></div><div class="recon"><article class="context-card window-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Janela de contexto</span><span class="context-badge">±{DIAS_CONTEXTO_RECONCILIACAO} dias</span></div><div class="card-summary-value date-range">{fmt_data(dt_contexto_ini, True)} <span>→</span> {fmt_data(dt_contexto_fim, True)}</div></div></div><div class="card-body"><div class="context-data"><div class="data-cell"><span>Janela oficial</span><strong>{fmt_data(dt_ini_j, True)} → {fmt_data(dt_fim_j, True)}</strong></div><div class="data-cell"><span>Contexto</span><strong>{fmt_data(dt_contexto_ini, True)} → {fmt_data(dt_contexto_fim, True)}</strong></div></div></div></article><div class="recon-events"><div class="recon-rule"><strong>Regra V6:</strong> além dos critérios vigentes, os bancos precisam ser conhecidos e diferentes nos dois lados (<code>NR_MCA_PCT_OPB</code>).</div><div class="funnel"><div class="step"><b>{qt_raw}</b>oficiais</div><div>→</div><div class="step"><b>{qt_pares_exatos_oficiais}</b>pares exatos</div><div>→</div><div class="step"><b>{qt_pares_borda}</b>pares de borda</div><div>→</div><div class="step"><b>{qt_efetivo}</b>efetivas</div></div>{render_eventos_reconciliacao()}{render_contexto_externo()}</div></div></section>

</main>
<div aria-live="polite" class="export-toast" data-role="export-toast" role="status">HTML já salvo no workdir: radar_financeiro_{res_dict.get('CD_CLI')}_{DATA_EXECUCAO.strftime('%Y%m%d')}.html</div>
<script>
(function(){{
  const root=document.getElementById('{radar_root_id}');
  if(!root||root.dataset.radarBound==='1')return;
  root.dataset.radarBound='1';
  const one=selector=>root.querySelector(selector);
  const all=selector=>root.querySelectorAll(selector);
  function bind(element,eventName,handler){{if(!element||element.dataset.radarEventBound==='1')return;element.dataset.radarEventBound='1';element.addEventListener(eventName,handler);}}
  const scenarioPayloads={payload_cenarios_json};
  Object.values(scenarioPayloads).forEach(payload=>Object.freeze(payload));
  Object.freeze(scenarioPayloads);
  let dataVisible=false;
  let privacyReady=false;
  let privacyTargets=[];
  const originalValues=new WeakMap();
  const privacyButton=one('[data-role="privacy-button"]');
  function getPrivateTargets(scope=root){{if(!scope)return[];return Array.from(scope.querySelectorAll('[data-private]')).filter(element=>!element.closest('[data-section="reconciliacao"]'));}}
  function capturePrivateTargets(forceScenarioSlots=false){{privacyTargets=getPrivateTargets(root);privacyTargets.forEach(element=>{{const scenarioSlot=element.matches('[data-radar-slot][data-private]');if(!originalValues.has(element)||(forceScenarioSlots&&scenarioSlot))originalValues.set(element,element.innerHTML);}});}}
  function renderPrivacy(){{privacyTargets.forEach(element=>{{const original=originalValues.get(element);if(original===undefined)return;element.innerHTML=dataVisible?original:'****';element.classList.toggle('masked-value',!dataVisible);}});root.classList.remove('privacy-pending');privacyButton.textContent=dataVisible?'Ocultar dados':'Mostrar dados';privacyButton.setAttribute('aria-pressed',String(dataVisible));}}
  function applyScenario(mode){{
    const payload=scenarioPayloads[mode];if(!payload)return;
    all('[data-radar-slot]').forEach(element=>{{const name=element.dataset.radarSlot;element.textContent=String(payload[name]??'—');}});
    all('[data-radar-slot-html]').forEach(element=>{{const name=element.dataset.radarSlotHtml;element.innerHTML=payload[name]||'';}});
    const meter=one('[data-radar-meter]');if(meter)meter.style.width=payload.largura_medidor;
    root.dataset.radarScenario=mode;
    all('[data-radar-mode]').forEach(button=>{{const active=button.dataset.radarMode===mode;button.classList.toggle('active',active);button.setAttribute('aria-pressed',String(active));}});
    if(privacyReady){{capturePrivateTargets(true);renderPrivacy();}}
  }}
  all('[data-radar-mode]').forEach(button=>bind(button,'click',()=>applyScenario(button.dataset.radarMode)));
  applyScenario('RENDA_PRESUMIDA');
  capturePrivateTargets();privacyReady=true;renderPrivacy();
  bind(privacyButton,'click',()=>{{dataVisible=!dataVisible;renderPrivacy();}});
  all('details').forEach(detail=>{{detail.open=detail.dataset.defaultOpen==='true';}});
  function setOpen(container,open){{if(container)container.querySelectorAll('details').forEach(detail=>detail.open=open);}}
  const history=one('[data-role="history-panel"]');
  const composition=one('[data-role="composition-explorer"]');
  const reconciliation=one('[data-section="reconciliacao"]');
  bind(one('[data-action="expand-history"]'),'click',()=>setOpen(history,true));
  bind(one('[data-action="collapse-history"]'),'click',()=>setOpen(history,false));
  bind(one('[data-action="expand-composition"]'),'click',()=>setOpen(composition,true));
  bind(one('[data-action="collapse-composition"]'),'click',()=>setOpen(composition,false));
  bind(one('[data-action="expand-reconciliation"]'),'click',()=>setOpen(reconciliation,true));
  bind(one('[data-action="collapse-reconciliation"]'),'click',()=>setOpen(reconciliation,false));
  const compSearch=one('[data-role="composition-search"]');
  bind(compSearch,'input',()=>{{const q=compSearch.value.trim().toLowerCase();composition.querySelectorAll('details.category').forEach(detail=>{{const hit=!q||detail.textContent.toLowerCase().includes(q);detail.style.display=hit?'':'none';if(q&&hit){{detail.open=true;let parent=detail.parentElement.closest('details');while(parent&&root.contains(parent)){{parent.open=true;parent=parent.parentElement.closest('details');}}}}}});}});
  const exportToast=one('[data-role="export-toast"]');
  function showExportToast(message){{if(!exportToast)return;exportToast.textContent=message;exportToast.classList.add('show');window.clearTimeout(showExportToast._timer);showExportToast._timer=window.setTimeout(()=>exportToast.classList.remove('show'),5000);}}
  const filename='radar_financeiro_{res_dict.get('CD_CLI')}_{DATA_EXECUCAO.strftime('%Y%m%d')}.html';
  const exportButton=one('[data-role="export-button"]');
  bind(exportButton,'click',()=>{{
    if(root.parentElement===document.body){{
      const clone=document.documentElement.cloneNode(true);
      const cloneRoot=clone.querySelector('[data-radar-root="individual"]');
      const cloneTargets=getPrivateTargets(cloneRoot);
      cloneTargets.forEach((element,index)=>{{const original=originalValues.get(privacyTargets[index]);if(original!==undefined){{element.innerHTML=original;element.classList.remove('masked-value');}}}});
      cloneRoot.classList.add('privacy-pending');
      const clonePrivacy=cloneRoot.querySelector('[data-role="privacy-button"]');
      if(clonePrivacy){{clonePrivacy.textContent='Mostrar dados';clonePrivacy.setAttribute('aria-pressed','false');}}
      clone.querySelectorAll('[data-radar-bound],[data-radar-event-bound]').forEach(element=>{{element.removeAttribute('data-radar-bound');element.removeAttribute('data-radar-event-bound');}});
      const source='<!DOCTYPE html>\\n'+clone.outerHTML;
      const url=URL.createObjectURL(new Blob([source],{{type:'text/html;charset=utf-8'}}));
      const link=document.createElement('a');link.href=url;link.download=filename;link.click();
      window.setTimeout(()=>URL.revokeObjectURL(url),1000);
      showExportToast('Cópia exportada: '+filename);
    }}else{{showExportToast('HTML salvo no workdir: '+filename);}}
  }});
}})();
</script>
</section>
</body></html>"""


In [ ]:
%%spark
html_dashboard = re.sub(r'>\s+<', '><', html_dashboard).strip()
html_bytes = html_dashboard.encode('utf-8')
tamanho_html = len(html_bytes)
checksum_html = hashlib.sha256(html_bytes).hexdigest()
if tamanho_html > LIMITE_PAYLOAD_BYTES:
    raise RuntimeError(f'Payload HTML de {tamanho_html} bytes excede o limite de 2 MiB.')


In [ ]:
%%spark
metadados_dashboard = {
    'tamanho_bytes': tamanho_html,
    'sha256': checksum_html,
    'cd_cli': CD_CLI,
    'data_execucao': DATA_EXECUCAO.strftime('%Y%m%d'),
    'filename': f'radar_financeiro_{CD_CLI}_{DATA_EXECUCAO.strftime("%Y%m%d")}.html',
    'root_id': radar_root_id,
}


In [ ]:
from IPython.display import HTML, Javascript, display
from pathlib import Path
import hashlib
import re

html_dashboard_local = spark.get_from_spark("html_dashboard")


In [ ]:
metadados_local = spark.get_from_spark("metadados_dashboard")
html_local_bytes = html_dashboard_local.encode('utf-8')
checksum_local = hashlib.sha256(html_local_bytes).hexdigest()
if len(html_local_bytes) != metadados_local['tamanho_bytes']:
    raise RuntimeError('Tamanho do HTML divergiu após transferência Spark → kernel local.')
if checksum_local != metadados_local['sha256']:
    raise RuntimeError('Checksum SHA-256 do HTML divergiu após transferência.')

destino_html = Path.cwd().resolve() / metadados_local['filename']
destino_html.write_text(html_dashboard_local, encoding='utf-8')
print(
    f"[RADAR_V8] HTML salvo: {destino_html} "
    f"({metadados_local['tamanho_bytes']} bytes, sha256={checksum_local})."
)


In [ ]:
display(HTML(html_dashboard_local))


In [ ]:
for script_dashboard in re.findall(r'<script>(.*?)</script>', html_dashboard_local, flags=re.S | re.I):
    display(Javascript(script_dashboard))
